# Sección 0 — Imports globales

Imports comunes: pandas, numpy, plotly, ipywidgets, make_subplots y datetime.


In [1]:
# ── Imports globales ──────────────────────────────────────────────────────────
import datetime

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ipywidgets import interact, Dropdown, BoundedIntText
from sklearn.feature_selection import mutual_info_regression
from ipywidgets import FloatSlider, HBox, VBox, Output
import ipywidgets as widgets


# Sección 0.5 — Globals y funciones auxiliares

Constantes, colores y funciones auxiliares reutilizadas en todo el notebook (`tec_color`, `moda_baja_fn`, `moda_alta_fn`, etc.).


In [2]:
# ── Sección 0.5 — Globals ─────────────────────────────────────────────────────

CUARTO_AMANECER  = 600
CUARTO_ATARDECER = 1815
RAMP_BIN_MW      = 10

tec_color = {
    'Hidro':   '#378ADD',
    'Solar':   '#E8A838',
    'Eolica':  '#1D9E75',
    'Termica': '#D85A30',
}

def calcular_bin(valores):
    rango = valores.max() - valores.min()
    return max(0.01, round(rango / 20, 2))

def bin_mas_frecuente(vals, bw=RAMP_BIN_MW):
    vals = vals.dropna()
    if vals.empty: return None, None, 0
    b0    = np.floor(vals.min() / bw) * bw
    b1    = np.ceil(vals.max()  / bw) * bw
    edges = np.arange(b0, b1 + bw, bw)
    if len(edges) < 2: return b0, b1, len(vals)
    counts, edges = np.histogram(vals, bins=edges)
    i = counts.argmax()
    return edges[i], edges[i+1], int(counts[i])

def cuarto_a_franja_2h(c):
    c = int(c)
    hh2_ini = (c // 100 // 2) * 2
    return f'{hh2_ini:02d}:00–{hh2_ini+2:02d}:00'

def cuarto_a_ventana_2h(cuarto):
    hh, mm  = int(cuarto)//100, int(cuarto)%100
    mins    = hh*60 + mm
    bin_ini = (mins//120)*120
    return f"{bin_ini//60:02d}:00–{(bin_ini+120)//60:02d}:00", bin_ini

def rango_temporal_str(full_s, dias_en_rango):
    if dias_en_rango.empty: return 'N/D'
    eventos = full_s[full_s['FECHA'].isin(dias_en_rango)].copy()
    if eventos.empty: return 'N/D'
    eventos['ventana'] = eventos['CUARTO'].apply(lambda c: cuarto_a_ventana_2h(c)[0])
    ventana_top = eventos['ventana'].value_counts().idxmax()
    en_ventana  = eventos[eventos['ventana'] == ventana_top]['CUARTO']
    p25, p75    = int(en_ventana.quantile(0.25)), int(en_ventana.quantile(0.75))
    return f"{ventana_top}  |  P25–P75: C-{p25} – C-{p75}  ({len(en_ventana)} eventos)"

def hex_to_rgba(hex_color, opacity=0.18):
    hex_color = hex_color.lstrip('#')
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f'rgba({r},{g},{b},{opacity})'

# ── Bins finos para histogramas estadísticos ──────────────────────────────────
BIN_SIZE = 0.5

def moda_rango(x):
    x = x.dropna()
    if x.empty or x.max() == 0:
        return 0.0
    bins = np.arange(0, x.max() + BIN_SIZE, BIN_SIZE)
    if len(bins) < 2:
        return x.median()
    return pd.cut(x, bins=bins).value_counts().idxmax().mid

def moda_baja_fn(x):
    sub = x[x <= x.median()].dropna()
    return moda_rango(sub)

def moda_alta_fn(x):
    sub = x[x >= x.median()].dropna()
    return moda_rango(sub)

# Sección 1 — Parámetros (año, mes, carpeta)

Cambiar `AÑO_ANALISIS` y `MES_ANALISIS` es suficiente para correr el análisis completo sobre cualquier mes y año.


In [54]:
# ── Parámetros del análisis ───────────────────────────────────────────────────
# Cambiar estos dos valores es suficiente para correr el análisis completo
# sobre cualquier mes y año.
AÑO_ANALISIS  = 2023
MES_ANALISIS  = ([1,2,3,4,5,6,7,8,9,10,11,12])       # int (3), lista ([1,2,3]) o 'todos'
CARPETA_DATOS = '.'  # ruta relativa donde están los Excel

# Nombre del mes en español — usado en títulos de gráficos a lo largo del notebook
_MESES_ES = {
    1: 'Enero',  2: 'Febrero',   3: 'Marzo',     4: 'Abril',
    5: 'Mayo',   6: 'Junio',     7: 'Julio',     8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre',
}

if MES_ANALISIS == 'todos':
    NOMBRE_MES = 'Año completo'
elif isinstance(MES_ANALISIS, int):
    NOMBRE_MES = _MESES_ES[MES_ANALISIS]
else:
    NOMBRE_MES = ', '.join(_MESES_ES[m] for m in MES_ANALISIS)

print(f"Análisis configurado para: {NOMBRE_MES} {AÑO_ANALISIS}")


Análisis configurado para: Enero, Febrero, Marzo, Abril, Mayo, Junio, Julio, Agosto, Septiembre, Octubre, Noviembre, Diciembre 2023


# Sección 2 — Carga de datos

Lectura paramétrica de los Excel del mes y año seleccionados, asignación de `TECNOLOGIA`, `ZONA`, `NODO` por planta, y máscara solar nocturna (DESPACHO=0 cuando CUARTO ≤ 645 o ≥ 1830).


In [80]:
# ── Carga paramétrica de Excel mensuales ──────────────────────────────────────
import glob, os

_COLS_REQ = {'FECHA', 'CUARTO', 'NOMBRE_PLANTA', 'DESPACHO'}
_SHEET_NAMES_PREFERIDOS = ['Sheet0','Sheet 0',
    'Sheet1', 'Sheet 1', 'Sheet2', 'Sheet 2', 'Sheet3', 'Sheet 3',
    'Exportar Hoja de Trabajo',
    'Hoja 1', 'Hoja 2', 'Hoja 3',
]
_meses_a_cargar = (
    list(range(1, 13)) if MES_ANALISIS == 'todos'
    else ([MES_ANALISIS] if isinstance(MES_ANALISIS, int)
          else list(MES_ANALISIS))
)

_dfs = []
_archivos_total = []

for _m in _meses_a_cargar:
    _archivos = sorted(glob.glob(
        os.path.join(CARPETA_DATOS, f'DESPACHO_{AÑO_ANALISIS}_{_m:02d}_*.xlsx')
    ))
    if not _archivos:
        print(f"  ⚠️  No se encontraron archivos para mes {_m:02d} de {AÑO_ANALISIS}")
        continue
    _archivos_total.extend(_archivos)
    for _archivo in _archivos:
        _hojas_existentes = pd.ExcelFile(_archivo).sheet_names
        for _nombre in _SHEET_NAMES_PREFERIDOS:
            if _nombre not in _hojas_existentes:
                continue
            _tmp = pd.read_excel(_archivo, sheet_name=_nombre)
            if not _COLS_REQ.issubset(set(_tmp.columns)):
                continue
            _dfs.append(_tmp)
            print(f"   ✅ {os.path.basename(_archivo)} → hoja: '{_nombre}' | shape: {_tmp.shape}")
            # sin break → sigue leyendo las demás hojas del mismo archivo

if not _dfs:
    raise FileNotFoundError(
        f"No se encontraron datos válidos para {NOMBRE_MES} {AÑO_ANALISIS}.\n"
        f"Carpeta buscada: {os.path.abspath(CARPETA_DATOS)}"
    )

df = (pd.concat(_dfs, axis=0)
        .drop_duplicates(subset=['FECHA', 'CUARTO', 'NOMBRE_PLANTA', 'CODIGO_UNIDAD'])
        .sort_values('FECHA')
        .reset_index(drop=True))

print(f"Meses cargados       : {_meses_a_cargar}")
print(f"Archivos encontrados : {len(_archivos_total)}")
for _a in _archivos_total:
    print(f"   • {os.path.basename(_a)}")
print(f"Shape del df         : {df.shape}")
print(f"Rango de fechas      : {df['FECHA'].min()} → {df['FECHA'].max()}")

   ✅ DESPACHO_2023_01_01-31.xlsx → hoja: 'Sheet 1' | shape: (999999, 5)
   ✅ DESPACHO_2023_01_01-31.xlsx → hoja: 'Sheet 2' | shape: (713985, 5)
   ✅ DESPACHO_2023_02_01-28.xlsx → hoja: 'Sheet 1' | shape: (999999, 5)
   ✅ DESPACHO_2023_02_01-28.xlsx → hoja: 'Sheet 2' | shape: (559329, 5)
   ✅ DESPACHO_2023_03_01-31.xlsx → hoja: 'Sheet 1' | shape: (999999, 5)
   ✅ DESPACHO_2023_03_01-31.xlsx → hoja: 'Sheet 2' | shape: (737697, 5)
   ✅ DESPACHO_2023_04_01-30.xlsx → hoja: 'Sheet 1' | shape: (999999, 5)
   ✅ DESPACHO_2023_04_01-30.xlsx → hoja: 'Sheet 2' | shape: (688161, 5)
   ✅ DESPACHO_2023_05_01-31.xlsx → hoja: 'Sheet 1' | shape: (999999, 5)
   ✅ DESPACHO_2023_05_01-31.xlsx → hoja: 'Sheet 2' | shape: (755841, 5)
   ✅ DESPACHO_2023_06_01-30.xlsx → hoja: 'Sheet 1' | shape: (999999, 5)
   ✅ DESPACHO_2023_06_01-30.xlsx → hoja: 'Sheet 2' | shape: (709185, 5)
   ✅ DESPACHO_2023_07_01-31.xlsx → hoja: 'Sheet 1' | shape: (999999, 5)
   ✅ DESPACHO_2023_07_01-31.xlsx → hoja: 'Sheet 2' | shape: (769

In [81]:
tecnologia_planta = {
    # Hidro
    'BAYANO':                       'Hidro',
    'FORTUNA':                      'Hidro',
    'BONYIC':                       'Hidro',
    'CHANGUINOLA 1':                'Hidro',
    'MINI CHAN':                    'Hidro',
    'GUALACA':                      'Hidro',
    'ESTÍ':                         'Hidro',
    'LORENA':                       'Hidro',
    'PRUDENCIA':                    'Hidro',
    'BARRO BLANCO':                 'Hidro',
    'MINI BARRO BLANCO':            'Hidro',
    'MONTE LIRIO':                  'Hidro',
    'EL ALTO':                      'Hidro',
    'MINI CENTRAL EL ALTO':         'Hidro',
    'PANDO':                        'Hidro',
    'LA POTRA':                     'Hidro',
    'MINI LA POTRA':                'Hidro',
    'BAITUN':                       'Hidro',
    'MINI BAITUN':                  'Hidro',
    'BAJO DE MINA':                 'Hidro',
    'MINI BAJOMINA':                'Hidro',
    'SALSIPUEDES':                  'Hidro',
    'LOS VALLES':                   'Hidro',
    'MENDRE':                       'Hidro',
    'MENDRE 2':                     'Hidro',
    'ALGARROBOS':                   'Hidro',
    'COCHEA':                       'Hidro',
    'LA ESTRELLA':                  'Hidro',
    'DOLEGA':                       'Hidro',
    'PASO ANCHO':                   'Hidro',
    'SAN LORENZO':                  'Hidro',
    'BUGABA 1':                     'Hidro',
    'BUGABA 2':                     'Hidro',
    'LOS PLANETAS 1':               'Hidro',
    'LOS PLANETAS 2':               'Hidro',
    'MACHO MONTE':                  'Hidro',
    'LA YEGUADA':                   'Hidro',
    'EL FRAILE':                    'Hidro',
    'LAS CRUCES':                   'Hidro',
    'MINI LAS CRUCES':              'Hidro',
    'SAN ANDRES':                   'Hidro',
    'BAJOS DEL TOTUMA':             'Hidro',
    'MACANO':                       'Hidro',
    'PEDREGALITO 1':                'Hidro',
    'PEDREGALITO 2':                'Hidro',
    'LAS PERLAS NORTE':             'Hidro',
    'LAS PERLAS SUR':               'Hidro',
    'LA CUCHILLA':                  'Hidro',
    'CONCEPCION':                   'Hidro',
    'RP490':                        'Hidro',
    'MINI PEDREGALITO 1':           'Hidro',

    # Eolica
    'NUEVO CHAGRES 2':              'Eolica',
    'PORTOBELO':                    'Eolica',
    'NUEVO CHAGRES 1':              'Eolica',
    'PARQUE EOLICO TOABRE':         'Eolica',
    'ROSA VIENTOS 1':               'Eolica',
    'ROSA VIENTOS 2':               'Eolica',
    'MARANON':                      'Eolica',
    'PARQUE EOLICO CAIMITILLO':     'Eolica',

    # Termica
    'TROPITERMICA':                 'Termica',
    'TERMOCOLON':                   'Termica',
    'COSTA NORTE GAS':              'Termica',
    'COSTA NORTE DIESEL':           'Termica',
    'BARCAZA ESPERANZA':            'Termica',
    'JINRO POWER':                  'Termica',
    'PANAM':                        'Termica',
    'PANAM II':                     'Termica',
    'SPARKLE 1':                    'Termica',
    'SPARKLE 2':                    'Termica',
    'COBRE PANAMA':                 'Termica',
    'URBALIA CERRO PATACON':        'Termica',
    'PACORA':                       'Termica',
    'MIRAFLORES':                   'Termica',
    'BLM':                          'Termica',
    'PROYECTO GATUN':               'Termica',
    'CATIVA':                       'Termica',
    'CADASA':                       'Termica',

    # Solar
    'DON FELIX':                    'Solar',
    'MENDRE SOLAR':                 'Solar',
    'CEDRO SOLAR':                  'Solar',
    'CAOBA SOLAR':                  'Solar',
    'MACANO SOLAR':                 'Solar',
    'MADRE VIEJA SOLAR':            'Solar',
    'PEDREGALITO SOLAR POWER':      'Solar',
    'MENDOZA SOLAR':                'Solar',
    'SUNRISE MASPV1':               'Solar',
    'SUNRISE MASPV2':               'Solar',
    'CHAME SOLAR':                  'Solar',
    'CAPIRA SOLAR':                 'Solar',
    'SOLAR FOTOVOLTAICA PENONOME':  'Solar',
    'COCLE SOLAR 1':                'Solar',
    'SOLAR COCLE':                  'Solar',
    'SAN CARLOS SOLAR':             'Solar',
    'RODEO SOLAR':                  'Solar',
    'PARQUE SOLAR PRUDENCIA':       'Solar',
    'DIVISA SOLAR':                 'Solar',
    'SOLAR PARIS':                  'Solar',
    'FARALLON SOLAR 2':             'Solar',
    'SOLAR LOS ANGELES':            'Solar',
    'ESTRELLA SOLAR':               'Solar',
    'MILTON SOLAR':                 'Solar',
    'VISTA ALEGRE':                 'Solar',
    'SOL REAL':                     'Solar',
    'EL ESPINAL':                   'Solar',
    'SOLAR POCRI':                  'Solar',
    'BEJUCO SOLAR':                 'Solar',
    'PANASOLAR':                    'Solar',
    'PESE SOLAR':                   'Solar',
    'MAYORCA SOLAR':                'Solar',
    'JAGUITO SOLAR':                'Solar',
    'DACONAN STAR SOLAR':           'Solar',
    'ANTON SOLAR 1':                'Solar',
    'ORO SOLAR':                    'Solar',
    'LOS SANTOS SOLAR':             'Solar',
    'RIO DE JESUS SOLAR':           'Solar',
    'LA VILLA SOLAR':               'Solar',
    'COROTU SOLAR':                 'Solar',
    'EL FRAILE SOLAR 1':            'Solar',
    'SARIGUA':                      'Solar',
    'IKAKO':                        'Solar',
    'IKAKO 1':                      'Solar',
    'IKAKO 2':                      'Solar',
    'IKAKO 3':                      'Solar',
    'SOLAR CHIRIQUI':               'Solar',
    'SOLAR BUGABA':                 'Solar',
    'ANDREAS POWER':                'Solar',
    'SOL DE DAVID':                 'Solar',
    'SOLAR CALDERA':                'Solar',
    'ECOSOLAR':                     'Solar',
    'ECOSOLAR 2':                   'Solar',
    'LA ESPERANZA SOLAR 20 MW':     'Solar',
    'SOLARPRO':                     'Solar',
    'SOLARPRO 2':                   'Solar',
    'BACO SOLAR':                   'Solar',
    'ECOSOLAR 3':                   'Solar',
    'ECOSOLAR 4':                   'Solar',
    'ECOSOLAR 5':                   'Solar',
    'ESTÍ SOLAR II':                'Solar',
    'PANASOLAR II':                 'Solar',
    'PANASOLAR III':                'Solar',
    'CAMPO SOLAR SANTIAGO 1':       'Solar',
    'CAMPO SOLAR SANTIAGO 2':       'Solar',
    'CAMPO SOLAR SANTIAGO 3':       'Solar',
    'CAMPO SOLAR SANTIAGO 4':       'Solar',
    'CAMPO SOLAR SANTIAGO 5':       'Solar',
    'CAMPO SOLAR SANTIAGO 6':       'Solar',
    'CAMPO SOLAR SANTIAGO 7':       'Solar',
    'FOTOVOLTAICA SANTIAGO GEN 1':  'Solar',
    'ECOENER SAN JUAN':             'Solar',
    'CHUPAMPA SOLAR':               'Solar',
    'UP 1':                         'Solar',
    'UP 2':                         'Solar',
    'UP 3':                         'Solar',
    'UP 4':                         'Solar',
    'PACORA II':                    'Solar',
    'BRILLO SOLAR':                 'Solar',
    'PARQUE SOLAR ALANJE 1':           None,
    'PARQUE SOLAR ALANJE 2':           None,
    'PARQUE SOLAR ALANJE 3':           None,
    'CAMPO SOLAR LA VICTORIA':      'Solar',
    'PARQUE FOTOVOLTAICO SAN JUAN': 'Solar',
    'DECA SOLAR':                   'Solar',
    'LA CANTERA':                   'Solar',
    'CACAO SOLAR':                  'Solar',


    # Revisar
    'ACP':                           None,
    'INTERCAMBIO':                   None,
    'ECO HIDRO TIZINGAL':            None,
    'ZONA FRANCA ALBROOK':           None,
    
}


rename_plantas = {
    'CAPIRA SOLAR OLD'        : 'CAPIRA SOLAR',
    'MADRE VIEJA SOLAR OLD'   : 'MADRE VIEJA SOLAR',
    'BACO SOLAR OLD'          : 'BACO SOLAR',
    'MENDRE 2 OLD'            : 'MENDRE 2',
    'JAGUITO SOLAR OLD'       : 'JAGUITO SOLAR',
    'RIO DE JESUS SOLAR OLD'  : 'RIO DE JESUS SOLAR'
}


df['NOMBRE_PLANTA'] = df['NOMBRE_PLANTA'].replace(rename_plantas)

# Aplicar (sin cambios aquí)
df['TECNOLOGIA'] = df['NOMBRE_PLANTA'].map(tecnologia_planta)
# Verificar
sin_mapeo = df[df['TECNOLOGIA'].isna()]['NOMBRE_PLANTA'].unique()
if len(sin_mapeo) > 0:
    print("  Sin mapeo:")
    for p in sin_mapeo: print(f"   '{p}'")
else:
    print(" Todas las plantas mapeadas")

  Sin mapeo:
   'ACP'
   'ECO HIDRO TIZINGAL'
   'INTERCAMBIO'
   'BLM OLD'
   'ZONA FRANCA ALBROOK'


In [82]:
zona_map = {
    # Zona Oriental
    'MENDOZA SOLAR':                'Zona Oriental',
    'SUNRISE MASPV1':               'Zona Oriental',
    'SUNRISE MASPV2':               'Zona Oriental',
    'CAPIRA SOLAR':                 'Zona Oriental',
    'SAN CARLOS SOLAR':             'Zona Oriental',
    'RODEO SOLAR':                  'Zona Oriental',
    'URBALIA CERRO PATACON':        'Zona Oriental',
    'PROYECTO GATUN':               'Zona Oriental',
    'COSTA NORTE GAS':              'Zona Oriental',
    'COSTA NORTE DIESEL':           'Zona Oriental',
    'MIRAFLORES':                   'Zona Oriental',
    'PANAM':                        'Zona Oriental',
    'PANAM II':                     'Zona Oriental',
    'PACORA':                       'Zona Oriental',
    'PACORA II':                    'Zona Oriental',
    'SPARKLE 1':                    'Zona Oriental',
    'SPARKLE 2':                    'Zona Oriental',
    'CATIVA':                       'Zona Oriental',
    'TROPITERMICA':                 'Zona Oriental',
    'TERMOCOLON':                   'Zona Oriental',
    'BLM':                          'Zona Oriental',
    'BARCAZA ESPERANZA':            'Zona Oriental',
    'JINRO POWER':                  'Zona Oriental',
    'CHAME SOLAR':                  'Zona Oriental',

    # Zona Central
    'NUEVO CHAGRES 1':              'Zona Central',
    'NUEVO CHAGRES 2':              'Zona Central',
    'ROSA VIENTOS 1':               'Zona Central',
    'ROSA VIENTOS 2':               'Zona Central',
    'MARANON':                      'Zona Central',
    'PORTOBELO':                    'Zona Central',
    'PARQUE EOLICO TOABRE':         'Zona Central',
    'DIVISA SOLAR':                 'Zona Central',
    'DON FELIX':                    'Zona Central',
    'SOLAR PARIS':                  'Zona Central',
    'FARALLON SOLAR 2':             'Zona Central',
    'SOLAR LOS ANGELES':            'Zona Central',
    'ESTRELLA SOLAR':               'Zona Central',
    'MILTON SOLAR':                 'Zona Central',
    'VISTA ALEGRE':                 'Zona Central',
    'SOL REAL':                     'Zona Central',
    'EL ESPINAL':                   'Zona Central',
    'SOLAR POCRI':                  'Zona Central',
    'BEJUCO SOLAR':                 'Zona Central',
    'PANASOLAR':                    'Zona Central',
    'SOLAR FOTOVOLTAICA PENONOME':  'Zona Central',
    'LA ESPERANZA SOLAR 20 MW':     'Zona Central',
    'PESE SOLAR':                   'Zona Central',
    'MAYORCA SOLAR':                'Zona Central',
    'JAGUITO SOLAR':                'Zona Central',
    'DACONAN STAR SOLAR':           'Zona Central',
    'ANTON SOLAR 1':                'Zona Central',
    'ORO SOLAR':                    'Zona Central',
    'LOS SANTOS SOLAR':             'Zona Central',
    'RIO DE JESUS SOLAR':           'Zona Central',
    'LA VILLA SOLAR':               'Zona Central',
    'COROTU SOLAR':                 'Zona Central',
    'COCLE SOLAR 1':                'Zona Central',
    'SOLAR COCLE':                  'Zona Central',
    'LA YEGUADA':                   'Zona Central',
    'EL FRAILE':                    'Zona Central',
    'EL FRAILE SOLAR 1':            'Zona Central',
    'PANASOLAR II':                 'Zona Central',
    'PANASOLAR III':                'Zona Central',
    'CAMPO SOLAR SANTIAGO 1':       'Zona Central',
    'CAMPO SOLAR SANTIAGO 2':       'Zona Central',
    'CAMPO SOLAR SANTIAGO 3':       'Zona Central',
    'CAMPO SOLAR SANTIAGO 4':       'Zona Central',
    'CAMPO SOLAR SANTIAGO 5':       'Zona Central',
    'CAMPO SOLAR SANTIAGO 6':       'Zona Central',
    'CAMPO SOLAR SANTIAGO 7':       'Zona Central',
    'FOTOVOLTAICA SANTIAGO GEN 1':  'Zona Central',
    'LAS CRUCES':                   'Zona Central',
    'SARIGUA':                      'Zona Central',
    'CHUPAMPA SOLAR':               'Zona Central',
    'SARIGUA':                      'Zona Central',
    'COBRE PANAMA':                 'Zona Central',

    # Zona Occidental
    'SOLAR CHIRIQUI':               'Zona Occidental',
    'SOL DE DAVID':                 'Zona Occidental',
    'SOLAR CALDERA':                'Zona Occidental',
    'SOLAR BUGABA':                 'Zona Occidental',
    'ECOSOLAR':                     'Zona Occidental',
    'SOLARPRO':                     'Zona Occidental',
    'SOLARPRO 2':                   'Zona Occidental',
    'BACO SOLAR':                   'Zona Occidental',
    'ECOSOLAR 2':                   'Zona Occidental',
    'ECOSOLAR 3':                   'Zona Occidental',
    'ECOSOLAR 4':                   'Zona Occidental',
    'ECOSOLAR 5':                   'Zona Occidental',
    'ESTÍ SOLAR II':                'Zona Occidental',
    'IKAKO':                        'Zona Occidental',
    'IKAKO 1':                      'Zona Occidental',
    'IKAKO 2':                      'Zona Occidental',
    'IKAKO 3':                      'Zona Occidental',
    'PARQUE SOLAR PRUDENCIA':       'Zona Occidental',
    'MADRE VIEJA SOLAR':            'Zona Occidental',
    'CEDRO SOLAR':                  'Zona Occidental',
    'CAOBA SOLAR':                  'Zona Occidental',
    'MACANO SOLAR':                 'Zona Occidental',
    'ANDREAS POWER':                'Zona Occidental',
    'MENDRE SOLAR':                 'Zona Occidental',
    'PEDREGALITO SOLAR POWER':      'Zona Occidental',
    'UP 1':                         'Zona Occidental',
    'UP 2':                         'Zona Occidental',
    'UP 3':                         'Zona Occidental',
    'UP 4':                         'Zona Occidental',
    'MINI LAS CRUCES':              'Zona Occidental',
    'MINI BAJOMINA':                'Zona Occidental',
    'MINI LA POTRA':                'Zona Occidental',
    'MINI BARRO BLANCO':            'Zona Occidental',
    'MINI BAITUN':                  'Zona Occidental',
    'MINI CHAN':                    'Zona Occidental',
    'MINI CENTRAL EL ALTO':         'Zona Occidental',
    'CONCEPCION':                   'Zona Occidental',
    'LOS VALLES':                   'Zona Occidental',
    'MENDRE':                       'Zona Occidental',
    'MENDRE 2':                     'Zona Occidental',
    'ALGARROBOS':                   'Zona Occidental',
    'DOLEGA':                       'Zona Occidental',
    'MACHO MONTE':                  'Zona Occidental',
    'GUALACA':                      'Zona Occidental',
    'LORENA':                       'Zona Occidental',
    'MACANO':                       'Zona Occidental',
    'PASO ANCHO':                   'Zona Occidental',
    'LOS PLANETAS 1':               'Zona Occidental',
    'LOS PLANETAS 2':               'Zona Occidental',
    'PEDREGALITO 2':                'Zona Occidental',
    'MINI PEDREGALITO 1':           'Zona Occidental',
    'LAS PERLAS NORTE':             'Zona Occidental',
    'LAS PERLAS SUR':               'Zona Occidental',
    'MONTE LIRIO':                  'Zona Occidental',
    'RP490':                        'Zona Occidental',
    'BUGABA 1':                     'Zona Occidental',
    'BUGABA 2':                     'Zona Occidental',
    'SALSIPUEDES':                  'Zona Occidental',
    'SAN ANDRES':                   'Zona Occidental',
    'BAJOS DEL TOTUMA':             'Zona Occidental',
    'LA CUCHILLA':                  'Zona Occidental',
    'SAN LORENZO':                  'Zona Occidental',
    'COCHEA':                       'Zona Occidental',
    'PEDREGALITO 1':                'Zona Occidental',
    'LA POTRA':                     'Zona Occidental',
    'BARRO BLANCO':                 'Zona Occidental',
    'BONYIC':                       'Zona Occidental',
    'PANDO':                        'Zona Occidental',
    'LA ESTRELLA':                  'Zona Occidental',
    'PRUDENCIA':                    'Zona Occidental',
    'BAJO DE MINA':                 'Zona Occidental',
    'EL ALTO':                      'Zona Occidental',
    'BAITUN':                       'Zona Occidental',
    'ESTÍ':                         'Zona Occidental',
    'CADASA':                       'Zona Occidental',
    'FORTUNA':                      'Zona Occidental',
    'CHANGUINOLA 1':                'Zona Occidental',
    'LA ESPERANZA SOLAR 20 MW':     'Zona Occidental',
    'ECOENER SAN JUAN':             'Zona Occidental',

    # Revisar
    'ACP':                          None,
    'INTERCAMBIO':                  None,
    'ECO HIDRO TIZINGAL':           None,
    'FORTUNA':                      'Zona Occidental FOR',
    'BAYANO':                       'Zona Oriental BAY',
    'BRILLO SOLAR':                 None,
    'PARQUE SOLAR ALANJE 1':        None,
    'PARQUE SOLAR ALANJE 2':        None,
    'PARQUE SOLAR ALANJE 3':        None,
    'CAMPO SOLAR LA VICTORIA':      None,
    'PARQUE FOTOVOLTAICO SAN JUAN': None,
    'DECA SOLAR':                   None,
    'LA CANTERA':                   None,
}

df['ZONA'] = df['NOMBRE_PLANTA'].map(zona_map)

sin_zona = df[df['ZONA'].isna()]['NOMBRE_PLANTA'].unique()
print(f"Sin zona: {len(sin_zona)} → {sin_zona}")
print("\nZona:")
print(df['ZONA'].value_counts())

Sin zona: 5 → ['ACP' 'ECO HIDRO TIZINGAL' 'INTERCAMBIO' 'BLM OLD' 'ZONA FRANCA ALBROOK']

Zona:
ZONA
Zona Central           9339744
Zona Occidental        7654368
Zona Oriental          3431040
Zona Occidental FOR     105120
Zona Oriental BAY       105120
Name: count, dtype: int64


In [83]:
nodo_map_completo = {
    # ACP
    'ACP ':                         'ACP',

    # ANT
    'PARQUE EOLICO TOABRE':         'ANT',

    # BAY
    'BAYANO':                       'BAY',

    # BLM
    'BLM':                          'BLM',

    # BOQIII
    'MADRE VIEJA SOLAR':            'BOQIII',
    'CEDRO SOLAR':                  'BOQIII',
    'CAOBA SOLAR':                  'BOQIII',
    'MACANO SOLAR':                 'BOQIII',
    'PEDREGALITO SOLAR POWER':      'BOQIII',
    'CONCEPCION':                   'BOQIII',
    'MACANO':                       'BOQIII',
    'PEDREGALITO 2':                'BOQIII',
    'LAS PERLAS NORTE':             'BOQIII',
    'LAS PERLAS SUR':               'BOQIII',
    'RP490':                        'BOQIII',
    'LA CUCHILLA':                  'BOQIII',
    'PEDREGALITO 1':                'BOQIII',
    'CADASA':                       'BOQIII',
    'MINI PEDREGALITO 1':           'BOQIII',
    'PARQUE SOLAR ALANJE 1':            None,
    'PARQUE SOLAR ALANJE 2':            None,
    'PARQUE SOLAR ALANJE 3':            None,

    # BVI
    'MINI BARRO BLANCO':            'BVI',
    'BARRO BLANCO':                 'BVI',

    # CAL
    'MENDRE SOLAR':                 'CAL',
    'LOS VALLES':                   'CAL',
    'MENDRE':                       'CAL',
    'ALGARROBOS':                   'CAL',
    'MENDRE 2':                     'CAL',
    'COCHEA':                       'CAL',
    'LA ESTRELLA':                  'CAL',

    # CAT
    'TROPITERMICA':                 'CAT',
    'TERMOCOLON':                   'CAT',

    # CHA
    'BONYIC':                       'CHA',

    # CHO
    'MENDOZA SOLAR':                'CHO',
    'SUNRISE MASPV1':               'CHO',
    'SUNRISE MASPV2':               'CHO',
    'CHAME SOLAR':                  'CHO',
    'CAPIRA SOLAR':                 'CHO',
    'PANAM':                        'CHO',
    'PANAM II':                     'CHO',
    'BRILLO SOLAR':                 'CHO',
    'CACAO SOLAR':                  "CHO",

    # CPA
    'SPARKLE 1':                    'CPA',
    'SPARKLE 2':                    'CPA',

    # DOM
    'MINI CENTRAL EL ALTO':         'DOM',
    'MONTE LIRIO':                  'DOM',
    'PANDO':                        'DOM',
    'EL ALTO':                      'DOM',
    'El ALTO':                      'DOM',

    # ECO
    'NUEVO CHAGRES 1':              'ECO',
    'ROSA VIENTOS 1':               'ECO',
    'ROSA VIENTOS 2':               'ECO',
    'MARANON':                      'ECO',
    'PORTOBELO':                    'ECO',
    'NUEVO CHAGRES 2':              'ECO',
    'SOLAR FOTOVOLTAICA PENONOME':  'ECO',
    'COBRE PANAMA':                 'LSA',

    # EHI
    'SAN CARLOS SOLAR':             'EHI',
    'RODEO SOLAR':                  'EHI',
    'ANTON SOLAR 1':                'EHI',

    # ESP
    'MINI CHAN':                    'ESP',
    'CHANGUINOLA 1':                'ESP',

    # FOR
    'FORTUNA':                      'FOR',

    # GAT
    'PROYECTO GATUN':               'GAT',

    # GUA
    'PARQUE SOLAR PRUDENCIA':       'GUA',
    'GUALACA':                      'GUA',
    'LORENA':                       'GUA',
    'PRUDENCIA':                    'GUA',
    'ESTÍ':                         'GUA',

    # LM
    'CATIVA':                       'LM',

    # LSA
    'DIVISA SOLAR':                 'LSA',
    'DON FELIX':                    'LSA',
    'SOLAR PARIS':                  'LSA',
    'FARALLON SOLAR 2':             'LSA',
    'SOLAR LOS ANGELES':            'LSA',
    'ESTRELLA SOLAR':               'LSA',
    'MILTON SOLAR':                 'LSA',
    'SOL REAL':                     'LSA',
    'EL ESPINAL':                   'LSA',
    'SOLAR POCRI':                  'LSA',
    'BEJUCO SOLAR':                 'LSA',
    'PANASOLAR':                    'LSA',
    'PESE SOLAR':                   'LSA',
    'MAYORCA SOLAR':                'LSA',
    'JAGUITO SOLAR':                'LSA',
    'ORO SOLAR':                    'LSA',
    'LOS SANTOS SOLAR':             'LSA',
    'RIO DE JESUS SOLAR':           'LSA',
    'LA VILLA SOLAR':               'LSA',
    'COROTU SOLAR':                 'LSA',
    'LA YEGUADA':                   'LSA',
    'EL FRAILE':                    'LSA',
    'EL FRAILE SOLAR 1':            'LSA',
    'SARIGUA':                      'LSA',
    'SOLAR COCLE':                  'LSA',
    'COCLE SOLAR 1':                'LSA',
    'DACONAN STAR SOLAR':           'LSA',
    'VISTA ALEGRE':                 'LSA',
    'PANASOLAR II':                 'LSA',
    'PANASOLAR III':                'LSA',
    'FOTOVOLTAICA SANTIAGO GEN 1':  'LSA',
    'CHUPAMPA SOLAR':               'LSA',
    'CAMPO SOLAR LA VICTORIA':      'LSA',

    # MDN
    'SOLAR CHIRIQUI':               'MDN',
    'SOLAR BUGABA':                 'MDN',
    'IKAKO 1':                      'MDN',
    'IKAKO 2':                      'MDN',
    'IKAKO 3':                      'MDN',
    'IKAKO':                        'MDN',
    'ANDREAS POWER':                'MDN',
    'UP 1':                         'MDN',
    'UP 2':                         'MDN',
    'UP 3':                         'MDN',
    'UP 4':                         'MDN',
    'SAN LORENZO':                  'MDN',
    'DOLEGA':                       'MDN',
    'MACHO MONTE':                  'MDN',
    'PASO ANCHO':                   'MDN',
    'LOS PLANETAS 1':               'MDN',
    'LOS PLANETAS 2':               'MDN',
    'BUGABA 1':                     'MDN',
    'BUGABA 2':                     'MDN',
    'SAN ANDRES':                   'MDN',
    'BAJOS DEL TOTUMA':             'MDN',

    # MIR
    'MIRAFLORES':                   'MIR',

    # PAC
    'PACORA':                       'PAC',
    'PACORA II':                    'PAC',

    # PRO
    'SOL DE DAVID':                 'PRO',
    'SOLAR CALDERA':                'PRO',
    'ECOSOLAR':                     'PRO',
    'ECOSOLAR 2':                   'PRO',
    'LA ESPERANZA SOLAR 20 MW':     'PRO',
    'SOLARPRO':                     'PRO',
    'SOLARPRO 2':                   'PRO',
    'BACO SOLAR':                   'PRO',
    'ECOSOLAR 3':                   'PRO',
    'ECOSOLAR 4':                   'PRO',
    'ECOSOLAR 5':                   'PRO',
    'ESTÍ SOLAR II':                'PRO',
    'MINI BAJOMINA':                'PRO',
    'MINI LA POTRA':                'PRO',
    'MINI BAITUN':                  'PRO',
    'SALSIPUEDES':                  'PRO',
    'LA POTRA':                     'PRO',
    'BAJO DE MINA':                 'PRO',
    'BAITUN':                       'PRO',

    # SAB
    'COSTA NORTE GAS':              'SAB',
    'COSTA NORTE DIESEL':           'SAB',

    # SBA
    'MINI LAS CRUCES':              'SBA',
    'LAS CRUCES':                   'SBA',
    'CAMPO SOLAR SANTIAGO 1':       'SBA',
    'CAMPO SOLAR SANTIAGO 2':       'SBA',
    'CAMPO SOLAR SANTIAGO 3':       'SBA',
    'CAMPO SOLAR SANTIAGO 4':       'SBA',
    'CAMPO SOLAR SANTIAGO 5':       'SBA',
    'CAMPO SOLAR SANTIAGO 6':       'SBA',
    'CAMPO SOLAR SANTIAGO 7':       'SBA',

    # SMA
    'URBALIA CERRO PATACON':        'SMA',

    # Sin nodo confirmado
 
    'ECOENER SAN JUAN':             None,
    'ECO HIDRO TIZINGAL':           None,
    'BARCAZA ESPERANZA':            None,
    'JINRO POWER':                  None,
    'ZONA FRANCA ALBROOK':          None,
    'PARQUE FOTOVOLTAICO SAN JUAN': None,
    'DECA SOLAR':                   None,
    'LA CANTERA':                   None,
    'PARQUE EOLICO CAIMITILLO':     None,
}


# ── APLICAR ───────────────────────────────────────────────────────────────────
df = df.drop(columns=['NODO'], errors='ignore')
df['NODO'] = df['NOMBRE_PLANTA'].str.strip().map(nodo_map_completo)

# ── VERIFICAR ─────────────────────────────────────────────────────────────────
sin_nodo = df[df['NODO'].isna()]['NOMBRE_PLANTA'].unique()
print(f"Sin nodo: {len(sin_nodo)} → {sin_nodo}")
print(df['NODO'].value_counts())

Sin nodo: 8 → ['ACP' 'ECO HIDRO TIZINGAL' 'INTERCAMBIO' 'BLM OLD' 'BARCAZA ESPERANZA'
 'JINRO POWER' 'ZONA FRANCA ALBROOK' 'ECOENER SAN JUAN']
NODO
LSA       7331904
PRO       2471136
BOQIII    1949280
MDN       1811232
ECO       1156320
ANT        700800
CAL        420480
CHO        385440
LM         350400
GUA        350400
DOM        296448
CPA        280320
SAB        280320
CAT        210240
PAC        210240
SBA        105120
MIR        105120
ESP        105120
BVI        105120
CHA        105120
BAY        105120
SMA        105120
FOR        105120
EHI         80640
BLM         67200
Name: count, dtype: int64


In [84]:
#PASAR TODAS LAS TODOS LOS VALORES DE DF EN HORARIO NOCTURNO a 0
mask = (
    (df['TECNOLOGIA'] == 'Solar') &
    (df['DESPACHO'] > 0) &
    ((df['CUARTO'] <= 645) | (df['CUARTO'] >= 1830))
)

df.loc[mask, 'DESPACHO'] = 0

# Verificar
restantes = df[
    (df['TECNOLOGIA'] == 'Solar') &
    (df['DESPACHO'] > 0) &
    ((df['CUARTO'] <= 645) | (df['CUARTO'] >= 1830))
]
print(f"Registros restantes: {len(restantes)}")

df = df.drop(columns=['NODO_x', 'NODO_y'], errors='ignore')


Registros restantes: 0


# Sección 3 — Pivots por planta, tecnología, zona, nodo

Construcción de `df_pivot_tec`, `df_pivot_plantas`, `df_pivot_zona_tec`, `df_pivot_nodo_tec2` y sus rampas asociadas.


In [85]:
# ── Pivot solo por tecnologia ─────────────────────────────────────────────────
df_pivot_tec = pd.pivot_table(
    df,
    index=['FECHA', 'CUARTO'],
    columns='TECNOLOGIA',
    values='DESPACHO',
    aggfunc='sum'
).round(4).astype('float32')

# ── Calcular rampas correctamente — resetear entre años ──────────────────────
cuarto_idx_t  = df_pivot_tec.index.get_level_values('CUARTO')
fecha_idx_t   = pd.to_datetime(df_pivot_tec.index.get_level_values('FECHA'))

df_ramp_tec   = df_pivot_tec.diff()
anio_cambio_t = pd.Series(fecha_idx_t.year).diff().fillna(0) != 0
mask_reset_t  = (cuarto_idx_t == 0) | anio_cambio_t.values
df_ramp_tec[mask_reset_t] = np.nan

tecs  = sorted(df_pivot_tec.columns.tolist())
anios = sorted(fecha_idx_t.year.unique())

In [86]:
# 2. Por planta individual
df_pivot_plantas = pd.pivot_table(df,
                                   index=['FECHA', 'CUARTO'],
                                   columns='NOMBRE_PLANTA',
                                   values='DESPACHO',
                                   aggfunc=np.sum).round(4)

# ── Rampa e índices derivados por planta ──────────────────────────────────────
cuarto_idx_plt  = df_pivot_plantas.index.get_level_values('CUARTO')
fecha_idx_plt   = pd.to_datetime(df_pivot_plantas.index.get_level_values('FECHA'))
anio_cambio_plt = pd.Series(fecha_idx_plt.year).diff().fillna(0) != 0

df_ramp_plantas = df_pivot_plantas.diff()
df_ramp_plantas[(cuarto_idx_plt == 0) | anio_cambio_plt.values] = np.nan


C:\Users\jorge\AppData\Local\Temp\ipykernel_18168\1747862904.py:2: FutureWarning:

The provided callable <function sum at 0x000001F2E8EE53A0> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.



In [87]:
# 1. Por zona y tecnologia — el mas util para ETESA
df_pivot_zona_tec = pd.pivot_table(df,
                                    index=['FECHA', 'CUARTO'],
                                    columns=['ZONA', 'TECNOLOGIA'],
                                    values='DESPACHO',
                                    aggfunc=np.sum).round(4)

# ── Rampa e índices derivados por zona-tecnología ─────────────────────────────
cuarto_idx_z  = df_pivot_zona_tec.index.get_level_values('CUARTO')
fecha_idx_z   = pd.to_datetime(df_pivot_zona_tec.index.get_level_values('FECHA'))
anio_cambio_z = pd.Series(fecha_idx_z.year).diff().fillna(0) != 0

df_ramp_zona_tec = df_pivot_zona_tec.diff()
df_ramp_zona_tec[(cuarto_idx_z == 0) | anio_cambio_z.values] = np.nan

# Alias para compatibilidad con código existente
df_ramp_zona = df_ramp_zona_tec


C:\Users\jorge\AppData\Local\Temp\ipykernel_18168\4294220567.py:2: FutureWarning:

The provided callable <function sum at 0x000001F2E8EE53A0> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.



In [88]:
# ── Pivot por NODO y TECNOLOGÍA ───────────────────────────────────────────────
df_pivot_nodo_tec2 = pd.pivot_table(
    df,
    index=['FECHA', 'CUARTO'],
    columns=['NODO', 'TECNOLOGIA'],
    values='DESPACHO',
    aggfunc='sum'
).round(4)

# Reemplazar solar nocturno por NaN
solar_cols = [c for c in df_pivot_nodo_tec2.columns if c[1] == 'Solar']
mask_noche = (
    (df_pivot_nodo_tec2.index.get_level_values('CUARTO') <= 645) |
    (df_pivot_nodo_tec2.index.get_level_values('CUARTO') >= 1830)
)
df_pivot_nodo_tec2.loc[mask_noche, solar_cols] = np.nan


In [89]:
# ── Rampa e índices derivados por nodo-tecnología ─────────────────────────────
cuarto_idx_n  = df_pivot_nodo_tec2.index.get_level_values('CUARTO')
fecha_idx_n   = pd.to_datetime(df_pivot_nodo_tec2.index.get_level_values('FECHA'))
anio_cambio_n = pd.Series(fecha_idx_n.year).diff().fillna(0) != 0

df_ramp_nodo_tec = df_pivot_nodo_tec2.diff()
df_ramp_nodo_tec[(cuarto_idx_n == 0) | anio_cambio_n.values] = np.nan


In [90]:
# ── Globals — Widget options ──────────────────────────────────────────────────
# Calculado una sola vez tras construir los pivots.
ANIOS_DISP   = ['Todos'] + sorted(fecha_idx_t.year.unique().tolist())
TECS_DISP    = sorted(df_pivot_tec.columns.tolist())
NODOS_DISP   = sorted(set(n for n, t in df_pivot_nodo_tec2.columns))
CUARTOS_DISP = (['Día entero', 'Solo día', 'Solo noche']
                + sorted(df_pivot_tec.index.get_level_values('CUARTO').unique().tolist()))
DIAS_DISP    = sorted(fecha_idx_t.day.unique().tolist())
MESES_DISP   = sorted(fecha_idx_t.month.unique().tolist())
MESES_NOMBRES = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',
                 6:'Junio',7:'Julio',8:'Agosto',9:'Septiembre',
                 10:'Octubre',11:'Noviembre',12:'Diciembre'}

# Sección 4 — Metadata de plantas (`planta_meta`)

Tabla `planta_meta` que mapea `NOMBRE_PLANTA → (TECNOLOGIA, ZONA, NODO)`. Se define después de los pivots para que esté disponible en todas las secciones de análisis.


In [91]:
# ── Metadata por planta (NOMBRE_PLANTA → TECNOLOGIA, ZONA, NODO) ──────────────
planta_meta = (
    df[['NOMBRE_PLANTA', 'TECNOLOGIA', 'ZONA', 'NODO']]
    .drop_duplicates('NOMBRE_PLANTA')
    .set_index('NOMBRE_PLANTA')
)

fecha_idx_plt = pd.to_datetime(df_pivot_plantas.index.get_level_values('FECHA'))


# Sección 5 — Capacidades instaladas (`inst_mw_manual` → `inst_mw_final`)

`inst_mw_manual` es el registro **autoritativo** de capacidades instaladas por planta. `inst_mw_final` se construye SOLO a partir de `inst_mw_manual` (plantas sin entrada manual ⇒ 0.0).

`inst_mw` (máximo despacho observado) se conserva como referencia diagnóstica de solo lectura.


In [92]:
# ── Capacidad máxima observada por planta en los datos ───────────────────────
# Base inicial para inst_mw_final antes de aplicar inst_mw_manual.
inst_mw = df.groupby('NOMBRE_PLANTA')['DESPACHO'].max()


In [93]:
inst_mw_manual = {

    # ══════════════════════════════════════════════════════════════════════════
    # ACP
    # ══════════════════════════════════════════════════════════════════════════

    # ── ACP | Hidro ───────────────────────────────────────────────────────────
    #'ACP':                          22.5,
    #'ACP ':                         22.5,

    # ══════════════════════════════════════════════════════════════════════════
    # ANT
    # ══════════════════════════════════════════════════════════════════════════

    # ── ANT | Eolica ──────────────────────────────────────────────────────────
    'PARQUE EOLICO TOABRE':         66.0,

    # ══════════════════════════════════════════════════════════════════════════
    # BAY
    # ══════════════════════════════════════════════════════════════════════════

    # ── BAY | Hidro ───────────────────────────────────────────────────────────
    'BAYANO':                       260.0,

    # ══════════════════════════════════════════════════════════════════════════
    # BLM
    # ══════════════════════════════════════════════════════════════════════════

    # ── BLM | Termica ─────────────────────────────────────────────────────────
    'BLM':                          0.0,

    # ══════════════════════════════════════════════════════════════════════════
    # BOQIII
    # ══════════════════════════════════════════════════════════════════════════

    # ── BOQIII | Hidro ────────────────────────────────────────────────────────
    'CONCEPCION':                   10.0,
    'MACANO':                       5.80,
    'PEDREGALITO 2':                12.52,
    'LAS PERLAS NORTE':             10.0,
    'LAS PERLAS SUR':               10.0,
    'RP490':                        14.00,
    'LA CUCHILLA':                  7.62,
    'PEDREGALITO 1':                19.86,
    'CADASA':                       30.0,
    'MINI PEDREGALITO 1':           2.5,

    # ── BOQIII | Solar ────────────────────────────────────────────────────────
    'MADRE VIEJA SOLAR':            26.0,
    'CEDRO SOLAR':                  9.96,
    'CAOBA SOLAR':                  9.96,
    'MACANO SOLAR':                 4.75,
    'PEDREGALITO SOLAR POWER':      10.0,
    'PARQUE SOLAR ALANJE 1':        9.99,
    'PARQUE SOLAR ALANJE 2':        9.99,
    'PARQUE SOLAR ALANJE 3':        9.99,

    # ══════════════════════════════════════════════════════════════════════════
    # BVI
    # ══════════════════════════════════════════════════════════════════════════

    # ── BVI | Hidro ───────────────────────────────────────────────────────────
    'MINI BARRO BLANCO':            1.89,
    'BARRO BLANCO':                 26.95,

    # ══════════════════════════════════════════════════════════════════════════
    # CAL
    # ══════════════════════════════════════════════════════════════════════════

    # ── CAL | Hidro ───────────────────────────────────────────────────────────
    'LOS VALLES':                   54.8,
    'MENDRE':                       19.75,
    'MENDRE 2':                     8.0,
    'ALGARROBOS':                   9.73,
    'COCHEA':                       15.50,
    'LA ESTRELLA':                  47.2,

    # ── CAL | Solar ───────────────────────────────────────────────────────────
    'MENDRE SOLAR':                 5.5,

    # ══════════════════════════════════════════════════════════════════════════
    # CAT
    # ══════════════════════════════════════════════════════════════════════════

    # ── CAT | Termica ─────────────────────────────────────────────────────────
    'TROPITERMICA':                 5.05,
    'TERMOCOLON':                   150.0,

    # ══════════════════════════════════════════════════════════════════════════
    # CHA
    # ══════════════════════════════════════════════════════════════════════════

    # ── CHA | Hidro ───────────────────────────────────────────────────────────
    'BONYIC':                       31.8,

    # ══════════════════════════════════════════════════════════════════════════
    # CHO
    # ══════════════════════════════════════════════════════════════════════════

    # ── CHO | Termica ─────────────────────────────────────────────────────────
    'PANAM':                        99.80,
    'PANAM II':                     49.80,

    # ── CHO | Solar ───────────────────────────────────────────────────────────
    'MENDOZA SOLAR':                3.0,
    'SUNRISE MASPV1':               0.50,
    'SUNRISE MASPV2':               3.30,
    'CHAME SOLAR':                  20.0,
    'CAPIRA SOLAR':                 9.9,
    'BRILLO SOLAR':                 9.90,

    # ══════════════════════════════════════════════════════════════════════════
    # CPA
    # ══════════════════════════════════════════════════════════════════════════

    # ── CPA | Termica ─────────────────────────────────────────────────────────
    'SPARKLE 1':                    15.30,
    'SPARKLE 2':                    34.80,

    # ══════════════════════════════════════════════════════════════════════════
    # DOM
    # ══════════════════════════════════════════════════════════════════════════

    # ── DOM | Hidro ───────────────────────────────────────────────────────────
    'MONTE LIRIO':                  51.65,
    'PANDO':                        32.6,
    'EL ALTO':                      72.20,
    'MINI CENTRAL EL ALTO':         1.14,

    # ══════════════════════════════════════════════════════════════════════════
    # ECO
    # ══════════════════════════════════════════════════════════════════════════

    # ── ECO | Eolica ──────────────────────────────────────────────────────────
    'NUEVO CHAGRES 1':              55.0,
    'NUEVO CHAGRES 2':              62.5,
    'ROSA VIENTOS 1':               52.5,
    'ROSA VIENTOS 2':               50.0,
    'MARANON':                      17.5,
    'PORTOBELO':                    32.5,

    # ── ECO | Solar ───────────────────────────────────────────────────────────
    'SOLAR FOTOVOLTAICA PENONOME':  120.0,

    # ══════════════════════════════════════════════════════════════════════════
    # EHI
    # ══════════════════════════════════════════════════════════════════════════

    # ── EHI | Solar ───────────────────────────────────────────────────────────
    'SAN CARLOS SOLAR':             9.99,
    'RODEO SOLAR':                  9.9,
    'ANTON SOLAR 1':                9.99,

    # ══════════════════════════════════════════════════════════════════════════
    # ESP
    # ══════════════════════════════════════════════════════════════════════════

    # ── ESP | Hidro ───────────────────────────────────────────────────────────
    'CHANGUINOLA 1':                213.2,
    'MINI CHAN':                    9.78,

    # ══════════════════════════════════════════════════════════════════════════
    # FOR
    # ══════════════════════════════════════════════════════════════════════════

    # ── FOR | Hidro ───────────────────────────────────────────────────────────
    'FORTUNA':                      300.0,

    # ══════════════════════════════════════════════════════════════════════════
    # GAT
    # ══════════════════════════════════════════════════════════════════════════

    # ── GAT | Hidro ───────────────────────────────────────────────────────────
    'PROYECTO GATUN':               670.0,

    # ══════════════════════════════════════════════════════════════════════════
    # GUA
    # ══════════════════════════════════════════════════════════════════════════

    # ── GUA | Hidro ───────────────────────────────────────────────────────────
    'GUALACA':                      25.30,
    'LORENA':                       35.0,
    'PRUDENCIA':                    58.69,
    'ESTÍ':                         120.0,

    # ── GUA | Solar ───────────────────────────────────────────────────────────
    'PARQUE SOLAR PRUDENCIA':       9.69,

    # ══════════════════════════════════════════════════════════════════════════
    # LM
    # ══════════════════════════════════════════════════════════════════════════

    # ── LM | Termica ──────────────────────────────────────────────────────────
    'CATIVA':                       87.0,

    # ══════════════════════════════════════════════════════════════════════════
    # LSA
    # ══════════════════════════════════════════════════════════════════════════

    # ── LSA | Hidro ───────────────────────────────────────────────────────────
    'LA YEGUADA':                   7.0,
    'EL FRAILE':                    6.66,

    # ── LSA | Termica ─────────────────────────────────────────────────────────
    'COBRE PANAMA':                 300.0,

    # ── LSA | Solar ───────────────────────────────────────────────────────────
    'DIVISA SOLAR':                 9.99,
    'DON FELIX':                    9.99,
    'SOLAR PARIS':                  8.99,
    'FARALLON SOLAR 2':             9.996,
    'SOLAR LOS ANGELES':            9.52,
    'ESTRELLA SOLAR':               6.00,
    'MILTON SOLAR':                 10.0,
    'VISTA ALEGRE':                 8.22,
    'SOL REAL':                     10.00,
    'EL ESPINAL':                   8.5,
    'SOLAR POCRI':                  16.0,
    'BEJUCO SOLAR':                 0.96,
    'PANASOLAR':                    9.90,
    'EL FRAILE SOLAR 1':            0.48,
    'PESE SOLAR':                   9.98,
    'MAYORCA SOLAR':                9.97,
    'JAGUITO SOLAR':                9.99,
    'ORO SOLAR':                    9.99,
    'LOS SANTOS SOLAR':             7.56,
    'RIO DE JESUS SOLAR':           12.5,
    'LA VILLA SOLAR':               9.99,
    'COROTU SOLAR':                 9.98,
    'SARIGUA':                      2.40,
    'SOLAR COCLE':                  8.99,
    'COCLE SOLAR 1':                0.96,
    'DACONAN STAR SOLAR':           5.61,
    'PANASOLAR II':                 5.0,
    'PANASOLAR III':                5.0,
    'FOTOVOLTAICA SANTIAGO GEN 1':  5.0,
    'CHUPAMPA SOLAR':               9.9,
    'CAMPO SOLAR LA VICTORIA':      10.00,

    # ══════════════════════════════════════════════════════════════════════════
    # MDN
    # ══════════════════════════════════════════════════════════════════════════

    # ── MDN | Hidro ───────────────────────────────────────────────────────────
    'SAN LORENZO':                  8.12,
    'DOLEGA':                       3.12,
    'MACHO MONTE':                  2.5,
    'PASO ANCHO':                   6.00,
    'LOS PLANETAS 1':               4.75,
    'LOS PLANETAS 2':               8.89,
    'BUGABA 1':                     5.12,
    'BUGABA 2':                     5.86,
    'SAN ANDRES':                   10.0,
    'BAJOS DEL TOTUMA':             6.30,

    # ── MDN | Solar ───────────────────────────────────────────────────────────
    'SOLAR CHIRIQUI':               9.87,
    'SOLAR BUGABA':                 2.56,
    'IKAKO':                        10.0,
    'IKAKO 1':                      10.0,
    'IKAKO 2':                      10.0,
    'IKAKO 3':                      10.0,
    'ANDREAS POWER':                0.99,
    'UP 1':                         9.75,
    'UP 2':                         9.75,
    'UP 3':                         9.75,
    'UP 4':                         9.75,

    # ══════════════════════════════════════════════════════════════════════════
    # MIR
    # ══════════════════════════════════════════════════════════════════════════

    # ── MIR | Termica ─────────────────────────────────────────────────────────
    'MIRAFLORES':                   99.62,

    # ══════════════════════════════════════════════════════════════════════════
    # PAC
    # ══════════════════════════════════════════════════════════════════════════

    # ── PAC | Termica ─────────────────────────────────────────────────────────
    'PACORA':                       53.53,
    'PACORA II':                    3.0,

    # ══════════════════════════════════════════════════════════════════════════
    # PRO
    # ══════════════════════════════════════════════════════════════════════════

    # ── PRO | Hidro ───────────────────────────────────────────────────────────
    'SALSIPUEDES':                  27.9,
    'LA POTRA':                     27.9,
    'BAJO DE MINA':                 56.80,
    'BAITUN':                       85.90,
    'MINI BAJOMINA':                0.60,
    'MINI LA POTRA':                2.10,
    'MINI BAITUN':                  1.70,

    # ── PRO | Solar ───────────────────────────────────────────────────────────
    'SOL DE DAVID':                 7.90,
    'SOLAR CALDERA':                5.45,
    'ECOSOLAR':                     10.0,
    'ECOSOLAR 2':                   10.0,
    'ECOSOLAR 3':                   10.0,
    'ECOSOLAR 4':                   10.0,
    'ECOSOLAR 5':                   10.0,
    'SOLARPRO':                     10.0,
    'SOLARPRO 2':                   10.0,
    'BACO SOLAR':                   25.90,
    'ESTÍ SOLAR II':                17.0,
    'LA ESPERANZA SOLAR 20 MW':     19.99,

    # ══════════════════════════════════════════════════════════════════════════
    # SAB
    # ══════════════════════════════════════════════════════════════════════════

    # ── SAB | Termica ─────────────────────────────────────────────────────────
    'COSTA NORTE GAS':              381.0,

    # ══════════════════════════════════════════════════════════════════════════
    # SBA
    # ══════════════════════════════════════════════════════════════════════════

    # ── SBA | Hidro ───────────────────────────────────────────────────────────
    'LAS CRUCES':                   18.83,
    'MINI LAS CRUCES':              0.97,

    # ── SBA | Solar ───────────────────────────────────────────────────────────
    'CAMPO SOLAR SANTIAGO 1':       10.0,
    'CAMPO SOLAR SANTIAGO 2':       10.0,
    'CAMPO SOLAR SANTIAGO 3':       10.0,
    'CAMPO SOLAR SANTIAGO 4':       10.0,
    'CAMPO SOLAR SANTIAGO 5':       10.0,
    'CAMPO SOLAR SANTIAGO 6':       10.0,
    'CAMPO SOLAR SANTIAGO 7':       10.0,

    # ══════════════════════════════════════════════════════════════════════════
    # SMA
    # ══════════════════════════════════════════════════════════════════════════

    # ── SMA | Termica ─────────────────────────────────────────────────────────
    'URBALIA CERRO PATACON':        8.10,

    # ══════════════════════════════════════════════════════════════════════════
    # Sin nodo confirmado
    # ══════════════════════════════════════════════════════════════════════════

    # ── Sin nodo | Termica ────────────────────────────────────────────────────
    'BARCAZA ESPERANZA':            50.0,
    'JINRO POWER':                  50.0,

    # ── Sin nodo | Eolica ─────────────────────────────────────────────────────
    'PARQUE EOLICO CAIMITILLO':     1.87,

    # ── Sin nodo | Solar ──────────────────────────────────────────────────────
    'ECOENER SAN JUAN':             5.0,
    'PARQUE FOTOVOLTAICO SAN JUAN': 5.00,
    'DECA SOLAR':                   0.50,

    # ── Sin nodo | Hidro ──────────────────────────────────────────────────────
    'ECO HIDRO TIZINGAL':           None,

    # ══════════════════════════════════════════════════════════════════════════
    # REVISAR — nodo o capacidad pendiente de confirmar
    # ══════════════════════════════════════════════════════════════════════════
    'INTERCAMBIO':                  0.0,
    'ZONA FRANCA ALBROOK':          0.1,
    'LA CANTERA':                   4.95,
}

# ── inst_mw_final: capacidad instalada por planta ────────────────────────────
# FUENTE DE VERDAD: inst_mw_manual.
# Plantas sin entrada manual reciben 0.0 (NO se usa max(DESPACHO) como fallback).
# inst_mw (max despacho) queda como referencia diagnóstica de solo lectura.
inst_mw_final = pd.Series(
    {p: inst_mw_manual.get(p, 0.0) for p in df_pivot_plantas.columns},
    dtype=float,
)

# ── Validación: cobertura de inst_mw_manual ──────────────────────────────────
_sin_manual = [p for p in df_pivot_plantas.columns if p not in inst_mw_manual]
_con_cero   = [p for p in df_pivot_plantas.columns
               if inst_mw_manual.get(p, -1) == 0.0
               and p not in ('BLM', 'ECO HIDRO TIZINGAL', 'INTERCAMBIO')]

print(f"Plantas en pivot:      {len(df_pivot_plantas.columns)}")
print(f"Con capacidad manual:  {sum(1 for p in df_pivot_plantas.columns if p in inst_mw_manual)}")

if _sin_manual:
    print(f"\nADVERTENCIA — Sin capacidad manual ({len(_sin_manual)}) — agregar al dict:")
    for p in sorted(_sin_manual):
        print(f"  '{p}':  {inst_mw.get(p, 0.0):.2f},   # max despacho observado")
else:
    print("OK — Todas las plantas tienen capacidad manual definida.")

if _con_cero:
    print(f"\nREVISAR — Capacidad 0 (¿correcto?):")
    for p in _con_cero:
        print(f"  '{p}': 0.0")

# ── Validación: bare-string keys (implicit concatenation bug) ────────────────
# Escanea la fuente de esta celda buscando llaves de tipo 'XYZ' sin
# ':<valor>,' a continuación — Python las concatena silenciosamente con
# la siguiente cadena, corrompiendo el dict.
try:
    _cell_src = In[-1]   # IPython global: source of current cell
except (NameError, IndexError):
    _cell_src = ''
import re as _re
_bare_keys = []
_in_dict   = False
for _ln in _cell_src.splitlines():
    if 'inst_mw_manual = {' in _ln:
        _in_dict = True
        continue
    if not _in_dict:
        continue
    if _ln.strip().startswith('}'):
        _in_dict = False
        continue
    _s = _ln.strip()
    if not _s or _s.startswith('#'):
        continue
    if _re.match(r"^'[^']+'\s*$", _s) or _re.match(r'^"[^"]+"\s*$', _s):
        _bare_keys.append(_ln.rstrip())
if _bare_keys:
    print(f"\nADVERTENCIA — Llaves sin valor en inst_mw_manual "
          f"(concatenación implícita de strings):")
    for _l in _bare_keys:
        print(f"  {_l}")


Plantas en pivot:      134
Con capacidad manual:  131

ADVERTENCIA — Sin capacidad manual (3) — agregar al dict:
  'ACP':  49.59,   # max despacho observado
  'BLM OLD':  29.27,   # max despacho observado
  'COSTA NORTE DIESEL':  0.00,   # max despacho observado


# Sección 6 — Rampas globales y métricas

Rampas globales con reset por cambio de FECHA, y resumen mensual de rampas por nodo (`df_ramp_resumen_nodo`).


In [94]:
# ── Rampas globales con reset por cambio de FECHA ─────────────────────────────
# df_ramp_plantas y df_ramp_nodo se recalculan aquí con el reset correcto
# (NaN al primer cuarto de cada día), para que ningún cálculo de rampa
# arrastre valores entre días distintos.

cuarto_idx_plt = df_pivot_plantas.index.get_level_values('CUARTO')
fecha_idx_plt  = pd.to_datetime(df_pivot_plantas.index.get_level_values('FECHA'))

fecha_change_plt = (
    pd.Series(fecha_idx_plt).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
)
df_ramp_plantas = df_pivot_plantas.diff()
df_ramp_plantas[fecha_change_plt.values] = np.nan

# Rampa por NODO/TECNOLOGIA (a partir de df_pivot_nodo_tec2)
fecha_idx_nodo  = pd.to_datetime(df_pivot_nodo_tec2.index.get_level_values('FECHA'))
fecha_change_nodo = (
    pd.Series(fecha_idx_nodo).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
)
df_ramp_nodo = df_pivot_nodo_tec2.diff()
df_ramp_nodo[fecha_change_nodo.values] = np.nan


In [95]:

# ── Agregar df_ramp_nodo por NODO (sumar tecnologías) ────────────────────────
df_ramp_nodo_agg = df_ramp_nodo.copy()
df_ramp_nodo_agg.columns = [n for n, t in df_ramp_nodo_agg.columns]   # quitar nivel TEC
df_ramp_nodo_agg = df_ramp_nodo_agg.T.groupby(level=0).sum().T        # sumar por NODO

# ── Índices ───────────────────────────────────────────────────────────────────
nodos_iter_na = sorted(df_ramp_nodo_agg.columns.tolist())

# ── Construcción del DataFrame ────────────────────────────────────────────────
registros = []

for nodo in nodos_iter_na:
    # ── Capacidad: buscar plantas por NODO en planta_meta ────────────────────
    plantas_nodo = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'NODO'] == nodo
    ]
    cap = sum(inst_mw_final.get(p, 0) for p in plantas_nodo)

    for anio in ANIOS_DISP:
      for mes_int in MESES_DISP:
        mask_yr  = (np.ones(len(fecha_idx_t), dtype=bool) if anio == 'Todos'
                    else (fecha_idx_t.year == anio))
        mask     = mask_yr & (fecha_idx_t.month == mes_int)
        anio_lbl = 'Todos' if anio == 'Todos' else str(anio)

        base = pd.DataFrame({
            'FECHA':  fecha_idx_t[mask].values,
            'CUARTO': cuarto_idx_t[mask],
            'RAMP':   df_ramp_nodo_agg.loc[mask, nodo].values,
        }).dropna(subset=['RAMP'])

        if base.empty: continue

        fila = {'Nodo': nodo, 'Año': anio_lbl, 'Mes': mes_int,
                'Cap_MW': round(cap, 1) if cap > 0 else None}

        for direccion, signo in [('UP', 1), ('DOWN', -1)]:
            full_s = base[base['RAMP'] * signo > 0].copy()
            full_s['RAMP'] = full_s['RAMP'].abs()

            if full_s.empty:
                fila.update({k: None for k in [
                    f'Prom_Max_{direccion}_MW', f'RampMax_{direccion}_MW',
                    f'RampMax_{direccion}_PctCap', f'Rango_Frec_{direccion}',
                    f'Rango_Frec_{direccion}_dias', f'Rango_Frec_{direccion}_PctDias',
                    f'Ventana_Temporal_{direccion}']})
                continue

            daily_max     = full_s.groupby('FECHA')['RAMP'].max().dropna()
            prom          = daily_max.mean()
            ramp_max      = daily_max.max()
            pct_cap       = (ramp_max / cap * 100) if cap and cap > 0 else None
            lo, hi, cnt   = bin_mas_frecuente(daily_max, RAMP_BIN_MW)
            n_dias_total  = len(daily_max)
            pct_dias      = (cnt / n_dias_total * 100) if n_dias_total > 0 else None
            dias_en_rango = (daily_max[(daily_max >= lo) & (daily_max < hi)].index
                             if lo is not None else pd.Index([]))

            fila.update({
                f'Prom_Max_{direccion}_MW'        : round(prom, 2),
                f'RampMax_{direccion}_MW'         : round(ramp_max, 2),
                f'RampMax_{direccion}_PctCap'     : round(pct_cap, 1) if pct_cap else None,
                f'Rango_Frec_{direccion}'         : f'{lo:.0f}–{hi:.0f} MW' if lo else 'N/D',
                f'Rango_Frec_{direccion}_dias'    : cnt,
                f'Rango_Frec_{direccion}_PctDias' : round(pct_dias, 1) if pct_dias else None,
                f'Ventana_Temporal_{direccion}'   : rango_temporal_str(full_s, dias_en_rango),
            })

        registros.append(fila)

df_ramp_resumen_nodo = pd.DataFrame(registros)[[
    'Nodo', 'Año', 'Mes', 'Cap_MW',
    'Prom_Max_UP_MW',   'RampMax_UP_MW',   'RampMax_UP_PctCap',
    'Rango_Frec_UP',    'Rango_Frec_UP_dias',    'Rango_Frec_UP_PctDias',
    'Ventana_Temporal_UP',
    'Prom_Max_DOWN_MW', 'RampMax_DOWN_MW', 'RampMax_DOWN_PctCap',
    'Rango_Frec_DOWN',  'Rango_Frec_DOWN_dias',  'Rango_Frec_DOWN_PctDias',
    'Ventana_Temporal_DOWN',
]]
print(f"df_ramp_resumen_nodo: {df_ramp_resumen_nodo.shape[0]} filas")

# ── Widget ────────────────────────────────────────────────────────────────────
_col_nombres = {
    'Cap_MW'                  : 'Cap. Instalada (MW)',
    'Prom_Max_UP_MW'          : 'Prom Rampa Máx UP (MW)',
    'RampMax_UP_MW'           : 'Rampa UP Máx (MW)',
    'RampMax_UP_PctCap'       : 'UP Máx / Cap (%)',
    'Rango_Frec_UP'           : 'Rango UP frecuente',
    'Rango_Frec_UP_dias'      : 'Días rango UP',
    'Rango_Frec_UP_PctDias'   : '% días rango UP',
    'Ventana_Temporal_UP'     : 'Ventana UP (P25–P75)',
    'Prom_Max_DOWN_MW'        : 'Prom Rampa Máx DOWN (MW)',
    'RampMax_DOWN_MW'         : 'Rampa DOWN Máx (MW)',
    'RampMax_DOWN_PctCap'     : 'DOWN Máx / Cap (%)',
    'Rango_Frec_DOWN'         : 'Rango DOWN frecuente',
    'Rango_Frec_DOWN_dias'    : 'Días rango DOWN',
    'Rango_Frec_DOWN_PctDias' : '% días rango DOWN',
    'Ventana_Temporal_DOWN'   : 'Ventana DOWN (P25–P75)',
}
_fmt = {
    'Cap. Instalada (MW)'      : '{:.0f}',
    'Prom Rampa Máx UP (MW)'   : '{:.2f}', 'Rampa UP Máx (MW)'   : '{:.2f}',
    'UP Máx / Cap (%)'         : '{:.1f}%', 'Días rango UP'       : '{:.0f}',
    '% días rango UP'          : '{:.1f}%',
    'Prom Rampa Máx DOWN (MW)' : '{:.2f}', 'Rampa DOWN Máx (MW)' : '{:.2f}',
    'DOWN Máx / Cap (%)'       : '{:.1f}%', 'Días rango DOWN'     : '{:.0f}',
    '% días rango DOWN'        : '{:.1f}%',
}

def mostrar_rampas_nodo(nodo, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    mask_anio = df_ramp_resumen_nodo['Año'] == str(anio)
    mask_mes  = df_ramp_resumen_nodo['Mes'] == mes
    mask_nodo = (df_ramp_resumen_nodo['Nodo'] == nodo
                 if nodo != 'Todos' else pd.Series(True, index=df_ramp_resumen_nodo.index))
    subset = (df_ramp_resumen_nodo[mask_anio & mask_mes & mask_nodo]
              .set_index('Nodo').drop(columns=['Año'])
              .rename(columns=_col_nombres))
    display(
        subset.style
        .format(_fmt, na_rep='N/D')
        .set_caption(f'Rampas por Nodo — {nodo} — {anio}')
        .background_gradient(subset=['UP Máx / Cap (%)', 'DOWN Máx / Cap (%)'],
                             cmap='RdYlGn_r')
    )

interact(
    mostrar_rampas_nodo,
    nodo = Dropdown(options=['Todos']+nodos_iter_na, description='Nodo:', value='Todos'),
    anio = Dropdown(options=['Todos']+[str(a) for a in ANIOS_DISP[1:]],
                    description='Año:', value='Todos'),
    mes  = Dropdown(options=MESES_DISP, description='Mes:', value=MESES_DISP[0]),
)

df_ramp_resumen_nodo: 600 filas


interactive(children=(Dropdown(description='Nodo:', options=('Todos', 'ANT', 'BAY', 'BLM', 'BOQIII', 'BVI', 'C…

<function __main__.mostrar_rampas_nodo(nodo, anio, mes)>

# Sección 7 — Funciones helper consolidadas

Definiciones únicas reutilizadas por las visualizaciones de la Sección 8 (`_cap_activa_tec`, `_cap_activa_zona_tec`, `_cap_activa_nodo2`). Las funciones de moda (`moda_baja_fn`, `moda_alta_fn`) están en la Sección 0.5.


In [96]:
# ── Funciones helper consolidadas ─────────────────────────────────────────────
# Capacidad activa por TECNOLOGIA / ZONA-TEC / NODO en un año dado,
# o promedio entre años activos si anio=None.


def _cap_activa_tec(tec, anio=None):
    plantas_tec = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'TECNOLOGIA'] == tec
    ]
    if anio is not None:
        mask_anio = fecha_idx_plt.year == anio
        activas   = [
            p for p in plantas_tec
            if df_pivot_plantas.loc[mask_anio, p].sum(skipna=True) > 0
        ]
        return sum(inst_mw_final.get(p, 0) for p in activas)
    caps = [_cap_activa_tec(tec, a) for a in ANIOS_DISP[1:]]
    caps = [c for c in caps if c > 0]
    return float(np.mean(caps)) if caps else 0.0


def _cap_activa_zona_tec(zona, tec, anio=None):
    plantas_col = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'ZONA']       == zona
        and planta_meta.loc[p, 'TECNOLOGIA'] == tec
    ]
    if anio is not None:
        mask_anio = fecha_idx_plt.year == anio
        activas   = [
            p for p in plantas_col
            if df_pivot_plantas.loc[mask_anio, p].sum(skipna=True) > 0
        ]
        return sum(inst_mw_final.get(p, 0) for p in activas)
    caps = [_cap_activa_zona_tec(zona, tec, a) for a in ANIOS_DISP[1:]]
    caps = [c for c in caps if c > 0]
    return float(np.mean(caps)) if caps else 0.0


def _cap_activa_nodo2(nodo, anio=None):
    plantas_nodo = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'NODO'] == nodo
    ]
    if anio is not None:
        mask_anio = fecha_idx_plt.year == anio
        activas   = [
            p for p in plantas_nodo
            if df_pivot_plantas.loc[mask_anio, p].sum(skipna=True) > 0
        ]
        return sum(inst_mw_final.get(p, 0) for p in activas)
    caps = [_cap_activa_nodo2(nodo, a) for a in ANIOS_DISP[1:]]
    caps = [c for c in caps if c > 0]
    return float(np.mean(caps)) if caps else 0.0


# Sección 8 — Visualizaciones interactivas

Bloques 8a–8e: análisis por Tecnología, Zona, Nodo, Planta, Correlaciones y Lags.


In [97]:
# ── DataFrame aplanado para correlaciones nodo/tecnología ────────────────────
df_corr_nodo_tec = df_pivot_nodo_tec2.copy()
df_corr_nodo_tec.columns = [f'{nodo} — {tec}' for nodo, tec in df_corr_nodo_tec.columns]
df_corr_nodo_tec = df_corr_nodo_tec.reset_index()

tecnologias_disponibles = sorted(set(
    c.split(' — ')[1] for c in df_corr_nodo_tec.columns
    if ' — ' in c and 'revisar' not in c
))
cuartos = sorted(df_corr_nodo_tec['CUARTO'].unique())


In [98]:
# ── Función de correlación cruzada nodo-tecnología ────────────────────────────
# tec_color ya está definido en Sección 0.5
def get_color(label):
    for tec, color in tec_color.items():
        if tec in label:
            return color
    return '#cccccc'

def plot_dos_tecnologias(cuarto, tec_eje_y, tec_eje_x, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    _fidx_24 = pd.to_datetime(df_pivot_nodo_tec2.index.get_level_values('FECHA'))
    mask_mes = _fidx_24.month == mes
    _df_pivot_mes = df_pivot_nodo_tec2.loc[mask_mes]
    cols_y = [c for c in df_corr_nodo_tec.columns if f'— {tec_eje_y}' in c]
    cols_x = [c for c in df_corr_nodo_tec.columns if f'— {tec_eje_x}' in c]
    cols = list(dict.fromkeys(cols_y + cols_x))

    subset = df_corr_nodo_tec[df_corr_nodo_tec['CUARTO'] == cuarto][cols]
    subset = subset.dropna(axis=1, how='all')
    subset = subset.loc[:, subset.std(skipna=True) > 0]

    if subset.shape[1] < 2:
        print(f"Insuficientes columnas activas en cuarto {cuarto}")
        return

    corr = subset.corr().round(2)

    # Eje Y: tec_eje_y, Eje X: tec_eje_x
    filas = [c for c in corr.index   if f'— {tec_eje_y}' in c]
    cols_  = [c for c in corr.columns if f'— {tec_eje_x}' in c]

    if not filas or not cols_:
        print("Sin datos suficientes para una de las tecnologías")
        return

    corr_cross = corr.loc[filas, cols_]

    z    = corr_cross.values
    text = corr_cross.values.astype(str)

    n_y = len(filas)
    n_x = len(cols_)

    fig = go.Figure(go.Heatmap(
        z=z,
        x=cols_,
        y=filas,
        colorscale='RdBu_r',
        zmid=0, zmin=-1, zmax=1,
        text=text,
        texttemplate='%{text}',
        textfont=dict(size=10),
        colorbar=dict(title='r', thickness=15, len=0.6),
        hovertemplate='<b>%{y}</b> vs <b>%{x}</b><br>r = %{z:.2f}<extra></extra>',
        showscale=True,
    ))
    fig.update_xaxes(
        tickangle=45,
        tickfont=dict(size=10),
        ticktext=[f'<span style="color:{get_color(c)}">{c}</span>' for c in cols_],
        tickvals=list(range(len(cols_)))
    )
    fig.update_yaxes(
        autorange='reversed',
        tickfont=dict(size=10),
        ticktext=[f'<span style="color:{get_color(c)}">{c}</span>' for c in filas],
        tickvals=list(range(len(filas)))
    )
    fig.update_layout(
        title=f'Correlación {tec_eje_y} vs {tec_eje_x} — Cuarto {cuarto}',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', size=11),
        height=max(400, n_y * 45 + 150),
        width=max(500, n_x * 45 + 200),
        margin=dict(t=80, l=180, r=60, b=180)
    )
    fig.show()

    corr_pairs = (
        corr_cross.stack()
        .reset_index()
        .rename(columns={'level_0': 'A', 'level_1': 'B', 0: 'Correlacion'})
        .sort_values('Correlacion')
    )
    print(f"\n── Top 5 correlaciones negativas ──")
    print(corr_pairs.head(5).to_string(index=False))
    print(f"\n── Top 5 correlaciones positivas ──")
    print(corr_pairs.tail(5).to_string(index=False))


In [99]:
# ── Widget interactivo para plot_dos_tecnologias ─────────────────────────────
# Movido aquí porque depende de MESES_DISP definido en la celda anterior.
interact(plot_dos_tecnologias,
         mes       = Dropdown(options=MESES_DISP, description='Mes:', value=MESES_DISP[0]),
    cuarto   = Dropdown(options=cuartos,                  description='Cuarto:',   value=900),
    tec_eje_y = Dropdown(options=tecnologias_disponibles, description='Eje Y:',    value='Solar'),
    tec_eje_x = Dropdown(options=tecnologias_disponibles, description='Eje X:',    value='Eolica')
)


interactive(children=(Dropdown(description='Cuarto:', index=35, options=(np.int64(15), np.int64(30), np.int64(…

<function __main__.plot_dos_tecnologias(cuarto, tec_eje_y, tec_eje_x, mes)>

In [100]:
gen = df_pivot_tec[['Solar', 'Eolica']].reset_index()

# Solo las horas en punto (CUARTO = 0, 100, 200, ... 2300)
gen = gen[gen['CUARTO'] % 100 == 0].copy()

gen['datetime'] = (
    pd.to_datetime(gen['FECHA'].astype(str))
    + pd.to_timedelta(gen['CUARTO'] // 100, unit='h')
)

gen_horario = (
    gen.set_index('datetime')[['Solar', 'Eolica']]
    .rename(columns={'Solar': 'solar_mw_real', 'Eolica': 'eolica_mw_real'})
)

gen_horario.to_parquet('solar_eolica_horario_{AÑO_ANALISIS}.parquet')
print(f'{len(gen_horario)} horas exportadas')
display(gen_horario.head())

8760 horas exportadas


TECNOLOGIA,solar_mw_real,eolica_mw_real
datetime,,
2023-01-01 01:00:00,0.0,32.662998
2023-01-01 02:00:00,0.0,24.846001
2023-01-01 03:00:00,0.0,10.436300
2023-01-01 04:00:00,0.0,11.479800
2023-01-01 05:00:00,0.0,9.416400


## 8a — Análisis por Tecnología


In [101]:
# ── Mix renovable vs térmica + Rampa total del sistema ───────────────────────
# Requiere: df_pivot_tec, df_ramp_tec, fecha_idx_t, tec_color
# Dataset paramétrico — mes configurado en celda de parámetros

RENOVABLES_SET = {'Solar', 'Eolica', 'Hidro'}
TERMICAS_SET   = {'Termica'}
DIAS_ES        = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']


def plot_mix_ramp(anio, mes, dia):
    nombre_mes_local = MESES_NOMBRES[mes]
    try:
        fecha_obj = datetime.date(anio, mes, dia)
    except ValueError:
        print(f"⚠️  {dia} de {nombre_mes_local.lower()} {anio} no existe")
        return

    nombre_dia = DIAS_ES[fecha_obj.weekday()]
    badge      = '🟡 Fin de semana' if fecha_obj.weekday() >= 5 else '🔵 Día hábil'

    mask_mes  = (fecha_idx_t.year == anio) & (fecha_idx_t.month == mes)
    subset_mes = df_pivot_tec.loc[mask_mes]
    _f_sub42   = pd.to_datetime(subset_mes.index.get_level_values('FECHA'))
    _fc42      = pd.Series(_f_sub42).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
    ramp_mes42 = subset_mes.diff()
    ramp_mes42[_fc42.values] = np.nan
    mask      = mask_mes & (fecha_idx_t.day == dia)
    pivot_dia = df_pivot_tec.loc[mask].droplevel('FECHA').dropna(how='all')
    # ramp_mes42 está alineado con subset_mes (no con df_pivot_tec): construir
    # un mask de día sobre su propio índice para evitar IndexError.
    mask_dia_42 = _f_sub42.day == dia
    ramp_dia  = ramp_mes42[mask_dia_42].droplevel('FECHA')

    if pivot_dia.empty:
        print(f"Sin datos para {dia} {nombre_mes_local.lower()} {anio}")
        return

    # ── mix % ─────────────────────────────────────────────────────────────────
    cols_renov = [c for c in pivot_dia.columns if c in RENOVABLES_SET]
    cols_term  = [c for c in pivot_dia.columns if c in TERMICAS_SET]
    renov  = pivot_dia[cols_renov].sum(axis=1) if cols_renov else pd.Series(0, index=pivot_dia.index)
    term   = pivot_dia[cols_term].sum(axis=1)  if cols_term  else pd.Series(0, index=pivot_dia.index)
    denom  = (renov + term).replace(0, np.nan)
    pct_renov = (renov / denom * 100).fillna(0)
    pct_term  = (term  / denom * 100).fillna(0)
    avg_r, avg_t = pct_renov.mean(), pct_term.mean()

    # ── rampa total del sistema ───────────────────────────────────────────────
    ramp_total = ramp_dia.sum(axis=1)
    ramp_vals  = ramp_total.fillna(0).values
    colors_ramp = [
        'rgba(29,158,117,0.85)'  if v > 0 else
        'rgba(216,90,48,0.85)'   if v < 0 else
        'rgba(0,0,0,0)'
        for v in ramp_vals
    ]
    ramp_max = max(abs(ramp_vals.max()), abs(ramp_vals.min()))

    tick_vals = list(pivot_dia.index)[::8]
    tick_text = [str(v) for v in tick_vals]

    # ── figura 2 filas ────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.50, 0.50],
        vertical_spacing=0.06,
        subplot_titles=(
            f'Mix renovable vs térmica  —  Ren. {avg_r:.0f}%  ·  Térm. {avg_t:.0f}%',
            'Rampa total del sistema (MW / 15 min)',
        ),
    )

    # ── fila 1: % mix ─────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=pct_renov.index, y=pct_renov.round(2).values,
        name=f'Renovables ({avg_r:.0f}%)',
        mode='lines',
        line=dict(color='#1D9E75', width=2.5),
        fill='tozeroy', fillcolor='rgba(29,158,117,0.18)',
        hovertemplate='Renovables: <b>%{y:.1f}%</b><extra></extra>',
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=pct_term.index, y=pct_term.round(2).values,
        name=f'Térmica ({avg_t:.0f}%)',
        mode='lines',
        line=dict(color='#D85A30', width=2.5, dash='dash'),
        hovertemplate='Térmica: <b>%{y:.1f}%</b><extra></extra>',
    ), row=1, col=1)

    fig.add_hline(
        y=50,
        line=dict(color='rgba(255,255,255,0.20)', width=1, dash='dot'),
        annotation_text='50 %', annotation_position='top left',
        annotation_font=dict(color='rgba(255,255,255,0.35)', size=9),
        row=1, col=1,
    )

    # ── fila 2: barras de rampa total ─────────────────────────────────────────
    fig.add_trace(go.Bar(
        x=ramp_total.index, y=ramp_vals,
        marker=dict(color=colors_ramp, line=dict(width=0)),
        name='Rampa total', showlegend=False,
        hovertemplate='Δ <b>%{y:+.1f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_hline(
        y=0,
        line=dict(color='rgba(255,255,255,0.25)', width=1),
        row=2, col=1,
    )

    # ── ejes ─────────────────────────────────────────────────────────────────
    fig.update_yaxes(
        title_text='Contribución (%)',
        range=[0, 105], tickvals=[0, 25, 50, 75, 100],
        showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        row=1, col=1,
    )
    fig.update_yaxes(
        title_text='Rampa (MW/15min)',
        range=[-ramp_max * 1.15, ramp_max * 1.15],
        showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        row=2, col=1,
    )
    fig.update_xaxes(
        tickmode='array', tickvals=tick_vals, ticktext=tick_text,
        showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        title_text='Cuarto (HHMM)', row=2, col=1,
    )

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{nombre_dia} {dia} {nombre_mes_local.lower()} {anio}</b>  <sup>{badge}</sup><br>'
                f'<sup>Renovables media: <b>{avg_r:.0f}%</b>  ·  '
                f'Térmica media: <b>{avg_t:.0f}%</b></sup>'
            ),
            font=dict(size=15),
        ),
        hovermode='x unified',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(
            orientation='h', y=1.06, x=1, xanchor='right',
            bgcolor='rgba(0,0,0,0)', font=dict(size=11),
        ),
        height=620, margin=dict(t=110, l=75, r=40, b=60),
        bargap=0.0,
    )
    fig.show()

interact(
    plot_mix_ramp,
    anio = Dropdown(options=ANIOS_DISP[1:], description='Año:',  value=ANIOS_DISP[1]),
    mes  = Dropdown(options=MESES_DISP,  description='Mes:',  value=MESES_DISP[0]),
    dia  = Dropdown(options=DIAS_DISP,  description='Día:',  value=DIAS_DISP[0]),
)

interactive(children=(Dropdown(description='Año:', options=(2023,), value=2023), Dropdown(description='Mes:', …

<function __main__.plot_mix_ramp(anio, mes, dia)>

In [102]:
# ── Tabla de rampas flexible por planta ───────────────────────────────────────
zonas  = ['Todas'] + sorted(planta_meta['ZONA'].dropna().unique())
tecs   = ['Todas'] + sorted(planta_meta['TECNOLOGIA'].dropna().unique())
nodos  = ['Todos'] + sorted(planta_meta['NODO'].dropna().unique())
def tabla_ramp_flexible(zona, tecnologia, nodo, anio, mes, dia, umbral):
    nombre_mes_local = MESES_NOMBRES[mes]
    plantas = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and (zona       == 'Todas' or planta_meta.loc[p, 'ZONA']       == zona)
        and (tecnologia == 'Todas' or planta_meta.loc[p, 'TECNOLOGIA'] == tecnologia)
        and (nodo       == 'Todos' or planta_meta.loc[p, 'NODO']       == nodo)
    ]
    if not plantas:
        print("Sin plantas para la combinación seleccionada.")
        return

    mask_plt = (fecha_idx_plt.day == dia) & (fecha_idx_plt.year == anio) & (fecha_idx_plt.month == mes)
    subset   = df_pivot_plantas.loc[mask_plt, plantas].diff()
    subset.iloc[0] = np.nan
    subset.index = subset.index.get_level_values('CUARTO')

    # ── rampa total del sistema por cuarto ────────────────────────────────────
    ramp_total = subset.sum(axis=1)

    # cuarto del pico máximo y mínimo del sistema
    cuarto_peak_up   = ramp_total.idxmax()
    cuarto_peak_down = ramp_total.idxmin()
    val_peak_up      = ramp_total[cuarto_peak_up]
    val_peak_down    = ramp_total[cuarto_peak_down]

    # solo continuar si superan el umbral
    if abs(val_peak_up) < umbral and abs(val_peak_down) < umbral:
        print(f"Ningún cuarto supera el umbral de ±{umbral} MW en el sistema.")
        return

    # ── contribución de cada planta en esos dos cuartos ──────────────────────
    rows = []
    for p in plantas:
        cap    = inst_mw_final.get(p, np.nan)
        tec_p  = planta_meta.loc[p, 'TECNOLOGIA'] if p in planta_meta.index else ''
        zona_p = planta_meta.loc[p, 'ZONA']       if p in planta_meta.index else ''
        nodo_p = planta_meta.loc[p, 'NODO']       if p in planta_meta.index else ''

        contrib_up   = subset.loc[cuarto_peak_up,   p] if cuarto_peak_up   in subset.index else np.nan
        contrib_down = subset.loc[cuarto_peak_down, p] if cuarto_peak_down in subset.index else np.nan

        # ignorar plantas con cero contribución en ambos momentos
        if (pd.isna(contrib_up) or contrib_up == 0) and (pd.isna(contrib_down) or contrib_down == 0):
            continue

        rows.append({
            'PLANTA':            p,
            'NODO':              nodo_p,
            'ZONA':              zona_p,
            'TECNOLOGIA':        tec_p,
            'INST_MW':           cap,
            'RAMP_UP (MW)':      round(contrib_up,   2) if pd.notna(contrib_up)   else np.nan,
            'RAMP_UP (%)':       round(contrib_up   / cap * 100, 1) if pd.notna(cap) and cap > 0 and pd.notna(contrib_up)   else np.nan,
            'RAMP_DOWN (MW)':    round(contrib_down, 2) if pd.notna(contrib_down) else np.nan,
            'RAMP_DOWN (%)':     round(abs(contrib_down) / cap * 100, 1) if pd.notna(cap) and cap > 0 and pd.notna(contrib_down) else np.nan,
        })

    if not rows:
        print("Sin contribuciones para la combinación seleccionada.")
        return

    df_out = (
        pd.DataFrame(rows)
        .set_index('PLANTA')
        .sort_values('RAMP_UP (MW)', ascending=False)
    )

    # ── totales ───────────────────────────────────────────────────────────────
    total_inst = df_out['INST_MW'].sum(skipna=True)
    total_up   = df_out['RAMP_UP (MW)'].sum(skipna=True)
    total_down = df_out['RAMP_DOWN (MW)'].sum(skipna=True)

    df_out = df_out.rename(columns={'INST_MW': 'INST_MW (MW)'})

    df_out.loc['⚡ TOTAL'] = {
        'NODO':           '',
        'ZONA':           '',
        'TECNOLOGIA':     '',
        'INST_MW (MW)':   round(total_inst, 2),
        'RAMP_UP (MW)':   round(total_up,   2),
        'RAMP_UP (%)':    round(total_up   / total_inst * 100, 1) if total_inst > 0 else np.nan,
        'RAMP_DOWN (MW)': round(total_down, 2),
        'RAMP_DOWN (%)':  round(abs(total_down) / total_inst * 100, 1) if total_inst > 0 else np.nan,
    }

    print(f"🟢 Pico ramp-UP   del sistema: {val_peak_up:+.2f} MW  @ cuarto {cuarto_peak_up}")
    print(f"🔴 Pico ramp-DOWN del sistema: {val_peak_down:+.2f} MW  @ cuarto {cuarto_peak_down}")
    print(f"   Contribución de {len(df_out)-1} plantas  |  Día {dia} {nombre_mes_local.lower()} {anio}  |  Umbral ±{umbral} MW\n")
    display(df_out)

interact(
    tabla_ramp_flexible,
    zona       = Dropdown(options=zonas,  description='Zona:',  value='Todas'),
    tecnologia = Dropdown(options=tecs,   description='Tec:',   value='Todas'),
    nodo       = Dropdown(options=nodos,  description='Nodo:',  value='Todos'),
    anio       = Dropdown(options=ANIOS_DISP[1:],  description='Año:',   value=ANIOS_DISP[1]),
    mes        = Dropdown(options=MESES_DISP,  description='Mes:',   value=MESES_DISP[0]),
    dia        = Dropdown(options=DIAS_DISP,   description='Día:',   value=DIAS_DISP[0]),
    umbral     = BoundedIntText(min=1, max=10000, step=1, value=5, description='Umbral MW:'),
)

interactive(children=(Dropdown(description='Zona:', options=('Todas', 'Zona Central', 'Zona Occidental', 'Zona…

<function __main__.tabla_ramp_flexible(zona, tecnologia, nodo, anio, mes, dia, umbral)>

In [103]:

# ── DATOS BASE ────────────────────────────────────────────────────────────────

# ── CACHE CURVA ───────────────────────────────────────────────────────────────
def calcular_stats_tec(tec, anio='Todos', mes=None, dia='Todos'):
    plantas_tec = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'TECNOLOGIA'] == tec
    ]
    cap_total = sum(inst_mw_final.get(p, 0) for p in plantas_tec)

    temp = df_pivot_tec[[tec]].reset_index()
    temp.columns = ['FECHA', 'CUARTO', 'DESPACHO']
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])
    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]
    if dia != 'Todos':
        temp = temp[temp['FECHA'].dt.day == dia]
    if temp.empty:
        return None

    s = temp.groupby('CUARTO')['DESPACHO'].agg(
        Minima     = 'min',
        Moda_baja  = moda_baja_fn,
        Mediana    = lambda x: x.quantile(0.50),
        Moda_alta  = moda_alta_fn,
        Maxima     = 'max',
        N_dias     = 'count',
        Pct_activo = lambda x: round((x > 0).sum() / len(x) * 100, 1),
    ).round(4)
    s['cap_total'] = cap_total
    return s

stats_cache_tec = {}
for tec in TECS_DISP:
    for anio in ANIOS_DISP:
        for mes in MESES_DISP:
            try:
                stats_cache_tec[(tec, anio, mes)] = calcular_stats_tec(tec, anio, mes)
            except Exception:
                stats_cache_tec[(tec, anio, mes)] = None

# ── FUNCIÓN CURVA ─────────────────────────────────────────────────────────────
def plot_curva_tec(tec, anio, mes, dia='Todos'):
    nombre_mes_local = MESES_NOMBRES[mes]
    if dia != 'Todos':
        # No cacheable por día: recalcular sobre el filtro completo.
        stats = calcular_stats_tec(tec, anio, mes, dia)
    else:
        key = (tec, anio, mes)
        if key not in stats_cache_tec:
            try:
                stats_cache_tec[key] = calcular_stats_tec(tec, anio, mes)
            except Exception:
                stats_cache_tec[key] = None
        stats = stats_cache_tec[key]
    if stats is None:
        print(f"Sin datos para {tec} — {anio} — mes {mes} — día {dia}")
        return

    color      = tec_color.get(tec, '#888780')
    r, g, b    = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    anio_label = str(anio) if anio != 'Todos' else 'Histórico'
    cap        = stats['cap_total'].iloc[0]
    cap_txt    = f'{cap:.0f}' if cap > 0 else 'N/D'

    def pct(v): return f'{v/cap*100:.1f}%' if cap > 0 else 'N/D'

    max_mw       = stats['Maxima'].max()
    moda_alta_mw = stats['Moda_alta'].max()
    med_mw       = stats['Mediana'].max()
    moda_baja_mw = stats['Moda_baja'].max()
    min_mw       = stats['Minima'][stats['Minima'] > 0].min() if (stats['Minima'] > 0).any() else 0
    n_dias       = int(stats['N_dias'].max())
    pct_act      = round(stats['Pct_activo'].mean(), 1)

    x     = stats.index.tolist()
    x_rev = x[::-1]
    fig   = go.Figure()

    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Maxima'].tolist() + stats['Minima'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.05)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Mín–Máx',
    ))
    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Maxima'].tolist() + stats['Moda_alta'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.20)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Moda alta–Máx',
    ))
    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Moda_baja'].tolist() + stats['Minima'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.20)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Mín–Moda baja',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Minima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Mínima  {min_mw:.1f} MW ({pct(min_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Mínima: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_baja'],
        line=dict(color=f'rgba({r},{g},{b},0.6)', width=1.5, dash='dashdot'),
        name=f'Moda baja  {moda_baja_mw:.1f} MW ({pct(moda_baja_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda baja: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Mediana'],
        line=dict(color=color, width=2.5),
        name=f'Mediana  {med_mw:.1f} MW ({pct(med_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Mediana: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_alta'],
        line=dict(color=f'rgba({r},{g},{b},0.6)', width=1.5, dash='dashdot'),
        name=f'Moda alta  {moda_alta_mw:.1f} MW ({pct(moda_alta_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda alta: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Maxima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Máxima  {max_mw:.1f} MW ({pct(max_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Máxima: %{y:.2f} MW<extra></extra>',
    ))

    fig.update_layout(
        title=dict(
            text=(
                f'<b><span style="color:{color}">▌</span> {tec}</b>  '
                f'<sup>· {anio_label}</sup><br>'
                f'<sup>'
                f'Cap. instalada: <b>{cap_txt} MW</b>  |  '
                f'Máx: <b>{max_mw:.1f} MW ({pct(max_mw)})</b>  |  '
                f'Moda alta: <b>{moda_alta_mw:.1f} MW ({pct(moda_alta_mw)})</b>  |  '
                f'Mediana: <b>{med_mw:.1f} MW ({pct(med_mw)})</b>  |  '
                f'Moda baja: <b>{moda_baja_mw:.1f} MW ({pct(moda_baja_mw)})</b>  |  '
                f'Mín: <b>{min_mw:.1f} MW ({pct(min_mw)})</b>  |  '
                f'Días: <b>{n_dias}</b>  |  '
                f'% activo: <b>{pct_act:.1f}%</b>'
                f'</sup>'
            ),
            font=dict(size=13),
        ),
        xaxis=dict(
            title='Cuarto del día', tickmode='array',
            tickvals=x[::8], ticktext=[str(v) for v in x[::8]],
            showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        ),
        yaxis=dict(title='Despacho (MW)', showgrid=True,
                   gridcolor='rgba(255,255,255,0.06)', zeroline=False),
        hovermode='x unified',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)', font=dict(size=11)),
        height=560, margin=dict(t=120, l=60, r=40, b=60),
    )
    fig.show()

# ── FUNCIÓN HISTOGRAMA ────────────────────────────────────────────────────────
# ── FUNCIÓN HISTOGRAMA ────────────────────────────────────────────────────────
def plot_histograma_tec(tec, cuarto, anio, mes, dia='Todos'):
    nombre_mes_local = MESES_NOMBRES[mes]
    if tec not in df_pivot_tec.columns:
        print(f"No existe tecnología {tec}")
        return

    temp = df_pivot_tec[[tec]].reset_index()
    temp.columns = ['FECHA', 'CUARTO', 'DESPACHO']
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])

    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]
    if dia != 'Todos':
        temp = temp[temp['FECHA'].dt.day == dia]

    if cuarto == 'Día entero':
        subset     = temp
        cuarto_txt = 'Día entero'
    elif cuarto == 'Solo día':
        subset     = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        cuarto_txt = f'Solo día ({CUARTO_AMANECER}–{CUARTO_ATARDECER})'
    elif cuarto == 'Solo noche':
        subset     = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        cuarto_txt = 'Solo noche'
    else:
        subset     = temp[temp['CUARTO'] == cuarto]
        cuarto_txt = f'Cuarto {cuarto}'

    subset_clean     = subset.dropna(subset=['DESPACHO'])
    daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
    idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
    daily_cuarto_max = subset_clean.loc[idx_max.values].set_index('FECHA')['CUARTO'].rename('CUARTO_MAX')
    daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
    valores = daily['MAX_MW']

    if valores.empty or valores.max() == 0:
        print(f"Sin generación para {tec} · {cuarto_txt}")
        return

    anio_label = str(anio) if anio != 'Todos' else 'Histórico'
    color      = tec_color.get(tec, '#888780')
    n_dias     = len(valores)
    mediana    = valores.median()

    # ── BINS FIJOS de 50 MW ───────────────────────────────────────────────────
    BIN_WIDTH = 50
    bin_start = np.floor(valores.min() / BIN_WIDTH) * BIN_WIDTH
    bin_end   = np.ceil(valores.max()  / BIN_WIDTH) * BIN_WIDTH
    bin_edges = np.arange(bin_start, bin_end + BIN_WIDTH, BIN_WIDTH)

    counts, _    = np.histogram(valores, bins=bin_edges)
    bin_idx_max  = counts.argmax()
    rango_lo     = bin_edges[bin_idx_max]
    rango_hi     = bin_edges[bin_idx_max + 1]
    dias_rango   = counts[bin_idx_max]

    # ── Top franjas de 2h dentro del rango más frecuente ─────────────────────
    en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()

    if not en_rango.empty:
        en_rango['FRANJA_2H'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
        top_franjas = (
            en_rango['FRANJA_2H']
            .value_counts()
            .head(5)
            .reset_index()
        )
        top_franjas.columns = ['FRANJA', 'DIAS']
        top_txt_grafico = '  |  '.join(
            [f'{r.FRANJA} ({r.DIAS}d)' for _, r in top_franjas.iterrows()]
        )
        top_txt_print = '\n'.join(
            [f'  {r.FRANJA}  →  {r.DIAS:>3} días  ({r.DIAS / dias_rango * 100:.1f}% del rango frecuente)'
             for _, r in top_franjas.iterrows()]
        )
    else:
        top_txt_grafico = 'N/D'
        top_txt_print   = '  N/D'

    # ── Histograma ────────────────────────────────────────────────────────────
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=valores,
        xbins=dict(start=bin_start, end=bin_end, size=BIN_WIDTH),
        marker=dict(color='rgba(120,140,255,0.75)', line=dict(color='#1a1a1a', width=1)),
        hovertemplate='Despacho: %{x} MW<br>Días: %{y}<extra></extra>',
        name='Días',
    ))

    fig.update_layout(
        title=(
            f'<span style="color:{color}">▌</span> <b>{tec}</b> · '
            f'{cuarto_txt} · {anio_label}  ({n_dias} días · máximo diario)'
        ),
    )

    fig.add_vrect(
        x0=rango_lo, x1=rango_hi,
        fillcolor=color, opacity=0.25, line_width=2, line_color=color,
        annotation_text=(
            f'<b>Rango más frecuente</b><br>'
            f'{rango_lo:.0f}–{rango_hi:.0f} MW · {dias_rango} días ({dias_rango/n_dias*100:.1f}%)<br>'
            f'{top_txt_grafico}'
        ),
        annotation_position='top',
        annotation_font=dict(size=11, color=color),
    )
    fig.add_vline(
        x=valores.min(), line_color='#378ADD', line_width=1.5, line_dash='dot',
        annotation_text=f'Mín: {valores.min():.1f} MW',
        annotation_position='bottom left', annotation_font=dict(size=10, color='#7ab8f5'),
    )
    fig.add_vline(
        x=mediana, line_color='orange', line_width=2,
        annotation_text=f'Mediana: {mediana:.1f} MW',
        annotation_position='top right',
        annotation_font=dict(size=10, color='orange'),
    )
    fig.add_vline(
        x=valores.max(), line_color='#e74c3c', line_width=1.5, line_dash='dot',
        annotation_text=f'Máx: {valores.max():.1f} MW',
        annotation_position='bottom right', annotation_font=dict(size=10, color='#e74c3c'),
    )

    fig.update_layout(
        xaxis=dict(
            title='Máximo diario de despacho (MW)',
            tickmode='array', tickvals=bin_edges,
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        ),
        yaxis=dict(title='Días', showgrid=True, gridcolor='rgba(255,255,255,0.06)'),
        bargap=0.05,
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        margin=dict(t=100, l=60, r=40, b=60), height=440,
        showlegend=False,
    )
    fig.show()

    print(f"{'─'*52}")
    print(f"Tecnología: {tec}  |  {cuarto_txt}  |  {anio_label}")
    print(f"{'─'*52}")
    print(f"Mínimo histórico:    {valores.min():.4f} MW")
    print(f"Mediana:             {mediana:.4f} MW")
    print(f"Máximo histórico:    {valores.max():.4f} MW")
    print(f"Desv. estándar:      {valores.std():.4f} MW")
    print(f"{'─'*52}")
    print(f"Rango más frecuente: {rango_lo:.0f} – {rango_hi:.0f} MW  →  {dias_rango} días ({dias_rango/n_dias*100:.1f}%)")
    print(f"{'─'*52}")
    print(f"Top franjas horarias en el rango más frecuente:")
    print(top_txt_print)

# ── WRAPPER ───────────────────────────────────────────────────────────────────
def plot_tec_completo(tecnologia, cuarto, anio, mes, dia='Todos'):
    plot_curva_tec(tecnologia, anio, mes, dia)
    plot_histograma_tec(tecnologia, cuarto, anio, mes, dia)

# ── WIDGETS ───────────────────────────────────────────────────────────────────

interact(
    plot_tec_completo,
    tecnologia = Dropdown(options=TECS_DISP,          description='Tecnología:', value=TECS_DISP[0]),
    cuarto     = Dropdown(options=CUARTOS_DISP,   description='Cuarto:',    value='Solo día'),
    anio       = Dropdown(options=ANIOS_DISP,          description='Año:',       value='Todos'),
    mes        = Dropdown(options=MESES_DISP,          description='Mes:',       value=MESES_DISP[0]),
    dia        = Dropdown(options=['Todos'] + list(range(1, 32)), description='Día:', value='Todos'),
)

interactive(children=(Dropdown(description='Tecnología:', options=('Eolica', 'Hidro', 'Solar', 'Termica'), val…

<function __main__.plot_tec_completo(tecnologia, cuarto, anio, mes, dia='Todos')>

In [104]:

def build_tabla_tec(cuarto, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    rows = []
    anio_int = None if anio == 'Todos' else int(anio)

    for tec in sorted(df_pivot_tec.columns.tolist()):
        cap = _cap_activa_tec(tec, anio_int)

        temp = df_pivot_tec[[tec]].reset_index()
        temp.columns = ['FECHA', 'CUARTO', 'DESPACHO']
        temp['FECHA'] = pd.to_datetime(temp['FECHA'])
        if anio != 'Todos':
            temp = temp[temp['FECHA'].dt.year == anio_int]
        temp = temp[temp['FECHA'].dt.month == mes]

        if cuarto == 'Día entero':
            subset = temp
        elif cuarto == 'Solo día':
            subset = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        elif cuarto == 'Solo noche':
            subset = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        else:
            subset = temp[temp['CUARTO'] == cuarto]

        subset_clean     = subset.dropna(subset=['DESPACHO'])
        daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
        idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
        daily_cuarto_max = subset_clean.loc[idx_max.values].set_index('FECHA')['CUARTO'].rename('CUARTO_MAX')
        daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
        valores = daily['MAX_MW']
        if valores.empty or valores.max() == 0:
            continue

        # ── bins 50 MW ────────────────────────────────────────────────────────
        BIN_WIDTH = calcular_bin(valores)
        bin_edges = np.arange(np.floor(valores.min()/BIN_WIDTH)*BIN_WIDTH,
                              np.ceil(valores.max()/BIN_WIDTH)*BIN_WIDTH + BIN_WIDTH, BIN_WIDTH)
        counts, _ = np.histogram(valores, bins=bin_edges)
        idx_bin   = counts.argmax()
        rango_lo  = bin_edges[idx_bin]
        rango_hi  = bin_edges[idx_bin + 1]

        en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()
        if not en_rango.empty:
            en_rango['FRANJA'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
            top         = en_rango['FRANJA'].value_counts().head(3)
            cuarto_pico = '  |  '.join([f'{f}({n}d)' for f, n in top.items()])
        else:
            cuarto_pico = 'N/D'

        mediana   = valores.median()
        max_hist  = valores.max()
        min_hist  = valores.min()
        moda_alta = (rango_lo + rango_hi) / 2

        moda_baja_vals = valores[valores < moda_alta]
        if not moda_baja_vals.empty:
            be2 = np.arange(np.floor(moda_baja_vals.min()/BIN_WIDTH)*BIN_WIDTH,
                            np.ceil(moda_baja_vals.max()/BIN_WIDTH)*BIN_WIDTH + BIN_WIDTH, BIN_WIDTH)
            if len(be2) > 1:
                c2, _     = np.histogram(moda_baja_vals, bins=be2)
                i2        = c2.argmax()
                moda_baja = (be2[i2] + be2[i2+1]) / 2
            else:
                moda_baja = moda_baja_vals.min()
        else:
            moda_baja = min_hist

        n_dias            = len(valores)
        dias_zona_baja    = (valores <= moda_baja).sum()
        dias_zona_central = ((valores > moda_baja) & (valores < moda_alta)).sum()
        dias_zona_alta    = (valores >= moda_alta).sum()

        cap_lbl = f'{cap:.0f} MW {"(prom)" if anio == "Todos" else ""}'

        rows.append({
            'TECNOLOGIA':          tec,
            'CAP_MW':              round(cap, 1) if cap > 0 else None,
            'CAP_LABEL':           cap_lbl       if cap > 0 else 'N/D',
            'UTIL_TIPICA_%':       round(mediana  / cap * 100, 1) if cap > 0 else None,
            'UTIL_MAX_%':          round(max_hist / cap * 100, 1) if cap > 0 else None,
            'BRECHA_MW':           round(cap - moda_alta, 1)      if cap > 0 else None,
            'DIAS_ZONA_BAJA_%':    round(dias_zona_baja    / n_dias * 100, 1),
            'DIAS_ZONA_CENTRAL_%': round(dias_zona_central / n_dias * 100, 1),
            'DIAS_ZONA_ALTA_%':    round(dias_zona_alta    / n_dias * 100, 1),
            'CUARTO_PICO':         cuarto_pico,
            'N_DIAS':              n_dias,
        })

    df_out = pd.DataFrame(rows).set_index('TECNOLOGIA').drop(columns=['CAP_MW'])
    display(df_out)


interact(
    build_tabla_tec,
    cuarto = Dropdown(options=CUARTOS_DISP, description='Cuarto:', value='Día entero'),
    anio   = Dropdown(options=ANIOS_DISP,        description='Año:',   value='Todos'),
    mes    = Dropdown(options=MESES_DISP,    description='Mes:',   value=MESES_DISP[0])
)

interactive(children=(Dropdown(description='Cuarto:', options=('Día entero', 'Solo día', 'Solo noche', 15, 30,…

<function __main__.build_tabla_tec(cuarto, anio, mes)>

In [105]:
# ── Rampas mensuales + comportamiento diario por tecnología ───────────────────
# Requiere: df, df_pivot_tec, df_ramp_tec, df_pivot_plantas, df_ramp_plantas,
#           fecha_idx_t, cuarto_idx_t, tec_color


def plot_rampa_tec(tecnologia, anio, mes, dia, umbral):
    nombre_mes_local = MESES_NOMBRES[mes]

    if tecnologia not in df_pivot_tec.columns:
        print(f"No existe {tecnologia}"); return

    color      = tec_color.get(tecnologia, '#888780')
    fill_color = hex_to_rgba(color, 0.18)
    mask_anio   = (fecha_idx_t.year == anio) & (fecha_idx_t.month == mes)
    fecha_anio  = fecha_idx_t[mask_anio]
    cuarto_anio = cuarto_idx_t[mask_anio]
    _sub46      = df_pivot_tec.loc[mask_anio]
    _f46        = pd.to_datetime(_sub46.index.get_level_values('FECHA'))
    _fc46       = pd.Series(_f46).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
    _ramp46     = _sub46.diff()
    _ramp46[_fc46.values] = np.nan

    ramp_df = pd.DataFrame({
        'DIA':  fecha_anio.day,
        'RAMP': _ramp46[tecnologia].values,
    }).dropna()

    if ramp_df.empty:
        print(f"Sin datos para {tecnologia} — {anio}"); return

    daily = ramp_df.groupby('DIA')['RAMP'].agg(
        ramp_up_max   = lambda x: x[x > 0].max() if (x > 0).any() else 0,
        ramp_down_max = lambda x: x[x < 0].min() if (x < 0).any() else 0,
        ramp_abs_mean = lambda x: x.abs().mean(),
    ).round(4)

    # ── datos del día seleccionado ────────────────────────────────────────────
    mask_dia  = fecha_anio.day == dia
    cuartos_d = cuarto_anio[mask_dia]

    desp_dia = pd.Series(
        df_pivot_tec.loc[mask_anio, tecnologia].values[mask_dia],
        index=cuartos_d,
    ).dropna()

    ramp_dia = pd.Series(
        _ramp46[tecnologia].values[mask_dia],
        index=cuartos_d,
    ).dropna()

    cuartos_up   = ramp_dia[ramp_dia >=  umbral].index
    cuartos_down = ramp_dia[ramp_dia <= -umbral].index

    # ── figura 2 filas ────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.52, 0.48],
        vertical_spacing=0.10,
        subplot_titles=(
            f'Comportamiento — Día {dia} {nombre_mes_local.lower()} {anio}  |  umbral ±{umbral} MW',
            f'Rampas diarias — {nombre_mes_local} {anio}  |  umbral ±{umbral} MW',
        ),
    )

    # ══ FILA 1: comportamiento intradiario ════════════════════════════════════
    if not desp_dia.empty:

        # curva de despacho
        fig.add_trace(go.Scatter(
            x=desp_dia.index, y=desp_dia.round(2).values,
            name='Despacho', mode='lines',
            line=dict(color=color, width=2.2),
            fill='tozeroy', fillcolor=fill_color,
            hovertemplate='Cuarto %{x} | <b>%{y:.1f} MW</b><extra></extra>',
        ), row=1, col=1)

        # línea de rampa superpuesta
        fig.add_trace(go.Scatter(
            x=ramp_dia.index, y=ramp_dia.round(2).values,
            name='Rampa (MW/15min)', mode='lines',
            line=dict(color='rgba(255,255,255,0.40)', width=1.2, dash='dot'),
            hovertemplate='Rampa: %{y:+.1f} MW<extra></extra>',
        ), row=1, col=1)

        # marcadores ramp-UP ▲
        if len(cuartos_up) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_up,
                y=desp_dia.reindex(cuartos_up).values,
                name=f'Ramp-UP ≥{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-up', size=12,
                    color='#2ecc71',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_up).values],
                hovertemplate='Ramp-UP cuarto %{x}<br><b>+%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        # marcadores ramp-DOWN ▼
        if len(cuartos_down) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_down,
                y=desp_dia.reindex(cuartos_down).values,
                name=f'Ramp-DOWN ≤-{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-down', size=12,
                    color='#e74c3c',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_down).values],
                hovertemplate='Ramp-DOWN cuarto %{x}<br><b>%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        tick_vals_d = list(desp_dia.index)[::8]
        fig.update_xaxes(
            tickmode='array', tickvals=tick_vals_d,
            ticktext=[str(v) for v in tick_vals_d],
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
            row=1, col=1,
        )
    else:
        fig.add_annotation(
            text=f'Sin datos para el día {dia}',
            xref='paper', yref='paper', x=0.5, y=0.75,
            showarrow=False, font=dict(color='#888', size=13),
        )

    # ══ FILA 2: barras mensuales ══════════════════════════════════════════════
    bar_colors_up = [
        'rgba(46,204,113,1.0)' if d == dia else 'rgba(46,204,113,0.55)'
        for d in daily.index
    ]
    bar_border_up = [
        '#ffffff' if d == dia else 'rgba(0,0,0,0)'
        for d in daily.index
    ]
    bar_colors_dn = [
        'rgba(231,76,60,1.0)' if d == dia else 'rgba(231,76,60,0.55)'
        for d in daily.index
    ]
    bar_border_dn = [
        '#ffffff' if d == dia else 'rgba(0,0,0,0)'
        for d in daily.index
    ]

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_up_max'],
        name='Ramp-up máx',
        marker=dict(color=bar_colors_up, line=dict(color=bar_border_up, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-up máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_down_max'],
        name='Ramp-down máx',
        marker=dict(color=bar_colors_dn, line=dict(color=bar_border_dn, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-down máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=daily.index, y=daily['ramp_abs_mean'],
        name='Media abs', mode='lines+markers',
        line=dict(color=color, width=2, dash='dot'),
        marker=dict(size=5),
        hovertemplate='Día %{x}<br>Media abs: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    # líneas de umbral en fila 2
    for y_ref in [umbral, -umbral]:
        fig.add_hline(
            y=y_ref,
            line=dict(color='rgba(255,255,0,0.55)', width=1.5, dash='dash'),
            annotation_text=f'±{umbral} MW' if y_ref > 0 else '',
            annotation_position='top left',
            annotation_font=dict(color='rgba(255,255,0,0.80)', size=9),
            row=2, col=1,
        )

    # ── ejes ─────────────────────────────────────────────────────────────────
    fig.update_yaxes(
        title_text='Despacho / Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=1, col=1,
    )
    fig.update_yaxes(
        title_text='Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=2, col=1,
    )
    fig.update_xaxes(
        title_text='Día', tickmode='linear', tick0=1, dtick=1,
        showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        row=2, col=1,
    )

    fig.update_layout(
        title=dict(
            text=(
                f'<span style="color:{color}"><b>{tecnologia}</b></span>  '
                f'<sup>{nombre_mes_local} {anio}  |  Día seleccionado: <b>{dia}</b>  |  Umbral: ±{umbral} MW</sup>'
            ),
            font=dict(size=15),
        ),
        hovermode='x unified',
        barmode='overlay',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.03,
            xanchor='right', x=1, bgcolor='rgba(0,0,0,0)',
            font=dict(size=11),
        ),
        height=750, margin=dict(t=90, l=75, r=40, b=60),
    )
    fig.show()

    # ── tabla texto ───────────────────────────────────────────────────────────
    print(f"\n── Rampas sobre umbral ±{umbral} MW · {tecnologia} · {anio} ──")

    ramp_col = _ramp46[tecnologia]

    for label, signo in [('🟢 Ramp-UP', 'up'), ('🔴 Ramp-DOWN', 'down')]:
        rf = (ramp_col[ramp_col >=  umbral] if signo == 'up'
              else ramp_col[ramp_col <= -umbral])
        if rf.dropna().empty:
            print(f"\n  {label}: ninguna rampa supera el umbral de {umbral} MW"); continue
        df_m = pd.DataFrame({
            'RAMP':   rf.abs().values,
            'CUARTO': rf.index.get_level_values('CUARTO').tolist(),
            'DIA':    pd.to_datetime(rf.index.get_level_values('FECHA')).day.tolist(),
        }).dropna()
        daily_max   = (df_m.loc[df_m.groupby('DIA')['RAMP'].idxmax()]
                       .set_index('DIA').sort_index())
        dia_critico = daily_max['RAMP'].idxmax()
        n_eventos   = len(df_m)
        print(f"\n  {label}  ({n_eventos} eventos sobre {umbral} MW)")
        print(f"  {'─'*45}\n  {'Día':>5}  {'Máx (MW)':>10}  {'Cuarto':>8}")
        print(f"  {'─'*45}")
        for d, row in daily_max.iterrows():
            crit = '  ◄ CRÍTICO' if d == dia_critico else ''
            print(f"  {d:>5}  {row['RAMP']:>10.2f} MW  @ {int(row['CUARTO']):>6} HHMM{crit}")
        print(f"  {'─'*45}")
        print(f"  Promedio máx diario: {daily_max['RAMP'].mean():.2f} MW")

    # ── plantas responsables ──────────────────────────────────────────────────
    plantas_tec = [
        p for p in df_pivot_plantas.columns
        if p in df[df['TECNOLOGIA'] == tecnologia]['NOMBRE_PLANTA'].values
    ]
    if not plantas_tec:
        print("Sin plantas para desglosar."); return

    dia_up   = daily['ramp_up_max'].idxmax()
    dia_down = daily['ramp_down_max'].idxmin()
    fecha_idx_plantas = pd.to_datetime(
        df_pivot_plantas.index.get_level_values('FECHA')
    )

    print(f"\n── Plantas responsables · {tecnologia} · {anio} ──")
    for label, d, col_ramp in [
        ('🟢 Ramp-UP  máximo', dia_up,   'ramp_up_max'),
        ('🔴 Ramp-DOWN máximo', dia_down, 'ramp_down_max'),
    ]:
        mask_dp = (fecha_idx_plantas.day == d) & (fecha_idx_plantas.year == anio) & (fecha_idx_plantas.month == mes)
        contrib = {}
        for p in plantas_tec:
            if p not in df_ramp_plantas.columns:
                continue
            ramp_p = df_ramp_plantas.loc[mask_dp, p].dropna()
            if ramp_p.empty:
                continue
            val = (ramp_p[ramp_p > 0].max() if col_ramp == 'ramp_up_max'
                   else ramp_p[ramp_p < 0].min())
            if pd.notna(val):
                contrib[p] = round(val, 4)
        contrib = dict(sorted(contrib.items(), key=lambda x: abs(x[1]), reverse=True))
        print(f"\n  {label} — Día {d} {nombre_mes_local.lower()} {anio}:")
        for p, v in contrib.items():
            print(f"    {p:<35} {v:+.2f} MW")

interact(
    plot_rampa_tec,
    tecnologia = Dropdown(options=TECS_DISP,  description='Tec:',      value=TECS_DISP[0]),
    anio       = Dropdown(options=ANIOS_DISP[1:], description='Año:',      value=ANIOS_DISP[1]),
    mes        = Dropdown(options=MESES_DISP, description='Mes:',      value=MESES_DISP[0]),
    dia        = Dropdown(options=DIAS_DISP,  description='Día:',      value=DIAS_DISP[0]),
    umbral     = BoundedIntText(min=1, max=10000, step=1, value=30, description='Umbral MW:'),
)

interactive(children=(Dropdown(description='Tec:', options=('Eolica', 'Hidro', 'Solar', 'Termica'), value='Eol…

<function __main__.plot_rampa_tec(tecnologia, anio, mes, dia, umbral)>

In [106]:

# ── Construcción del DataFrame ────────────────────────────────────────────────
registros = []
tecs_iter = sorted(df_ramp_tec.columns.tolist())

for tec in tecs_iter:
    for anio in ANIOS_DISP:
      for mes_int in MESES_DISP:
        anio_int = None if anio == 'Todos' else int(anio)
        anio_lbl = 'Todos' if anio == 'Todos' else str(anio)

        cap  = _cap_activa_tec(tec, anio_int)
        mask_yr = (np.ones(len(fecha_idx_t), dtype=bool) if anio == 'Todos'
                   else (fecha_idx_t.year == anio_int))
        mask = mask_yr & (fecha_idx_t.month == mes_int)

        base = pd.DataFrame({
            'FECHA':  fecha_idx_t[mask].values,
            'CUARTO': cuarto_idx_t[mask],
            'RAMP':   df_ramp_tec.loc[mask, tec].values,
        }).dropna(subset=['RAMP'])

        if base.empty: continue

        fila = {'Tecnologia': tec, 'Año': anio_lbl, 'Mes': mes_int,
                'Cap_MW': round(cap, 1) if cap > 0 else None}

        for direccion, signo in [('UP', 1), ('DOWN', -1)]:
            full_s = base[base['RAMP'] * signo > 0].copy()
            full_s['RAMP'] = full_s['RAMP'].abs()

            if full_s.empty:
                fila.update({k: None for k in [
                    f'Prom_Max_{direccion}_MW',    f'RampMax_{direccion}_MW',
                    f'RampMax_{direccion}_PctCap', f'Rango_Frec_{direccion}',
                    f'Rango_Frec_{direccion}_dias', f'Rango_Frec_{direccion}_PctDias',
                    f'Ventana_Temporal_{direccion}']})
                continue

            daily_max    = full_s.groupby('FECHA')['RAMP'].max().dropna()
            prom         = daily_max.mean()
            ramp_max     = daily_max.max()
            pct_cap      = (ramp_max / cap * 100) if cap > 0 else None
            lo, hi, cnt  = bin_mas_frecuente(daily_max, RAMP_BIN_MW)
            n_dias_total = len(daily_max)
            pct_dias     = (cnt / n_dias_total * 100) if n_dias_total > 0 else None
            dias_en_rango = (daily_max[(daily_max >= lo) & (daily_max < hi)].index
                             if lo is not None else pd.Index([]))

            fila.update({
                f'Prom_Max_{direccion}_MW'        : round(prom, 2),
                f'RampMax_{direccion}_MW'         : round(ramp_max, 2),
                f'RampMax_{direccion}_PctCap'     : round(pct_cap, 1) if pct_cap else None,
                f'Rango_Frec_{direccion}'         : f'{lo:.1f}–{hi:.1f} MW' if lo is not None else 'N/D',
                f'Rango_Frec_{direccion}_dias'    : cnt,
                f'Rango_Frec_{direccion}_PctDias' : round(pct_dias, 1) if pct_dias else None,
                f'Ventana_Temporal_{direccion}'   : rango_temporal_str(full_s, dias_en_rango),
            })

        registros.append(fila)

df_ramp_resumen = pd.DataFrame(registros)[[
    'Tecnologia', 'Año', 'Mes', 'Cap_MW',
    'Prom_Max_UP_MW',   'RampMax_UP_MW',   'RampMax_UP_PctCap',
    'Rango_Frec_UP',    'Rango_Frec_UP_dias',    'Rango_Frec_UP_PctDias',
    'Ventana_Temporal_UP',
    'Prom_Max_DOWN_MW', 'RampMax_DOWN_MW', 'RampMax_DOWN_PctCap',
    'Rango_Frec_DOWN',  'Rango_Frec_DOWN_dias',  'Rango_Frec_DOWN_PctDias',
    'Ventana_Temporal_DOWN',
]]
print(f"df_ramp_resumen: {df_ramp_resumen.shape[0]} filas × {df_ramp_resumen.shape[1]} columnas")

# ── Widget ────────────────────────────────────────────────────────────────────
_col_nombres = {
    'Cap_MW'                   : 'Cap. Instalada (MW)',
    'Prom_Max_UP_MW'           : 'Promedio Rampa Máx UP (MW)',
    'RampMax_UP_MW'            : 'Rampa UP Histórica Máx (MW)',
    'RampMax_UP_PctCap'        : 'Rampa UP Máx / Cap (%)',
    'Rango_Frec_UP'            : 'Rango UP más frecuente',
    'Rango_Frec_UP_dias'       : 'Días en rango UP',
    'Rango_Frec_UP_PctDias'    : '% días en rango UP',
    'Ventana_Temporal_UP'      : 'Ventana horaria UP  (P25–P75)',
    'Prom_Max_DOWN_MW'         : 'Promedio Rampa Máx DOWN (MW)',
    'RampMax_DOWN_MW'          : 'Rampa DOWN Histórica Máx (MW)',
    'RampMax_DOWN_PctCap'      : 'Rampa DOWN Máx / Cap (%)',
    'Rango_Frec_DOWN'          : 'Rango DOWN más frecuente',
    'Rango_Frec_DOWN_dias'     : 'Días en rango DOWN',
    'Rango_Frec_DOWN_PctDias'  : '% días en rango DOWN',
    'Ventana_Temporal_DOWN'    : 'Ventana horaria DOWN  (P25–P75)',
}

_fmt_orig = {
    'Cap_MW'                  : '{:.0f}',
    'Prom_Max_UP_MW'          : '{:.1f}',
    'RampMax_UP_MW'           : '{:.1f}',
    'RampMax_UP_PctCap'       : '{:.1f}%',
    'Rango_Frec_UP_dias'      : '{:.0f}',
    'Rango_Frec_UP_PctDias'   : '{:.1f}%',
    'Prom_Max_DOWN_MW'        : '{:.1f}',
    'RampMax_DOWN_MW'         : '{:.1f}',
    'RampMax_DOWN_PctCap'     : '{:.1f}%',
    'Rango_Frec_DOWN_dias'    : '{:.0f}',
    'Rango_Frec_DOWN_PctDias' : '{:.1f}%',
}

_fmt_legible = {_col_nombres[k]: v for k, v in _fmt_orig.items() if k in _col_nombres}

def mostrar_resumen_rampas(anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    subset = (
        df_ramp_resumen[(df_ramp_resumen['Año'] == str(anio)) & (df_ramp_resumen['Mes'] == mes)]
        .set_index('Tecnologia')
        .drop(columns=['Año'])
        .rename(columns=_col_nombres)   # renombrar PRIMERO
    )

    col_up   = _col_nombres['RampMax_UP_PctCap']
    col_down = _col_nombres['RampMax_DOWN_PctCap']

    display(
        subset.style
        .format(_fmt_legible, na_rep='N/D')
        .set_caption(
            f'Resumen de rampas — {anio}'
            + (' (Cap = promedio años activos)' if anio == 'Todos'
               else ' (Cap = plantas activas ese año)')
        )
        .background_gradient(
            subset=pd.IndexSlice[:, [col_up, col_down]],  # IndexSlice evita el KeyError
            cmap='RdYlGn_r'
        )
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size','12px'),('font-weight','bold'),
                       ('text-align','left'),('padding-bottom','8px')]},
            {'selector': 'th',
             'props': [('font-size','11px'),('text-align','center')]},
            {'selector': 'td',
             'props': [('font-size','11px'),('text-align','center')]},
        ])
    )

_anios_widget = ['Todos'] + sorted(
    [a for a in df_ramp_resumen['Año'].unique() if a != 'Todos']
)

interact(
    mostrar_resumen_rampas,
    anio = Dropdown(options=_anios_widget, description='Año:', value='Todos'),
    mes  = Dropdown(options=MESES_DISP, description='Mes:', value=MESES_DISP[0]),
)

df_ramp_resumen: 96 filas × 18 columnas


interactive(children=(Dropdown(description='Año:', options=('Todos', '2023'), value='Todos'), Dropdown(descrip…

<function __main__.mostrar_resumen_rampas(anio, mes)>

## 8b — Análisis por Zona


In [107]:
# ── Rampas mensuales + comportamiento diario con marcadores de umbral ─────────
# Requiere: df, df_pivot_zona_tec, df_pivot_plantas, df_ramp_plantas,
#           fecha_idx_z, cuarto_idx_z, tec_color

# ── pivots y rampas ───────────────────────────────────────────────────────────
cuarto_idx_z  = df_pivot_zona_tec.index.get_level_values('CUARTO')
fecha_idx_z   = pd.to_datetime(df_pivot_zona_tec.index.get_level_values('FECHA'))
df_ramp_zona   = df_pivot_zona_tec.diff()
fecha_change_z = pd.Series(fecha_idx_z).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
df_ramp_zona[fecha_change_z.values] = np.nan

df_pivot_zona_total = pd.pivot_table(
    df, index=['FECHA', 'CUARTO'], columns='ZONA',
    values='DESPACHO', aggfunc='sum'
).round(4)
cuarto_idx_zt  = df_pivot_zona_total.index.get_level_values('CUARTO')
fecha_idx_zt   = pd.to_datetime(df_pivot_zona_total.index.get_level_values('FECHA'))
df_ramp_zona_total  = df_pivot_zona_total.diff()
fecha_change_zt = pd.Series(fecha_idx_zt).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
df_ramp_zona_total[fecha_change_zt.values] = np.nan

zonas = sorted(set(z for z, t in df_pivot_zona_tec.columns))
tecs  = ['Todas'] + sorted(set(t for z, t in df_pivot_zona_tec.columns))
anios = sorted(fecha_idx_z.year.unique())
dias  = list(range(1, 32))

# ── función principal ─────────────────────────────────────────────────────────
def plot_rampa_zona(zona, tecnologia, anio, mes, dia, umbral):
    nombre_mes_local = MESES_NOMBRES[mes]

    mask_anio = (fecha_idx_z.year == anio) & (fecha_idx_z.month == mes)

    if tecnologia == 'Todas':
        if zona not in df_pivot_zona_total.columns:
            print(f"No existe zona: {zona}"); return
        mask_at    = (fecha_idx_zt.year == anio) & (fecha_idx_zt.month == mes)
        _sub49zt   = df_pivot_zona_total.loc[mask_at]
        _f49zt     = pd.to_datetime(_sub49zt.index.get_level_values('FECHA'))
        _fc49zt    = pd.Series(_f49zt).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
        _ramp49zt  = _sub49zt.diff()
        _ramp49zt[_fc49zt.values] = np.nan
        ramp_mes   = _ramp49zt[zona]
        desp_mes   = df_pivot_zona_total.loc[mask_at, zona]
        fecha_mes  = fecha_idx_zt[mask_at]
        cuarto_mes = cuarto_idx_zt[mask_at]
        color      = '#cccccc'
    else:
        col = (zona, tecnologia)
        if col not in df_pivot_zona_tec.columns:
            print(f"No existe {zona} — {tecnologia}"); return
        _sub49z    = df_pivot_zona_tec.loc[mask_anio]
        _f49z      = pd.to_datetime(_sub49z.index.get_level_values('FECHA'))
        _fc49z     = pd.Series(_f49z).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
        _ramp49z   = _sub49z.diff()
        _ramp49z[_fc49z.values] = np.nan
        ramp_mes   = _ramp49z[col]
        desp_mes   = df_pivot_zona_tec.loc[mask_anio, col]
        fecha_mes  = fecha_idx_z[mask_anio]
        cuarto_mes = cuarto_idx_z[mask_anio]
        color      = tec_color.get(tecnologia, '#888780')

    fill_color = hex_to_rgba(color, 0.18)

    ramp_df = pd.DataFrame({
        'DIA': fecha_mes.day, 'RAMP': ramp_mes.values
    }).dropna()

    if ramp_df.empty:
        print(f"Sin datos para {zona} — {tecnologia} — {anio}"); return

    daily = ramp_df.groupby('DIA')['RAMP'].agg(
        ramp_up_max   = lambda x: x[x > 0].max() if (x > 0).any() else 0,
        ramp_down_max = lambda x: x[x < 0].min() if (x < 0).any() else 0,
        ramp_abs_mean = lambda x: x.abs().mean(),
    ).round(4)

    # ── datos del día seleccionado ────────────────────────────────────────────
    mask_dia  = fecha_mes.day == dia
    cuartos_d = cuarto_mes[mask_dia]

    desp_dia = pd.Series(
        desp_mes.values[mask_dia],
        index=cuartos_d,
    ).dropna()

    ramp_dia = pd.Series(
        ramp_mes.values[mask_dia],
        index=cuartos_d,
    ).dropna()

    cuartos_up   = ramp_dia[ramp_dia >=  umbral].index
    cuartos_down = ramp_dia[ramp_dia <= -umbral].index

    tec_label = tecnologia if tecnologia != 'Todas' else 'Todas las tec.'

    # ── figura 2 filas ────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.52, 0.48],
        vertical_spacing=0.10,
        subplot_titles=(
            f'Comportamiento — Día {dia} {nombre_mes_local.lower()} {anio}  |  umbral ±{umbral} MW',
            f'Rampas diarias — {nombre_mes_local} {anio}  |  umbral ±{umbral} MW',
        ),
    )

    # ══ FILA 1: comportamiento intradiario ════════════════════════════════════
    if not desp_dia.empty:

        # curva de despacho
        fig.add_trace(go.Scatter(
            x=desp_dia.index, y=desp_dia.round(2).values,
            name='Despacho', mode='lines',
            line=dict(color=color, width=2.2),
            fill='tozeroy', fillcolor=fill_color,
            hovertemplate='Cuarto %{x} | <b>%{y:.1f} MW</b><extra></extra>',
        ), row=1, col=1)

        # línea de rampa superpuesta
        fig.add_trace(go.Scatter(
            x=ramp_dia.index, y=ramp_dia.round(2).values,
            name='Rampa (MW/15min)', mode='lines',
            line=dict(color='rgba(255,255,255,0.40)', width=1.2, dash='dot'),
            hovertemplate='Rampa: %{y:+.1f} MW<extra></extra>',
        ), row=1, col=1)

        # marcadores ramp-UP ▲
        if len(cuartos_up) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_up,
                y=desp_dia.reindex(cuartos_up).values,
                name=f'Ramp-UP ≥{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-up', size=12,
                    color='#2ecc71',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_up).values],
                hovertemplate='Ramp-UP cuarto %{x}<br><b>+%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        # marcadores ramp-DOWN ▼
        if len(cuartos_down) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_down,
                y=desp_dia.reindex(cuartos_down).values,
                name=f'Ramp-DOWN ≤-{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-down', size=12,
                    color='#e74c3c',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_down).values],
                hovertemplate='Ramp-DOWN cuarto %{x}<br><b>%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        tick_vals_d = list(desp_dia.index)[::8]
        fig.update_xaxes(
            tickmode='array', tickvals=tick_vals_d,
            ticktext=[str(v) for v in tick_vals_d],
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
            row=1, col=1,
        )
    else:
        fig.add_annotation(
            text=f'Sin datos para el día {dia}',
            xref='paper', yref='paper', x=0.5, y=0.75,
            showarrow=False, font=dict(color='#888', size=13),
        )

    # ══ FILA 2: barras mensuales ══════════════════════════════════════════════
    bar_colors_up = [
        'rgba(46,204,113,1.0)' if d == dia else 'rgba(46,204,113,0.55)'
        for d in daily.index
    ]
    bar_border_up = [
        '#ffffff' if d == dia else 'rgba(0,0,0,0)'
        for d in daily.index
    ]
    bar_colors_dn = [
        'rgba(231,76,60,1.0)' if d == dia else 'rgba(231,76,60,0.55)'
        for d in daily.index
    ]
    bar_border_dn = [
        '#ffffff' if d == dia else 'rgba(0,0,0,0)'
        for d in daily.index
    ]

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_up_max'],
        name='Ramp-up máx',
        marker=dict(color=bar_colors_up, line=dict(color=bar_border_up, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-up máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_down_max'],
        name='Ramp-down máx',
        marker=dict(color=bar_colors_dn, line=dict(color=bar_border_dn, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-down máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=daily.index, y=daily['ramp_abs_mean'],
        name='Media abs', mode='lines+markers',
        line=dict(color=color, width=2, dash='dot'),
        marker=dict(size=5),
        hovertemplate='Día %{x}<br>Media abs: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    # líneas de umbral en fila 2
    for y_ref in [umbral, -umbral]:
        fig.add_hline(
            y=y_ref,
            line=dict(color='rgba(255,255,0,0.55)', width=1.5, dash='dash'),
            annotation_text=f'±{umbral} MW' if y_ref > 0 else '',
            annotation_position='top left',
            annotation_font=dict(color='rgba(255,255,0,0.80)', size=9),
            row=2, col=1,
        )

    # ── ejes ─────────────────────────────────────────────────────────────────
    fig.update_yaxes(
        title_text='Despacho / Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=1, col=1,
    )
    fig.update_yaxes(
        title_text='Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=2, col=1,
    )
    fig.update_xaxes(
        title_text='Día', tickmode='linear', tick0=1, dtick=1,
        showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        row=2, col=1,
    )

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{zona}</b> · <span style="color:{color}">{tec_label}</span>  '
                f'<sup>{nombre_mes_local} {anio}  |  Día seleccionado: <b>{dia}</b>  |  Umbral: ±{umbral} MW</sup>'
            ),
            font=dict(size=15),
        ),
        hovermode='x unified',
        barmode='overlay',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.03,
            xanchor='right', x=1, bgcolor='rgba(0,0,0,0)',
            font=dict(size=11),
        ),
        height=750, margin=dict(t=90, l=75, r=40, b=60),
    )
    fig.show()

    # ── tabla texto ───────────────────────────────────────────────────────────
    print(f"\n── Rampas sobre umbral ±{umbral} MW · {zona} · {tec_label} · {anio} ──")

    ramp_source = (
        df_ramp_zona_total.loc[fecha_idx_zt.year == anio, zona]
        if tecnologia == 'Todas'
        else df_ramp_zona.loc[mask_anio, (zona, tecnologia)]
    )

    for label, signo in [('🟢 Ramp-UP', 'up'), ('🔴 Ramp-DOWN', 'down')]:
        rf = (ramp_source[ramp_source >=  umbral] if signo == 'up'
              else ramp_source[ramp_source <= -umbral])
        if rf.dropna().empty:
            print(f"\n  {label}: ninguna rampa supera {umbral} MW"); continue
        df_m = pd.DataFrame({
            'RAMP':   rf.abs().values,
            'CUARTO': rf.index.get_level_values('CUARTO').tolist(),
            'DIA':    pd.to_datetime(rf.index.get_level_values('FECHA')).day.tolist(),
        }).dropna()
        daily_max   = (df_m.loc[df_m.groupby('DIA')['RAMP'].idxmax()]
                       .set_index('DIA').sort_index())
        dia_critico = daily_max['RAMP'].idxmax()
        print(f"\n  {label}  ({len(df_m)} eventos sobre {umbral} MW)")
        print(f"  {'─'*45}\n  {'Día':>5}  {'Máx (MW)':>10}  {'Cuarto':>8}")
        print(f"  {'─'*45}")
        for d, row in daily_max.iterrows():
            crit = '  ◄ CRÍTICO' if d == dia_critico else ''
            print(f"  {d:>5}  {row['RAMP']:>10.2f} MW  @ {int(row['CUARTO']):>6} HHMM{crit}")
        print(f"  {'─'*45}")
        print(f"  Promedio máx diario: {daily_max['RAMP'].mean():.2f} MW")

    # ── plantas responsables ──────────────────────────────────────────────────
    if tecnologia == 'Todas':
        print("\n  (Selecciona una tecnología específica para ver plantas responsables)")
        return

    planta_meta = (df[['NOMBRE_PLANTA', 'ZONA', 'TECNOLOGIA']]
                   .drop_duplicates('NOMBRE_PLANTA')
                   .set_index('NOMBRE_PLANTA'))
    plantas_tec = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'ZONA']       == zona
        and planta_meta.loc[p, 'TECNOLOGIA'] == tecnologia
    ]
    if not plantas_tec:
        print("Sin plantas individuales para desglosar."); return

    dia_up   = daily['ramp_up_max'].idxmax()
    dia_down = daily['ramp_down_max'].idxmin()
    fecha_idx_plantas = pd.to_datetime(
        df_pivot_plantas.index.get_level_values('FECHA')
    )

    print(f"\n── Plantas responsables · {zona} · {tecnologia} · {anio} ──")
    for label, d, col_ramp in [
        ('🟢 Ramp-UP  máximo', dia_up,   'ramp_up_max'),
        ('🔴 Ramp-DOWN máximo', dia_down, 'ramp_down_max'),
    ]:
        mask_dp = (fecha_idx_plantas.day == d) & (fecha_idx_plantas.year == anio) & (fecha_idx_plantas.month == mes)
        contrib = {}
        for p in plantas_tec:
            if p not in df_ramp_plantas.columns:
                continue
            ramp_p = df_ramp_plantas.loc[mask_dp, p].dropna()
            if ramp_p.empty:
                continue
            val = (ramp_p[ramp_p > 0].max() if col_ramp == 'ramp_up_max'
                   else ramp_p[ramp_p < 0].min())
            if pd.notna(val):
                contrib[p] = round(val, 4)
        contrib = dict(sorted(contrib.items(), key=lambda x: abs(x[1]), reverse=True))
        print(f"\n  {label} — Día {d} {nombre_mes_local.lower()} {anio}:")
        for p, v in contrib.items():
            print(f"    {p:<35} {v:+.2f} MW")

interact(
    plot_rampa_zona,
    zona       = Dropdown(options=zonas,  description='Zona:',    value=zonas[0]),
    tecnologia = Dropdown(options=tecs,   description='Tec:',     value='Todas'),
    anio       = Dropdown(options=anios,  description='Año:',     value=anios[0]),
    mes        = Dropdown(options=MESES_DISP, description='Mes:', value=MESES_DISP[0]),
    dia        = Dropdown(options=dias,   description='Día:',     value=dias[0]),
    umbral     = BoundedIntText(min=1, max=10000, step=1, value=30, description='Umbral MW:'),
)

interactive(children=(Dropdown(description='Zona:', options=('Zona Central', 'Zona Occidental', 'Zona Occident…

<function __main__.plot_rampa_zona(zona, tecnologia, anio, mes, dia, umbral)>

In [108]:

# ── Índices ───────────────────────────────────────────────────────────────────
fecha_idx_z  = pd.to_datetime(df_ramp_zona.index.get_level_values('FECHA'))
cuarto_idx_z = df_ramp_zona.index.get_level_values('CUARTO')
cols_iter    = df_ramp_zona.columns.tolist()

# ── Construcción del DataFrame ────────────────────────────────────────────────
registros = []

for col in cols_iter:
    zona, tec = col
    for anio in ANIOS_DISP:
      for mes_int in MESES_DISP:
        anio_int = None if anio == 'Todos' else int(anio)
        anio_lbl = 'Todos' if anio == 'Todos' else str(anio)

        cap = _cap_activa_zona_tec(zona, tec, anio_int)

        mask_yr = (np.ones(len(fecha_idx_z), dtype=bool) if anio == 'Todos'
                   else (fecha_idx_z.year == anio_int))
        mask = mask_yr & (fecha_idx_z.month == mes_int)

        base = pd.DataFrame({
            'FECHA':  fecha_idx_z[mask].values,
            'CUARTO': cuarto_idx_z[mask],
            'RAMP':   df_ramp_zona.loc[mask, col].values,
        }).dropna(subset=['RAMP'])

        if base.empty: continue

        fila = {'Zona': zona, 'Tecnologia': tec, 'Año': anio_lbl, 'Mes': mes_int,
                'Cap_MW': round(cap, 1) if cap > 0 else None}

        for direccion, signo in [('UP', 1), ('DOWN', -1)]:
            full_s = base[base['RAMP'] * signo > 0].copy()
            full_s['RAMP'] = full_s['RAMP'].abs()

            if full_s.empty:
                fila.update({k: None for k in [
                    f'Prom_Max_{direccion}_MW',    f'RampMax_{direccion}_MW',
                    f'RampMax_{direccion}_PctCap', f'Rango_Frec_{direccion}',
                    f'Rango_Frec_{direccion}_dias', f'Rango_Frec_{direccion}_PctDias',
                    f'Ventana_Temporal_{direccion}']})
                continue

            daily_max    = full_s.groupby('FECHA')['RAMP'].max().dropna()
            prom         = daily_max.mean()
            ramp_max     = daily_max.max()
            pct_cap      = (ramp_max / cap * 100) if cap > 0 else None
            lo, hi, cnt  = bin_mas_frecuente(daily_max, RAMP_BIN_MW)
            n_dias_total = len(daily_max)
            pct_dias     = (cnt / n_dias_total * 100) if n_dias_total > 0 else None
            dias_en_rango = (daily_max[(daily_max >= lo) & (daily_max < hi)].index
                             if lo is not None else pd.Index([]))

            fila.update({
                f'Prom_Max_{direccion}_MW'        : round(prom, 2),
                f'RampMax_{direccion}_MW'         : round(ramp_max, 2),
                f'RampMax_{direccion}_PctCap'     : round(pct_cap, 1) if pct_cap else None,
                f'Rango_Frec_{direccion}'         : f'{lo:.1f}–{hi:.1f} MW' if lo is not None else 'N/D',
                f'Rango_Frec_{direccion}_dias'    : cnt,
                f'Rango_Frec_{direccion}_PctDias' : round(pct_dias, 1) if pct_dias else None,
                f'Ventana_Temporal_{direccion}'   : rango_temporal_str(full_s, dias_en_rango),
            })

        registros.append(fila)

df_ramp_resumen_zona = pd.DataFrame(registros)[[
    'Zona', 'Tecnologia', 'Año', 'Mes', 'Cap_MW',
    'Prom_Max_UP_MW',   'RampMax_UP_MW',   'RampMax_UP_PctCap',
    'Rango_Frec_UP',    'Rango_Frec_UP_dias',    'Rango_Frec_UP_PctDias',
    'Ventana_Temporal_UP',
    'Prom_Max_DOWN_MW', 'RampMax_DOWN_MW', 'RampMax_DOWN_PctCap',
    'Rango_Frec_DOWN',  'Rango_Frec_DOWN_dias',  'Rango_Frec_DOWN_PctDias',
    'Ventana_Temporal_DOWN',
]]
print(f"df_ramp_resumen_zona: {df_ramp_resumen_zona.shape[0]} filas × {df_ramp_resumen_zona.shape[1]} columnas")

# ── Widget ────────────────────────────────────────────────────────────────────
_col_nombres = {
    'Cap_MW'                   : 'Cap. Instalada (MW)',
    'Prom_Max_UP_MW'           : 'Prom Rampa Máx UP (MW)',
    'RampMax_UP_MW'            : 'Rampa UP Máx (MW)',
    'RampMax_UP_PctCap'        : 'UP Máx / Cap (%)',
    'Rango_Frec_UP'            : 'Rango UP frecuente',
    'Rango_Frec_UP_dias'       : 'Días rango UP',
    'Rango_Frec_UP_PctDias'    : '% días rango UP',
    'Ventana_Temporal_UP'      : 'Ventana UP (P25–P75)',
    'Prom_Max_DOWN_MW'         : 'Prom Rampa Máx DOWN (MW)',
    'RampMax_DOWN_MW'          : 'Rampa DOWN Máx (MW)',
    'RampMax_DOWN_PctCap'      : 'DOWN Máx / Cap (%)',
    'Rango_Frec_DOWN'          : 'Rango DOWN frecuente',
    'Rango_Frec_DOWN_dias'     : 'Días rango DOWN',
    'Rango_Frec_DOWN_PctDias'  : '% días rango DOWN',
    'Ventana_Temporal_DOWN'    : 'Ventana DOWN (P25–P75)',
}
_fmt = {
    'Cap. Instalada (MW)'      : '{:.0f}',
    'Prom Rampa Máx UP (MW)'   : '{:.2f}', 'Rampa UP Máx (MW)'   : '{:.2f}',
    'UP Máx / Cap (%)'         : '{:.1f}%', 'Días rango UP'       : '{:.0f}',
    '% días rango UP'          : '{:.1f}%',
    'Prom Rampa Máx DOWN (MW)' : '{:.2f}', 'Rampa DOWN Máx (MW)' : '{:.2f}',
    'DOWN Máx / Cap (%)'       : '{:.1f}%', 'Días rango DOWN'     : '{:.0f}',
    '% días rango DOWN'        : '{:.1f}%',
}

def mostrar_rampas_zona(anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    subset = (
        df_ramp_resumen_zona[(df_ramp_resumen_zona['Año'] == str(anio)) & (df_ramp_resumen_zona['Mes'] == mes)]
        .set_index(['Zona', 'Tecnologia'])
        .drop(columns=['Año'])
        .rename(columns=_col_nombres)
    )
    display(
        subset.style
        .format(_fmt, na_rep='N/D')
        .set_caption(f'Rampas por Zona/Tecnología — {anio}'
                     + (' (Cap = promedio años activos)' if anio == 'Todos'
                        else ' (Cap = plantas activas ese año)'))
        .background_gradient(subset=['UP Máx / Cap (%)', 'DOWN Máx / Cap (%)'],
                             cmap='RdYlGn_r')
    )

_anios_w = ['Todos'] + sorted([a for a in df_ramp_resumen_zona['Año'].unique() if a != 'Todos'])
interact(mostrar_rampas_zona, anio=Dropdown(options=_anios_w, description='Año:', value='Todos'),
         mes=Dropdown(options=MESES_DISP, description='Mes:', value=MESES_DISP[0]))

df_ramp_resumen_zona: 264 filas × 19 columnas


interactive(children=(Dropdown(description='Año:', options=('Todos', '2023'), value='Todos'), Dropdown(descrip…

<function __main__.mostrar_rampas_zona(anio, mes)>

In [109]:

# ── DATOS BASE ────────────────────────────────────────────────────────────────
columnas    = df_pivot_zona_tec.columns.tolist()
cols_labels = [f'{z} — {t}' for z, t in columnas]

# ── CACHE CURVA ───────────────────────────────────────────────────────────────
def calcular_stats(col, anio='Todos', mes=None):
    zona, tec = col
    label = f'{zona}_{tec}'
    plantas_col = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'ZONA']       == zona
        and planta_meta.loc[p, 'TECNOLOGIA'] == tec
    ]
    cap_total = sum(inst_mw_final.get(p, 0) for p in plantas_col)
    temp = df_pivot_zona_tec[col].reset_index()
    temp.columns = ['FECHA', 'CUARTO', label]
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])
    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]
    if temp.empty:
        return None
    s = temp.groupby('CUARTO')[label].agg(
        Minima     = 'min',
        P5         = lambda x: x.quantile(0.05),
        Moda_baja  = moda_baja_fn,
        Mediana    = lambda x: x.quantile(0.50),
        Moda_alta  = moda_alta_fn,
        P95        = lambda x: x.quantile(0.95),
        Maxima     = 'max',
        N_dias     = 'count',
        Pct_activo = lambda x: round((x > 0).sum() / len(x) * 100, 1),
    ).round(4)
    s['cap_total'] = cap_total
    return s

stats_cache = {}
for col in columnas:
    for anio in ANIOS_DISP:
        try:
            stats_cache[(col, anio)] = calcular_stats(col, anio)
        except Exception:
            stats_cache[(col, anio)] = None

# ── FUNCIÓN CURVA ─────────────────────────────────────────────────────────────
def plot_curva(col, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    _ck = (col, anio, mes)
    if _ck not in stats_cache:
        try:
            stats_cache[_ck] = calcular_stats(col, anio, mes)
        except Exception:
            stats_cache[_ck] = None

    stats = stats_cache[_ck]
    if stats is None:
        print(f"Sin datos para curva — {anio}")
        return

    zona, tec  = col
    color      = tec_color.get(tec, '#888780')
    r, g, b    = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    anio_label = str(anio) if anio != 'Todos' else 'Histórico'
    cap        = stats['cap_total'].iloc[0]
    cap_txt    = f'{cap:.0f}' if cap > 0 else 'N/D'

    def pct(v): return f'{v/cap*100:.1f}%' if cap > 0 else 'N/D'

    max_mw       = stats['Maxima'].max()
    moda_alta_mw = stats['Moda_alta'].max()
    med_mw       = stats['Mediana'].max()
    moda_baja_mw = stats['Moda_baja'].max()
    min_mw       = stats['Minima'][stats['Minima'] > 0].min() if (stats['Minima'] > 0).any() else 0
    n_dias       = int(stats['N_dias'].max())
    pct_act      = round(stats['Pct_activo'].mean(), 1)

    x     = stats.index.tolist()
    x_rev = x[::-1]
    fig   = go.Figure()

    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Maxima'].tolist() + stats['Minima'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.05)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Mín–Máx',
    ))
    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Maxima'].tolist() + stats['Moda_alta'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.20)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Moda alta–Máx',
    ))
    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Moda_baja'].tolist() + stats['Minima'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.20)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Mín–Moda baja',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Minima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Mínima  {min_mw:.1f} MW ({pct(min_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Mínima: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_baja'],
        line=dict(color=f'rgba({r},{g},{b},0.6)', width=1.5, dash='dashdot'),
        name=f'Moda baja  {moda_baja_mw:.1f} MW ({pct(moda_baja_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda baja: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Mediana'],
        line=dict(color=color, width=2.5),
        name=f'Mediana  {med_mw:.1f} MW ({pct(med_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Mediana: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_alta'],
        line=dict(color=f'rgba({r},{g},{b},0.6)', width=1.5, dash='dashdot'),
        name=f'Moda alta  {moda_alta_mw:.1f} MW ({pct(moda_alta_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda alta: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Maxima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Máxima  {max_mw:.1f} MW ({pct(max_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Máxima: %{y:.2f} MW<extra></extra>',
    ))

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{zona}</b>  '
                f'<span style="color:{color}">▌ {tec}</span>  '
                f'<sup>· {anio_label}</sup><br>'
                f'<sup>'
                f'Cap. instalada: <b>{cap_txt} MW</b>  |  '
                f'Máx: <b>{max_mw:.1f} MW ({pct(max_mw)})</b>  |  '
                f'Moda alta: <b>{moda_alta_mw:.1f} MW ({pct(moda_alta_mw)})</b>  |  '
                f'Mediana: <b>{med_mw:.1f} MW ({pct(med_mw)})</b>  |  '
                f'Moda baja: <b>{moda_baja_mw:.1f} MW ({pct(moda_baja_mw)})</b>  |  '
                f'Mín: <b>{min_mw:.1f} MW ({pct(min_mw)})</b>  |  '
                f'Días: <b>{n_dias}</b>  |  '
                f'% activo: <b>{pct_act:.1f}%</b>'
                f'</sup>'
            ),
            font=dict(size=13),
        ),
        xaxis=dict(
            title='Cuarto del día', tickmode='array',
            tickvals=x[::8], ticktext=[str(v) for v in x[::8]],
            showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        ),
        yaxis=dict(title='Despacho (MW)', showgrid=True,
                   gridcolor='rgba(255,255,255,0.06)', zeroline=False),
        hovermode='x unified',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)', font=dict(size=11)),
        height=560, margin=dict(t=120, l=60, r=40, b=60),
    )
    fig.show()

# ── FUNCIÓN HISTOGRAMA ────────────────────────────────────────────────────────
# ── FUNCIÓN HISTOGRAMA ────────────────────────────────────────────────────────
def plot_histograma(col, cuarto, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    zona, tec = col
    if col not in df_pivot_zona_tec.columns:
        print(f"No existe combinación {zona} — {tec}")
        return

    temp = df_pivot_zona_tec[col].to_frame('DESPACHO').reset_index()
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])

    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]

    if cuarto == 'Día entero':
        subset     = temp
        cuarto_txt = 'Día entero'
    elif cuarto == 'Solo día':
        subset     = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        cuarto_txt = f'Solo día ({CUARTO_AMANECER}–{CUARTO_ATARDECER})'
    elif cuarto == 'Solo noche':
        subset     = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        cuarto_txt = 'Solo noche'
    else:
        subset     = temp[temp['CUARTO'] == cuarto]
        cuarto_txt = f'Cuarto {cuarto}'

    # ── máximo diario + cuarto donde ocurrió ─────────────────────────────────
    subset_clean     = subset.dropna(subset=['DESPACHO'])
    daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
    idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
    daily_cuarto_max = (
        subset_clean.loc[idx_max.values]
        .set_index('FECHA')['CUARTO']
        .rename('CUARTO_MAX')
    )
    daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
    valores = daily['MAX_MW']

    if valores.empty or valores.max() == 0:
        print(f"Sin generación para {zona} — {tec} · {cuarto_txt}")
        return

    anio_label = str(anio) if anio != 'Todos' else 'Histórico'
    color      = tec_color.get(tec, '#888780')
    n_dias     = len(valores)
    mediana    = valores.median()

    # ── BINS FIJOS de 50 MW ───────────────────────────────────────────────────
    BIN_WIDTH = 50
    bin_start = np.floor(valores.min() / BIN_WIDTH) * BIN_WIDTH
    bin_end   = np.ceil(valores.max()  / BIN_WIDTH) * BIN_WIDTH
    bin_edges = np.arange(bin_start, bin_end + BIN_WIDTH, BIN_WIDTH)

    counts, _   = np.histogram(valores, bins=bin_edges)
    bin_idx_max = counts.argmax()
    rango_lo    = bin_edges[bin_idx_max]
    rango_hi    = bin_edges[bin_idx_max + 1]
    dias_rango  = counts[bin_idx_max]

    # ── Top franjas de 2h dentro del rango más frecuente ─────────────────────
    en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()

    if not en_rango.empty:
        en_rango['FRANJA_2H'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
        top_franjas = (
            en_rango['FRANJA_2H']
            .value_counts()
            .head(5)
            .reset_index()
        )
        top_franjas.columns = ['FRANJA', 'DIAS']
        top_txt_grafico = '  |  '.join(
            [f'{r.FRANJA} ({r.DIAS}d)' for _, r in top_franjas.iterrows()]
        )
        top_txt_print = '\n'.join(
            [f'  {r.FRANJA}  →  {r.DIAS:>3} días  ({r.DIAS / dias_rango * 100:.1f}% del rango frecuente)'
             for _, r in top_franjas.iterrows()]
        )
    else:
        top_txt_grafico = 'N/D'
        top_txt_print   = '  N/D'

    # ── Histograma ────────────────────────────────────────────────────────────
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=valores,
        xbins=dict(start=bin_start, end=bin_end, size=BIN_WIDTH),
        marker=dict(color='rgba(120,140,255,0.75)', line=dict(color='#1a1a1a', width=1)),
        hovertemplate='Despacho: %{x} MW<br>Días: %{y}<extra></extra>',
        name='Días',
    ))

    fig.update_layout(
        title=(
            f'<span style="color:{color}">▌</span> <b>{zona} — {tec}</b> · '
            f'{cuarto_txt} · {anio_label}  ({n_dias} días · máximo diario)'
        ),
    )

    fig.add_vrect(
        x0=rango_lo, x1=rango_hi,
        fillcolor=color, opacity=0.25, line_width=2, line_color=color,
        annotation_text=(
            f'<b>Rango más frecuente</b><br>'
            f'{rango_lo:.0f}–{rango_hi:.0f} MW · {dias_rango} días ({dias_rango/n_dias*100:.1f}%)<br>'
            f'{top_txt_grafico}'
        ),
        annotation_position='top',
        annotation_font=dict(size=11, color=color),
    )
    fig.add_vline(
        x=valores.min(), line_color='#378ADD', line_width=1.5, line_dash='dot',
        annotation_text=f'Mín: {valores.min():.1f} MW',
        annotation_position='bottom left', annotation_font=dict(size=10, color='#7ab8f5'),
    )
    fig.add_vline(
        x=mediana, line_color='orange', line_width=2,
        annotation_text=f'Mediana: {mediana:.1f} MW',
        annotation_position='top right',
        annotation_font=dict(size=10, color='orange'),
    )
    fig.add_vline(
        x=valores.max(), line_color='#e74c3c', line_width=1.5, line_dash='dot',
        annotation_text=f'Máx: {valores.max():.1f} MW',
        annotation_position='bottom right', annotation_font=dict(size=10, color='#e74c3c'),
    )

    fig.update_layout(
        xaxis=dict(
            title='Máximo diario de despacho (MW)',
            tickmode='array', tickvals=bin_edges,
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        ),
        yaxis=dict(title='Días', showgrid=True, gridcolor='rgba(255,255,255,0.06)'),
        bargap=0.05,
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        margin=dict(t=100, l=60, r=40, b=60), height=440,
        showlegend=False,
    )
    fig.show()

    print(f"{'─'*52}")
    print(f"Zona: {zona}  |  Tecnología: {tec}  |  {cuarto_txt}  |  {anio_label}")
    print(f"{'─'*52}")
    print(f"Mínimo histórico:    {valores.min():.4f} MW")
    print(f"Mediana:             {mediana:.4f} MW")
    print(f"Máximo histórico:    {valores.max():.4f} MW")
    print(f"Desv. estándar:      {valores.std():.4f} MW")
    print(f"{'─'*52}")
    print(f"Rango más frecuente: {rango_lo:.0f} – {rango_hi:.0f} MW  →  {dias_rango} días ({dias_rango/n_dias*100:.1f}%)")
    print(f"{'─'*52}")
    print(f"Top franjas horarias en el rango más frecuente:")
    print(top_txt_print)

# ── FUNCIÓN WRAPPER ───────────────────────────────────────────────────────────
def plot_zona_tec_completo(col_label, cuarto, anio, mes):
    idx = cols_labels.index(col_label)
    col = columnas[idx]
    plot_curva(col, anio, mes)
    plot_histograma(col, cuarto, anio, mes)

# ── WIDGETS ───────────────────────────────────────────────────────────────────

interact(
    plot_zona_tec_completo,
    col_label = Dropdown(options=cols_labels,    description='Zona/Tec:', value=cols_labels[0]),
    cuarto    = Dropdown(options=CUARTOS_DISP, description='Cuarto:',  value='Solo día'),
    anio      = Dropdown(options=ANIOS_DISP,     description='Año:',     value='Todos'),
    mes       = Dropdown(options=MESES_DISP,    description='Mes:',     value=MESES_DISP[0]),
)

interactive(children=(Dropdown(description='Zona/Tec:', options=('Zona Central — Eolica', 'Zona Central — Hidr…

<function __main__.plot_zona_tec_completo(col_label, cuarto, anio, mes)>

In [110]:

def build_tabla_zona(cuarto, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    anio_int = None if anio == 'Todos' else int(anio)
    rows = []

    for col in df_pivot_zona_tec.columns:
        zona, tec = col

        # ── Capacidad activa para ese año ─────────────────────────────────────
        cap = _cap_activa_zona_tec(zona, tec, anio_int)

        temp = df_pivot_zona_tec[col].to_frame('DESPACHO').reset_index()
        temp['FECHA'] = pd.to_datetime(temp['FECHA'])
        if anio != 'Todos':
            temp = temp[temp['FECHA'].dt.year == anio_int]
        temp = temp[temp['FECHA'].dt.month == mes]

        if cuarto == 'Día entero':
            subset = temp
        elif cuarto == 'Solo día':
            subset = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        elif cuarto == 'Solo noche':
            subset = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        else:
            subset = temp[temp['CUARTO'] == cuarto]

        subset_clean     = subset.dropna(subset=['DESPACHO'])
        daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
        idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
        daily_cuarto_max = subset_clean.loc[idx_max.values].set_index('FECHA')['CUARTO'].rename('CUARTO_MAX')
        daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
        valores = daily['MAX_MW']
        if valores.empty or valores.max() == 0:
            continue

        # ── bins automáticos ──────────────────────────────────────────────────
        BIN_WIDTH = calcular_bin(valores)
        bin_edges = np.arange(np.floor(valores.min()/BIN_WIDTH)*BIN_WIDTH,
                              np.ceil(valores.max()/BIN_WIDTH)*BIN_WIDTH + BIN_WIDTH, BIN_WIDTH)
        counts, _ = np.histogram(valores, bins=bin_edges)
        idx_bin   = counts.argmax()
        rango_lo  = bin_edges[idx_bin]
        rango_hi  = bin_edges[idx_bin + 1]

        en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()
        if not en_rango.empty:
            en_rango['FRANJA'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
            top         = en_rango['FRANJA'].value_counts().head(3)
            cuarto_pico = '  |  '.join([f'{f}({n}d)' for f, n in top.items()])
        else:
            cuarto_pico = 'N/D'

        mediana   = valores.median()
        max_hist  = valores.max()
        moda_alta = (rango_lo + rango_hi) / 2

        moda_baja_vals = valores[valores < moda_alta]
        if not moda_baja_vals.empty:
            bw2 = calcular_bin(moda_baja_vals)
            be2 = np.arange(np.floor(moda_baja_vals.min()/bw2)*bw2,
                            np.ceil(moda_baja_vals.max()/bw2)*bw2 + bw2, bw2)
            if len(be2) > 1:
                c2, _     = np.histogram(moda_baja_vals, bins=be2)
                i2        = c2.argmax()
                moda_baja = (be2[i2] + be2[i2+1]) / 2
            else:
                moda_baja = moda_baja_vals.min()
        else:
            moda_baja = valores.min()

        n_dias            = len(valores)
        dias_zona_baja    = (valores <= moda_baja).sum()
        dias_zona_central = ((valores > moda_baja) & (valores < moda_alta)).sum()
        dias_zona_alta    = (valores >= moda_alta).sum()

        rows.append({
            'ZONA':                zona,
            'TECNOLOGIA':          tec,
            'CAP_MW':              round(cap, 1) if cap > 0 else None,
            'UTIL_TIPICA_%':       round(mediana  / cap * 100, 1) if cap > 0 else None,
            'UTIL_MAX_%':          round(max_hist / cap * 100, 1) if cap > 0 else None,
            'BRECHA_MW':           round(cap - moda_alta, 1)      if cap > 0 else None,
            'DIAS_ZONA_BAJA_%':    round(dias_zona_baja    / n_dias * 100, 1),
            'DIAS_ZONA_CENTRAL_%': round(dias_zona_central / n_dias * 100, 1),
            'DIAS_ZONA_ALTA_%':    round(dias_zona_alta    / n_dias * 100, 1),
            'CUARTO_PICO':         cuarto_pico,
            'N_DIAS':              n_dias,
        })

    df_out = pd.DataFrame(rows).set_index(['ZONA', 'TECNOLOGIA'])
    cap_nota = '(prom años activos)' if anio == 'Todos' else ''
    df_out.columns.name = f'Zona/Tecnología — {anio}  |  CAP_MW {cap_nota}'
    display(df_out)


interact(
    build_tabla_zona,
    cuarto = Dropdown(options=CUARTOS_DISP, description='Cuarto:', value='Día entero'),
    anio   = Dropdown(options=ANIOS_DISP,        description='Año:',   value='Todos'),
    mes    = Dropdown(options=MESES_DISP,    description='Mes:',   value=MESES_DISP[0])
)

interactive(children=(Dropdown(description='Cuarto:', options=('Día entero', 'Solo día', 'Solo noche', 15, 30,…

<function __main__.build_tabla_zona(cuarto, anio, mes)>

## 8c — Análisis por Nodo


In [111]:
# ── Rampas mensuales + comportamiento diario por NODO ─────────────────────────
# Requiere: df, df_pivot_nodo_tec2, df_pivot_plantas, df_ramp_plantas,
#           tec_color

# ── pivots y rampas ───────────────────────────────────────────────────────────
cuarto_idx   = df_pivot_nodo_tec2.index.get_level_values('CUARTO')
fecha_idx    = pd.to_datetime(df_pivot_nodo_tec2.index.get_level_values('FECHA'))
df_ramp_nodo       = df_pivot_nodo_tec2.diff()
fecha_change_nodo  = pd.Series(fecha_idx).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
df_ramp_nodo[fecha_change_nodo.values] = np.nan

cuarto_idx_plt  = df_pivot_plantas.index.get_level_values('CUARTO')
fecha_idx_plt   = pd.to_datetime(df_pivot_plantas.index.get_level_values('FECHA'))
df_ramp_plantas    = df_pivot_plantas.diff()
fecha_change_plt   = pd.Series(fecha_idx_plt).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
df_ramp_plantas[fecha_change_plt.values] = np.nan

planta_nodo = df[['NOMBRE_PLANTA', 'NODO']].drop_duplicates().set_index('NOMBRE_PLANTA')

nodos = sorted(set(n for n, t in df_pivot_nodo_tec2.columns))
tecs  = sorted(set(t for n, t in df_pivot_nodo_tec2.columns))
anios = sorted(fecha_idx.year.unique())
dias  = list(range(1, 32))

# ── función principal ─────────────────────────────────────────────────────────
def plot_rampa_nodo(nodo, tecnologia, anio, mes, dia, umbral):
    nombre_mes_local = MESES_NOMBRES[mes]

    col = (nodo, tecnologia)
    if col not in df_pivot_nodo_tec2.columns:
        print(f"No existe {nodo} — {tecnologia}"); return

    color      = tec_color.get(tecnologia, '#888780')
    fill_color = hex_to_rgba(color, 0.18)
    mask_anio   = (fecha_idx.year == anio) & (fecha_idx.month == mes)
    fecha_anio  = fecha_idx[mask_anio]
    cuarto_anio = cuarto_idx[mask_anio]
    _sub54      = df_pivot_nodo_tec2.loc[mask_anio]
    _f54        = pd.to_datetime(_sub54.index.get_level_values('FECHA'))
    _fc54       = pd.Series(_f54).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
    _ramp54     = _sub54.diff()
    _ramp54[_fc54.values] = np.nan

    ramp_df = pd.DataFrame({
        'DIA':  fecha_anio.day,
        'RAMP': _ramp54[col].values,
    }).dropna()

    if ramp_df.empty:
        print(f"Sin datos para {nodo} — {tecnologia} — {anio}"); return

    daily = ramp_df.groupby('DIA')['RAMP'].agg(
        ramp_up_max   = lambda x: x[x > 0].max() if (x > 0).any() else 0,
        ramp_down_max = lambda x: x[x < 0].min() if (x < 0).any() else 0,
        ramp_abs_mean = lambda x: x.abs().mean(),
    ).round(4)

    # ── datos del día seleccionado ────────────────────────────────────────────
    mask_dia    = fecha_anio.day == dia
    cuartos_d   = cuarto_anio[mask_dia]

    desp_dia = pd.Series(
        df_pivot_nodo_tec2.loc[mask_anio, col].values[mask_dia],
        index=cuartos_d,
    ).dropna()

    ramp_dia = pd.Series(
        _ramp54[col].values[mask_dia],
        index=cuartos_d,
    ).dropna()

    cuartos_up   = ramp_dia[ramp_dia >=  umbral].index
    cuartos_down = ramp_dia[ramp_dia <= -umbral].index

    # ── figura 2 filas ────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.52, 0.48],
        vertical_spacing=0.10,
        subplot_titles=(
            f'Comportamiento — Día {dia} {nombre_mes_local.lower()} {anio}  |  umbral ±{umbral} MW',
            f'Rampas diarias — {nombre_mes_local} {anio}  |  umbral ±{umbral} MW',
        ),
    )

    # ══ FILA 1: comportamiento intradiario ════════════════════════════════════
    if not desp_dia.empty:

        fig.add_trace(go.Scatter(
            x=desp_dia.index, y=desp_dia.round(2).values,
            name='Despacho', mode='lines',
            line=dict(color=color, width=2.2),
            fill='tozeroy', fillcolor=fill_color,
            hovertemplate='Cuarto %{x} | <b>%{y:.1f} MW</b><extra></extra>',
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=ramp_dia.index, y=ramp_dia.round(2).values,
            name='Rampa (MW/15min)', mode='lines',
            line=dict(color='rgba(255,255,255,0.40)', width=1.2, dash='dot'),
            hovertemplate='Rampa: %{y:+.1f} MW<extra></extra>',
        ), row=1, col=1)

        if len(cuartos_up) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_up,
                y=desp_dia.reindex(cuartos_up).values,
                name=f'Ramp-UP ≥{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-up', size=12,
                    color='#2ecc71',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_up).values],
                hovertemplate='Ramp-UP cuarto %{x}<br><b>+%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        if len(cuartos_down) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_down,
                y=desp_dia.reindex(cuartos_down).values,
                name=f'Ramp-DOWN ≤-{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-down', size=12,
                    color='#e74c3c',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_down).values],
                hovertemplate='Ramp-DOWN cuarto %{x}<br><b>%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        tick_vals_d = list(desp_dia.index)[::8]
        fig.update_xaxes(
            tickmode='array', tickvals=tick_vals_d,
            ticktext=[str(v) for v in tick_vals_d],
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
            row=1, col=1,
        )
    else:
        fig.add_annotation(
            text=f'Sin datos para el día {dia}',
            xref='paper', yref='paper', x=0.5, y=0.75,
            showarrow=False, font=dict(color='#888', size=13),
        )

    # ══ FILA 2: barras mensuales ══════════════════════════════════════════════
    bar_colors_up = [
        'rgba(46,204,113,1.0)' if d == dia else 'rgba(46,204,113,0.55)'
        for d in daily.index
    ]
    bar_border_up = [
        '#ffffff' if d == dia else 'rgba(0,0,0,0)'
        for d in daily.index
    ]
    bar_colors_dn = [
        'rgba(231,76,60,1.0)' if d == dia else 'rgba(231,76,60,0.55)'
        for d in daily.index
    ]
    bar_border_dn = [
        '#ffffff' if d == dia else 'rgba(0,0,0,0)'
        for d in daily.index
    ]

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_up_max'],
        name='Ramp-up máx',
        marker=dict(color=bar_colors_up, line=dict(color=bar_border_up, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-up máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_down_max'],
        name='Ramp-down máx',
        marker=dict(color=bar_colors_dn, line=dict(color=bar_border_dn, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-down máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=daily.index, y=daily['ramp_abs_mean'],
        name='Media abs', mode='lines+markers',
        line=dict(color=color, width=2, dash='dot'),
        marker=dict(size=5),
        hovertemplate='Día %{x}<br>Media abs: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    for y_ref in [umbral, -umbral]:
        fig.add_hline(
            y=y_ref,
            line=dict(color='rgba(255,255,0,0.55)', width=1.5, dash='dash'),
            annotation_text=f'±{umbral} MW' if y_ref > 0 else '',
            annotation_position='top left',
            annotation_font=dict(color='rgba(255,255,0,0.80)', size=9),
            row=2, col=1,
        )

    # ── ejes ─────────────────────────────────────────────────────────────────
    fig.update_yaxes(
        title_text='Despacho / Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=1, col=1,
    )
    fig.update_yaxes(
        title_text='Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=2, col=1,
    )
    fig.update_xaxes(
        title_text='Día', tickmode='linear', tick0=1, dtick=1,
        showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        row=2, col=1,
    )

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{nodo}</b> · <span style="color:{color}">{tecnologia}</span>  '
                f'<sup>{nombre_mes_local} {anio}  |  Día seleccionado: <b>{dia}</b>  |  Umbral: ±{umbral} MW</sup>'
            ),
            font=dict(size=15),
        ),
        hovermode='x unified',
        barmode='overlay',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.03,
            xanchor='right', x=1, bgcolor='rgba(0,0,0,0)',
            font=dict(size=11),
        ),
        height=750, margin=dict(t=90, l=75, r=40, b=60),
    )
    fig.show()

    # ── plantas responsables ──────────────────────────────────────────────────
    plantas_nodo = [
        p for p in df_pivot_plantas.columns
        if p in planta_nodo.index
        and planta_nodo.loc[p, 'NODO'] == nodo
    ]
    plantas_tec = [
        p for p in plantas_nodo
        if df[df['NOMBRE_PLANTA'] == p]['TECNOLOGIA'].iloc[0] == tecnologia
    ] if plantas_nodo else []

    if not plantas_tec:
        print("Sin plantas individuales para desglosar."); return

    dia_up   = daily['ramp_up_max'].idxmax()
    dia_down = daily['ramp_down_max'].idxmin()
    fecha_idx_plantas = pd.to_datetime(
        df_pivot_plantas.index.get_level_values('FECHA')
    )

    print(f"\n── Plantas responsables · {nodo} · {tecnologia} · {anio} ──")
    for label, d, col_ramp in [
        ('🟢 Ramp-UP  máximo', dia_up,   'ramp_up_max'),
        ('🔴 Ramp-DOWN máximo', dia_down, 'ramp_down_max'),
    ]:
        mask_dp = (fecha_idx_plantas.day == d) & (fecha_idx_plantas.year == anio) & (fecha_idx_plantas.month == mes)
        contrib = {}
        for p in plantas_tec:
            if p not in df_ramp_plantas.columns:
                continue
            ramp_p = df_ramp_plantas.loc[mask_dp, p].dropna()
            if ramp_p.empty:
                continue
            val = (ramp_p[ramp_p > 0].max() if col_ramp == 'ramp_up_max'
                   else ramp_p[ramp_p < 0].min())
            if pd.notna(val):
                contrib[p] = round(val, 4)
        contrib = dict(sorted(contrib.items(), key=lambda x: abs(x[1]), reverse=True))
        print(f"\n  {label} — Día {d} {nombre_mes_local.lower()} {anio}:")
        for p, v in contrib.items():
            print(f"    {p:<35} {v:+.2f} MW")

    # ── tabla rampas sobre umbral ─────────────────────────────────────────────
    print(f"\n── Rampas sobre umbral ±{umbral} MW · {nodo} · {tecnologia} · {anio} ──")

    ramp_col = _ramp54[col]

    for label, signo in [('🟢 Ramp-UP', 'up'), ('🔴 Ramp-DOWN', 'down')]:
        rf = (ramp_col[ramp_col >=  umbral] if signo == 'up'
              else ramp_col[ramp_col <= -umbral])
        if rf.dropna().empty:
            print(f"\n  {label}: ninguna rampa supera {umbral} MW"); continue
        df_m = pd.DataFrame({
            'RAMP':   rf.abs().values,
            'CUARTO': rf.index.get_level_values('CUARTO').tolist(),
            'DIA':    pd.to_datetime(rf.index.get_level_values('FECHA')).day.tolist(),
        }).dropna()
        daily_max   = (df_m.loc[df_m.groupby('DIA')['RAMP'].idxmax()]
                       .set_index('DIA').sort_index())
        dia_critico = daily_max['RAMP'].idxmax()
        n_eventos   = len(df_m)
        print(f"\n  {label}  ({n_eventos} eventos sobre {umbral} MW)")
        print(f"  {'─'*45}\n  {'Día':>5}  {'Máx (MW)':>10}  {'Cuarto':>8}")
        print(f"  {'─'*45}")
        for d, row in daily_max.iterrows():
            crit = '  ◄ CRÍTICO' if d == dia_critico else ''
            print(f"  {d:>5}  {row['RAMP']:>10.2f} MW  @ {int(row['CUARTO']):>6} HHMM{crit}")
        print(f"  {'─'*45}")
        print(f"  Promedio máx diario: {daily_max['RAMP'].mean():.2f} MW")

interact(
    plot_rampa_nodo,
    nodo       = Dropdown(options=nodos, description='Nodo:',    value=nodos[0]),
    tecnologia = Dropdown(options=tecs,  description='Tec:',     value=tecs[0]),
    anio       = Dropdown(options=anios, description='Año:',     value=anios[0]),
    mes        = Dropdown(options=MESES_DISP, description='Mes:', value=MESES_DISP[0]),
    dia        = Dropdown(options=dias,  description='Día:',     value=dias[0]),
    umbral     = BoundedIntText(min=1, max=10000, step=1, value=30, description='Umbral MW:'),
)

interactive(children=(Dropdown(description='Nodo:', options=('ANT', 'BAY', 'BLM', 'BOQIII', 'BVI', 'CAL', 'CAT…

<function __main__.plot_rampa_nodo(nodo, tecnologia, anio, mes, dia, umbral)>

In [112]:
# ── Análisis por Nodo/Tecnología — curva + histograma ────────────────────────
# Requiere: df_pivot_nodo_tec2, df_pivot_plantas, planta_meta,
#           inst_mw_final, tec_color, ANIOS_DISP, CUARTOS_DISP,
#           CUARTO_AMANECER, CUARTO_ATARDECER, cuarto_a_franja_2h

columnas_nodo    = df_pivot_nodo_tec2.columns.tolist()
cols_labels_nodo = [f'{n} — {t}' for n, t in columnas_nodo]

# ── CACHE ─────────────────────────────────────────────────────────────────────
def calcular_stats_nodo(col, anio='Todos', mes=None):
    nodo, tec   = col
    label       = f'{nodo}_{tec}'
    plantas_col = [
        p for p in df_pivot_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'NODO']       == nodo
        and planta_meta.loc[p, 'TECNOLOGIA'] == tec
    ]
    cap_total = sum(inst_mw_final.get(p, 0) for p in plantas_col)
    temp = df_pivot_nodo_tec2[col].to_frame(label).reset_index()
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])
    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]
    if temp.empty:
        return None
    s = temp.groupby('CUARTO')[label].agg(
        Minima     = 'min',
        Moda_baja  = moda_baja_fn,
        Mediana    = lambda x: x.quantile(0.50),
        Moda_alta  = moda_alta_fn,
        Maxima     = 'max',
        N_dias     = 'count',
        Pct_activo = lambda x: round((x > 0).sum() / len(x) * 100, 1),
    ).round(4)
    s['cap_total'] = cap_total
    return s

stats_cache_nodo = {}
for col in columnas_nodo:
    for anio in ANIOS_DISP:
        try:
            stats_cache_nodo[(col, anio, None)] = calcular_stats_nodo(col, anio)
        except Exception:
            stats_cache_nodo[(col, anio, None)] = None

# ── FUNCIÓN CURVA ─────────────────────────────────────────────────────────────
def plot_curva_nodo(col, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    _ck = (col, anio, mes)
    if _ck not in stats_cache_nodo:
        try:
            stats_cache_nodo[_ck] = calcular_stats_nodo(col, anio, mes)
        except Exception:
            stats_cache_nodo[_ck] = None

    stats = stats_cache_nodo[_ck]
    if stats is None:
        print(f"Sin datos para curva — {anio}")
        return

    nodo, tec  = col
    color      = tec_color.get(tec, '#888780')
    r, g, b    = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    anio_label = str(anio) if anio != 'Todos' else 'Histórico'
    cap        = stats['cap_total'].iloc[0]
    cap_txt    = f'{cap:.0f}' if cap > 0 else 'N/D'

    def pct(v): return f'{v/cap*100:.1f}%' if cap > 0 else 'N/D'

    max_mw       = stats['Maxima'].max()
    moda_alta_mw = stats['Moda_alta'].max()
    med_mw       = stats['Mediana'].max()
    moda_baja_mw = stats['Moda_baja'].max()
    min_mw       = stats['Minima'][stats['Minima'] > 0].min() if (stats['Minima'] > 0).any() else 0
    n_dias       = int(stats['N_dias'].max())
    pct_act      = round(stats['Pct_activo'].mean(), 1)

    x     = stats.index.tolist()
    x_rev = x[::-1]
    fig   = go.Figure()

    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Maxima'].tolist() + stats['Minima'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.05)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Mín–Máx',
    ))
    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Maxima'].tolist() + stats['Moda_alta'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.20)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Moda alta–Máx',
    ))
    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Moda_baja'].tolist() + stats['Minima'].tolist()[::-1],
        fill='toself', hoverinfo='skip', fillcolor=f'rgba({r},{g},{b},0.20)',
        line=dict(color='rgba(0,0,0,0)'), name='Rango Mín–Moda baja',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Minima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Mínima  {min_mw:.1f} MW ({pct(min_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Mínima: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_baja'],
        line=dict(color=f'rgba({r},{g},{b},0.6)', width=1.5, dash='dashdot'),
        name=f'Moda baja  {moda_baja_mw:.1f} MW ({pct(moda_baja_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda baja: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Mediana'],
        line=dict(color=color, width=2.5),
        name=f'Mediana  {med_mw:.1f} MW ({pct(med_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Mediana: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_alta'],
        line=dict(color=f'rgba({r},{g},{b},0.6)', width=1.5, dash='dashdot'),
        name=f'Moda alta  {moda_alta_mw:.1f} MW ({pct(moda_alta_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda alta: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Maxima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Máxima  {max_mw:.1f} MW ({pct(max_mw)})',
        hovertemplate='<b>Cuarto %{x}</b><br>Máxima: %{y:.2f} MW<extra></extra>',
    ))

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{nodo}</b>  '
                f'<span style="color:{color}">▌ {tec}</span>  '
                f'<sup>· {anio_label}</sup><br>'
                f'<sup>'
                f'Cap. instalada: <b>{cap_txt} MW</b>  |  '
                f'Máx: <b>{max_mw:.1f} MW ({pct(max_mw)})</b>  |  '
                f'Moda alta: <b>{moda_alta_mw:.1f} MW ({pct(moda_alta_mw)})</b>  |  '
                f'Mediana: <b>{med_mw:.1f} MW ({pct(med_mw)})</b>  |  '
                f'Moda baja: <b>{moda_baja_mw:.1f} MW ({pct(moda_baja_mw)})</b>  |  '
                f'Mín: <b>{min_mw:.1f} MW ({pct(min_mw)})</b>  |  '
                f'Días: <b>{n_dias}</b>  |  '
                f'% activo: <b>{pct_act:.1f}%</b>'
                f'</sup>'
            ),
            font=dict(size=13),
        ),
        xaxis=dict(
            title='Cuarto del día', tickmode='array',
            tickvals=x[::8], ticktext=[str(v) for v in x[::8]],
            showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        ),
        yaxis=dict(title='Despacho (MW)', showgrid=True,
                   gridcolor='rgba(255,255,255,0.06)', zeroline=False),
        hovermode='x unified',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)', font=dict(size=11)),
        height=560, margin=dict(t=120, l=60, r=40, b=60),
    )
    fig.show()

# ── FUNCIÓN HISTOGRAMA ────────────────────────────────────────────────────────
def plot_histograma_nodo(col, cuarto, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    nodo, tec = col
    if col not in df_pivot_nodo_tec2.columns:
        print(f"No existe combinación {nodo} — {tec}")
        return

    temp = df_pivot_nodo_tec2[col].to_frame('DESPACHO').reset_index()
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])

    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]

    if cuarto == 'Día entero':
        subset     = temp
        cuarto_txt = 'Día entero'
    elif cuarto == 'Solo día':
        subset     = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        cuarto_txt = f'Solo día ({CUARTO_AMANECER}–{CUARTO_ATARDECER})'
    elif cuarto == 'Solo noche':
        subset     = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        cuarto_txt = 'Solo noche'
    else:
        subset     = temp[temp['CUARTO'] == cuarto]
        cuarto_txt = f'Cuarto {cuarto}'

    subset_clean     = subset.dropna(subset=['DESPACHO'])
    daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
    idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
    daily_cuarto_max = (
        subset_clean.loc[idx_max.values]
        .set_index('FECHA')['CUARTO']
        .rename('CUARTO_MAX')
    )
    daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
    valores = daily['MAX_MW']

    if valores.empty or valores.max() == 0:
        print(f"Sin generación para {nodo} — {tec} · {cuarto_txt}")
        return

    anio_label = str(anio) if anio != 'Todos' else 'Histórico'
    color      = tec_color.get(tec, '#888780')
    n_dias     = len(valores)
    mediana    = valores.median()

    # ── BINS FIJOS de 50 MW ───────────────────────────────────────────────────
    BIN_WIDTH = 10
    bin_start = np.floor(valores.min() / BIN_WIDTH) * BIN_WIDTH
    bin_end   = np.ceil(valores.max()  / BIN_WIDTH) * BIN_WIDTH
    bin_edges = np.arange(bin_start, bin_end + BIN_WIDTH, BIN_WIDTH)

    counts, _   = np.histogram(valores, bins=bin_edges)
    bin_idx_max = counts.argmax()
    rango_lo    = bin_edges[bin_idx_max]
    rango_hi    = bin_edges[bin_idx_max + 1]
    dias_rango  = counts[bin_idx_max]

    en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()
    if not en_rango.empty:
        en_rango['FRANJA_2H'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
        top_franjas = (
            en_rango['FRANJA_2H']
            .value_counts()
            .head(5)
            .reset_index()
        )
        top_franjas.columns = ['FRANJA', 'DIAS']
        top_txt_grafico = '  |  '.join(
            [f'{r.FRANJA} ({r.DIAS}d)' for _, r in top_franjas.iterrows()]
        )
        top_txt_print = '\n'.join(
            [f'  {r.FRANJA}  →  {r.DIAS:>3} días  ({r.DIAS / dias_rango * 100:.1f}% del rango frecuente)'
             for _, r in top_franjas.iterrows()]
        )
    else:
        top_txt_grafico = 'N/D'
        top_txt_print   = '  N/D'

    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=valores,
        xbins=dict(start=bin_start, end=bin_end, size=BIN_WIDTH),
        marker=dict(color='rgba(120,140,255,0.75)', line=dict(color='#1a1a1a', width=1)),
        hovertemplate='Despacho: %{x} MW<br>Días: %{y}<extra></extra>',
        name='Días',
    ))
    fig.update_layout(
        title=(
            f'<span style="color:{color}">▌</span> <b>{nodo} — {tec}</b> · '
            f'{cuarto_txt} · {anio_label}  ({n_dias} días · máximo diario)'
        ),
    )
    fig.add_vrect(
        x0=rango_lo, x1=rango_hi,
        fillcolor=color, opacity=0.25, line_width=2, line_color=color,
        annotation_text=(
            f'<b>Rango más frecuente</b><br>'
            f'{rango_lo:.0f}–{rango_hi:.0f} MW · {dias_rango} días ({dias_rango/n_dias*100:.1f}%)<br>'
            f'{top_txt_grafico}'
        ),
        annotation_position='top',
        annotation_font=dict(size=11, color=color),
    )
    fig.add_vline(
        x=valores.min(), line_color='#378ADD', line_width=1.5, line_dash='dot',
        annotation_text=f'Mín: {valores.min():.1f} MW',
        annotation_position='bottom left', annotation_font=dict(size=10, color='#7ab8f5'),
    )
    fig.add_vline(
        x=mediana, line_color='orange', line_width=2,
        annotation_text=f'Mediana: {mediana:.1f} MW',
        annotation_position='top right',
        annotation_font=dict(size=10, color='orange'),
    )
    fig.add_vline(
        x=valores.max(), line_color='#e74c3c', line_width=1.5, line_dash='dot',
        annotation_text=f'Máx: {valores.max():.1f} MW',
        annotation_position='bottom right', annotation_font=dict(size=10, color='#e74c3c'),
    )
    fig.update_layout(
        xaxis=dict(
            title='Máximo diario de despacho (MW)',
            tickmode='array', tickvals=bin_edges,
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        ),
        yaxis=dict(title='Días', showgrid=True, gridcolor='rgba(255,255,255,0.06)'),
        bargap=0.05,
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        margin=dict(t=100, l=60, r=40, b=60), height=440,
        showlegend=False,
    )
    fig.show()

    print(f"{'─'*52}")
    print(f"Nodo: {nodo}  |  Tecnología: {tec}  |  {cuarto_txt}  |  {anio_label}")
    print(f"{'─'*52}")
    print(f"Mínimo histórico:    {valores.min():.4f} MW")
    print(f"Mediana:             {mediana:.4f} MW")
    print(f"Máximo histórico:    {valores.max():.4f} MW")
    print(f"Desv. estándar:      {valores.std():.4f} MW")
    print(f"{'─'*52}")
    print(f"Rango más frecuente: {rango_lo:.0f} – {rango_hi:.0f} MW  →  {dias_rango} días ({dias_rango/n_dias*100:.1f}%)")
    print(f"{'─'*52}")
    print(f"Top franjas horarias en el rango más frecuente:")
    print(top_txt_print)

# ── FUNCIÓN WRAPPER ───────────────────────────────────────────────────────────
def plot_nodo_tec_completo(col_label, cuarto, anio, mes):
    idx = cols_labels_nodo.index(col_label)
    col = columnas_nodo[idx]
    plot_curva_nodo(col, anio, mes)
    plot_histograma_nodo(col, cuarto, anio, mes)

# ── WIDGETS ───────────────────────────────────────────────────────────────────
interact(
    plot_nodo_tec_completo,
    col_label = Dropdown(options=cols_labels_nodo, description='Nodo/Tec:', value=cols_labels_nodo[0]),
    cuarto    = Dropdown(options=CUARTOS_DISP,     description='Cuarto:',   value='Solo día'),
    anio      = Dropdown(options=ANIOS_DISP,       description='Año:',      value='Todos'),
    mes       = Dropdown(options=MESES_DISP,       description='Mes:',      value=MESES_DISP[0]),
)

interactive(children=(Dropdown(description='Nodo/Tec:', options=('ANT — Eolica', 'BAY — Hidro', 'BLM — Termica…

<function __main__.plot_nodo_tec_completo(col_label, cuarto, anio, mes)>

In [113]:

def build_tabla_nodo(cuarto, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    anio_int = None if anio == 'Todos' else int(anio)
    rows = []

    for nodo in NODOS_DISP:
        cols_nodo = [c for c in df_pivot_nodo_tec2.columns if c[0] == nodo]

        # ── Capacidad activa para ese año ─────────────────────────────────────
        cap = _cap_activa_nodo2(nodo, anio_int)

        temp = (df_pivot_nodo_tec2[cols_nodo].sum(axis=1)
                .to_frame('DESPACHO').reset_index())
        temp['FECHA'] = pd.to_datetime(temp['FECHA'])
        if anio != 'Todos':
            temp = temp[temp['FECHA'].dt.year == anio_int]
        temp = temp[temp['FECHA'].dt.month == mes]

        if cuarto == 'Día entero':
            subset = temp
        elif cuarto == 'Solo día':
            subset = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        elif cuarto == 'Solo noche':
            subset = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        else:
            subset = temp[temp['CUARTO'] == cuarto]

        subset_clean     = subset.dropna(subset=['DESPACHO'])
        daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
        idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
        daily_cuarto_max = subset_clean.loc[idx_max.values].set_index('FECHA')['CUARTO'].rename('CUARTO_MAX')
        daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
        valores = daily['MAX_MW']
        if valores.empty or valores.max() == 0:
            continue

        tecs_nodo = sorted(set(t for n, t in cols_nodo))

        # ── Bins automáticos ─────────────────────────────────────────────────
        BIN_WIDTH = calcular_bin(valores)
        bin_edges = np.arange(np.floor(valores.min()/BIN_WIDTH)*BIN_WIDTH,
                              np.ceil(valores.max()/BIN_WIDTH)*BIN_WIDTH + BIN_WIDTH, BIN_WIDTH)
        counts, _ = np.histogram(valores, bins=bin_edges)
        idx_bin   = counts.argmax()
        rango_lo  = bin_edges[idx_bin]
        rango_hi  = bin_edges[idx_bin + 1]

        en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()
        if not en_rango.empty:
            en_rango['FRANJA'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
            top         = en_rango['FRANJA'].value_counts().head(3)
            cuarto_pico = '  |  '.join([f'{f}({n}d)' for f, n in top.items()])
        else:
            cuarto_pico = 'N/D'

        mediana   = valores.median()
        max_hist  = valores.max()
        moda_alta = (rango_lo + rango_hi) / 2

        moda_baja_vals = valores[valores < moda_alta]
        if not moda_baja_vals.empty:
            bw2 = calcular_bin(moda_baja_vals)
            be2 = np.arange(np.floor(moda_baja_vals.min()/bw2)*bw2,
                            np.ceil(moda_baja_vals.max()/bw2)*bw2 + bw2, bw2)
            if len(be2) > 1:
                c2, _     = np.histogram(moda_baja_vals, bins=be2)
                i2        = c2.argmax()
                moda_baja = (be2[i2] + be2[i2+1]) / 2
            else:
                moda_baja = moda_baja_vals.min()
        else:
            moda_baja = valores.min()

        n_dias            = len(valores)
        dias_zona_baja    = (valores <= moda_baja).sum()
        dias_zona_central = ((valores > moda_baja) & (valores < moda_alta)).sum()
        dias_zona_alta    = (valores >= moda_alta).sum()

        rows.append({
            'NODO':                nodo,
            'TECNOLOGIAS':         ' · '.join(tecs_nodo),
            'CAP_MW':              round(cap, 1) if cap > 0 else None,
            'UTIL_TIPICA_%':       round(mediana  / cap * 100, 1) if cap > 0 else None,
            'UTIL_MAX_%':          round(max_hist / cap * 100, 1) if cap > 0 else None,
            'BRECHA_MW':           round(cap - moda_alta, 1)      if cap > 0 else None,
            'DIAS_ZONA_BAJA_%':    round(dias_zona_baja    / n_dias * 100, 1),
            'DIAS_ZONA_CENTRAL_%': round(dias_zona_central / n_dias * 100, 1),
            'DIAS_ZONA_ALTA_%':    round(dias_zona_alta    / n_dias * 100, 1),
            'CUARTO_PICO':         cuarto_pico,
            'N_DIAS':              n_dias,
        })

    df_out = pd.DataFrame(rows).set_index('NODO')
    cap_nota = '(prom años activos)' if anio == 'Todos' else '(plantas activas ese año)'
    print(f'CAP_MW {cap_nota}')
    display(df_out)


interact(
    build_tabla_nodo,
    cuarto = Dropdown(options=CUARTOS_DISP, description='Cuarto:', value='Día entero'),
    anio   = Dropdown(options=ANIOS_DISP,        description='Año:',   value='Todos'),
    mes    = Dropdown(options=MESES_DISP,    description='Mes:',   value=MESES_DISP[0])
)

interactive(children=(Dropdown(description='Cuarto:', options=('Día entero', 'Solo día', 'Solo noche', 15, 30,…

<function __main__.build_tabla_nodo(cuarto, anio, mes)>

## 8d — Análisis por Planta


In [114]:

meta = planta_meta  # alias — planta_meta contiene TECNOLOGIA, ZONA y NODO

def _cap_activa_planta(p, anio=None):
    """Capacidad si la planta despachó ese año.
    Si anio=None devuelve promedio entre años donde despachó."""
    if anio is not None:
        mask_anio = fecha_idx_plt.year == anio
        activa    = df_pivot_plantas.loc[mask_anio, p].sum(skipna=True) > 0
        return inst_mw_final.get(p, 0) if activa else 0.0
    else:
        caps = [_cap_activa_planta(p, a) for a in ANIOS_DISP[1:]]
        caps = [c for c in caps if c > 0]
        return float(np.mean(caps)) if caps else 0.0

def build_tabla_planta(planta, cuarto, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    plantas_sel = plantas if planta == 'Todas' else [planta]
    anio_int    = None if anio == 'Todos' else int(anio)
    rows        = []

    for p in plantas_sel:
        # ── Capacidad activa para ese año ─────────────────────────────────────
        cap = _cap_activa_planta(p, anio_int)

        temp = (df_pivot_plantas[[p]].reset_index()
                .rename(columns={p: 'DESPACHO'}))
        temp['FECHA'] = pd.to_datetime(temp['FECHA'])
        if anio != 'Todos':
            temp = temp[temp['FECHA'].dt.year == anio_int]
        temp = temp[temp['FECHA'].dt.month == mes]

        if cuarto == 'Día entero':
            subset = temp
        elif cuarto == 'Solo día':
            subset = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        elif cuarto == 'Solo noche':
            subset = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        else:
            subset = temp[temp['CUARTO'] == cuarto]

        subset_clean     = subset.dropna(subset=['DESPACHO'])
        daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
        idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
        daily_cuarto_max = subset_clean.loc[idx_max.values].set_index('FECHA')['CUARTO'].rename('CUARTO_MAX')
        daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
        valores = daily['MAX_MW']
        if valores.empty or valores.max() == 0:
            continue

        tec = meta.loc[p, 'TECNOLOGIA'] if p in meta.index else 'N/D'

        # ── Bins automáticos ──────────────────────────────────────────────────
        BIN_WIDTH = calcular_bin(valores)
        bin_edges = np.arange(np.floor(valores.min()/BIN_WIDTH)*BIN_WIDTH,
                              np.ceil(valores.max()/BIN_WIDTH)*BIN_WIDTH + BIN_WIDTH, BIN_WIDTH)
        counts, _ = np.histogram(valores, bins=bin_edges)
        idx_bin   = counts.argmax()
        rango_lo  = bin_edges[idx_bin]
        rango_hi  = bin_edges[idx_bin + 1]

        en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()
        if not en_rango.empty:
            en_rango['FRANJA'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
            top         = en_rango['FRANJA'].value_counts().head(3)
            cuarto_pico = '  |  '.join([f'{f}({n}d)' for f, n in top.items()])
        else:
            cuarto_pico = 'N/D'

        mediana   = valores.median()
        max_hist  = valores.max()
        moda_alta = (rango_lo + rango_hi) / 2

        moda_baja_vals = valores[valores < moda_alta]
        if not moda_baja_vals.empty:
            bw2 = calcular_bin(moda_baja_vals)
            be2 = np.arange(np.floor(moda_baja_vals.min()/bw2)*bw2,
                            np.ceil(moda_baja_vals.max()/bw2)*bw2 + bw2, bw2)
            if len(be2) > 1:
                c2, _     = np.histogram(moda_baja_vals, bins=be2)
                i2        = c2.argmax()
                moda_baja = (be2[i2] + be2[i2+1]) / 2
            else:
                moda_baja = moda_baja_vals.min()
        else:
            moda_baja = valores.min()

        n_dias            = len(valores)
        dias_zona_baja    = (valores <= moda_baja).sum()
        dias_zona_central = ((valores > moda_baja) & (valores < moda_alta)).sum()
        dias_zona_alta    = (valores >= moda_alta).sum()

        rows.append({
            'PLANTA':              p,
            'TECNOLOGIA':          tec,
            'CAP_MW':              round(cap, 1) if cap > 0 else None,
            'UTIL_TIPICA_%':       round(mediana  / cap * 100, 1) if cap > 0 else None,
            'UTIL_MAX_%':          round(max_hist / cap * 100, 1) if cap > 0 else None,
            'BRECHA_MW':           round(cap - moda_alta, 1)      if cap > 0 else None,
            'DIAS_ZONA_BAJA_%':    round(dias_zona_baja    / n_dias * 100, 1),
            'DIAS_ZONA_CENTRAL_%': round(dias_zona_central / n_dias * 100, 1),
            'DIAS_ZONA_ALTA_%':    round(dias_zona_alta    / n_dias * 100, 1),
            'CUARTO_PICO':         cuarto_pico,
            'N_DIAS':              n_dias,
        })

    df_out = pd.DataFrame(rows).set_index(['PLANTA', 'TECNOLOGIA'])
    cap_nota = '(prom años activos)' if anio == 'Todos' else '(plantas activas ese año)'
    print(f'CAP_MW {cap_nota}')
    display(df_out)

plantas           = sorted(df_pivot_plantas.columns.tolist())
opciones_planta   = ['Todas'] + plantas

interact(
    build_tabla_planta,
    planta = Dropdown(options=opciones_planta,   description='Planta:', value='Todas'),
    cuarto = Dropdown(options=CUARTOS_DISP, description='Cuarto:', value='Día entero'),
    anio   = Dropdown(options=ANIOS_DISP,        description='Año:',   value='Todos'),
    mes    = Dropdown(options=MESES_DISP,    description='Mes:',   value=MESES_DISP[0]),
)

interactive(children=(Dropdown(description='Planta:', options=('Todas', 'ACP', 'ALGARROBOS', 'ANDREAS POWER', …

<function __main__.build_tabla_planta(planta, cuarto, anio, mes)>

In [115]:

meta_plt = (df[['NOMBRE_PLANTA', 'TECNOLOGIA', 'ZONA']]
            .drop_duplicates('NOMBRE_PLANTA')
            .set_index('NOMBRE_PLANTA'))

_plantas   = sorted(df_pivot_plantas.columns.tolist())
fecha_idx_p  = pd.to_datetime(df_pivot_plantas.index.get_level_values('FECHA'))
cuarto_idx_p = df_pivot_plantas.index.get_level_values('CUARTO')
_anios     = sorted(fecha_idx_p.year.unique())
_dias      = list(range(1, 32))

def plot_rampa_planta(planta, anio, mes, dia, umbral):
    nombre_mes_local = MESES_NOMBRES[mes]

    if planta not in df_pivot_plantas.columns:
        print(f"No existe {planta}"); return

    tec        = meta_plt.loc[planta, 'TECNOLOGIA'] if planta in meta_plt.index else 'N/D'
    zona       = meta_plt.loc[planta, 'ZONA']       if planta in meta_plt.index else 'N/D'
    cap        = inst_mw_final.get(planta, None)
    cap_txt    = f'{cap:.1f} MW' if cap else 'N/D'
    color      = tec_color.get(tec, '#888780')
    fill_color = hex_to_rgba(color, 0.18)

    mask_anio   = (fecha_idx_p.year == anio) & (fecha_idx_p.month == mes)
    _sub63      = df_pivot_plantas.loc[mask_anio]
    _f63        = pd.to_datetime(_sub63.index.get_level_values('FECHA'))
    _fc63       = pd.Series(_f63).diff().fillna(pd.Timedelta(0)) != pd.Timedelta(0)
    _ramp63     = _sub63.diff()
    _ramp63[_fc63.values] = np.nan
    fecha_anio  = fecha_idx_p[mask_anio]
    cuarto_anio = cuarto_idx_p[mask_anio]

    # ── rampas mensuales ──────────────────────────────────────────────────────
    ramp_df = pd.DataFrame({
        'DIA':  fecha_anio.day,
        'RAMP': _ramp63[planta].values,
    }).dropna()

    if ramp_df.empty:
        print(f"Sin datos para {planta} — {anio}"); return

    daily = ramp_df.groupby('DIA')['RAMP'].agg(
        ramp_up_max   = lambda x: x[x > 0].max() if (x > 0).any() else 0,
        ramp_down_max = lambda x: x[x < 0].min() if (x < 0).any() else 0,
        ramp_abs_mean = lambda x: x.abs().mean(),
    ).round(4)

    # ── datos del día seleccionado ────────────────────────────────────────────
    mask_dia  = fecha_anio.day == dia
    cuartos_d = cuarto_anio[mask_dia]

    desp_dia = pd.Series(
        df_pivot_plantas.loc[mask_anio, planta].values[mask_dia],
        index=cuartos_d,
    ).dropna()

    ramp_dia = pd.Series(
        _ramp63[planta].values[mask_dia],
        index=cuartos_d,
    ).dropna()

    cuartos_up   = ramp_dia[ramp_dia >=  umbral].index
    cuartos_down = ramp_dia[ramp_dia <= -umbral].index

    # ── figura 2 filas ────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.52, 0.48],
        vertical_spacing=0.10,
        subplot_titles=(
            f'Comportamiento — Día {dia} / {anio}  |  umbral ±{umbral} MW',
            f'Rampas diarias — {anio}  |  umbral ±{umbral} MW',
        ),
    )

    # ══ FILA 1: comportamiento intradiario ════════════════════════════════════
    if not desp_dia.empty:

        fig.add_trace(go.Scatter(
            x=desp_dia.index, y=desp_dia.round(2).values,
            name='Despacho', mode='lines',
            line=dict(color=color, width=2.2),
            fill='tozeroy', fillcolor=fill_color,
            hovertemplate='Cuarto %{x} | <b>%{y:.1f} MW</b><extra></extra>',
        ), row=1, col=1)

        # línea cap instalada
        if cap:
            fig.add_hline(
                y=cap,
                line=dict(color='rgba(255,255,255,0.30)', width=1, dash='dot'),
                annotation_text=f'Cap. inst.: {cap_txt}',
                annotation_position='top right',
                annotation_font=dict(color='#cccccc', size=9),
                row=1, col=1,
            )

        fig.add_trace(go.Scatter(
            x=ramp_dia.index, y=ramp_dia.round(2).values,
            name='Rampa (MW/15min)', mode='lines',
            line=dict(color='rgba(255,255,255,0.40)', width=1.2, dash='dot'),
            hovertemplate='Rampa: %{y:+.1f} MW<extra></extra>',
        ), row=1, col=1)

        if len(cuartos_up) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_up,
                y=desp_dia.reindex(cuartos_up).values,
                name=f'Ramp-UP ≥{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-up', size=12, color='#2ecc71',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_up).values],
                hovertemplate='Ramp-UP cuarto %{x}<br><b>+%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        if len(cuartos_down) > 0:
            fig.add_trace(go.Scatter(
                x=cuartos_down,
                y=desp_dia.reindex(cuartos_down).values,
                name=f'Ramp-DOWN ≤-{umbral} MW',
                mode='markers',
                marker=dict(
                    symbol='triangle-down', size=12, color='#e74c3c',
                    line=dict(color='#ffffff', width=0.8),
                ),
                text=[f'{v:.1f}' for v in ramp_dia.reindex(cuartos_down).values],
                hovertemplate='Ramp-DOWN cuarto %{x}<br><b>%{text} MW</b><extra></extra>',
            ), row=1, col=1)

        tick_vals_d = list(desp_dia.index)[::8]
        fig.update_xaxes(
            tickmode='array', tickvals=tick_vals_d,
            ticktext=[str(v) for v in tick_vals_d],
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
            row=1, col=1,
        )
    else:
        fig.add_annotation(
            text=f'Sin datos para el día {dia}',
            xref='paper', yref='paper', x=0.5, y=0.75,
            showarrow=False, font=dict(color='#888', size=13),
        )

    # ══ FILA 2: barras mensuales ══════════════════════════════════════════════
    bar_colors_up = [
        'rgba(46,204,113,1.0)' if d == dia else 'rgba(46,204,113,0.55)'
        for d in daily.index
    ]
    bar_border_up = ['#ffffff' if d == dia else 'rgba(0,0,0,0)' for d in daily.index]
    bar_colors_dn = [
        'rgba(231,76,60,1.0)' if d == dia else 'rgba(231,76,60,0.55)'
        for d in daily.index
    ]
    bar_border_dn = ['#ffffff' if d == dia else 'rgba(0,0,0,0)' for d in daily.index]

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_up_max'],
        name='Ramp-up máx',
        marker=dict(color=bar_colors_up, line=dict(color=bar_border_up, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-up máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Bar(
        x=daily.index, y=daily['ramp_down_max'],
        name='Ramp-down máx',
        marker=dict(color=bar_colors_dn, line=dict(color=bar_border_dn, width=1.5)),
        hovertemplate='Día %{x}<br>Ramp-down máx: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=daily.index, y=daily['ramp_abs_mean'],
        name='Media abs', mode='lines+markers',
        line=dict(color=color, width=2, dash='dot'),
        marker=dict(size=5),
        hovertemplate='Día %{x}<br>Media abs: <b>%{y:.2f} MW</b><extra></extra>',
    ), row=2, col=1)

    for y_ref in [umbral, -umbral]:
        fig.add_hline(
            y=y_ref,
            line=dict(color='rgba(255,255,0,0.55)', width=1.5, dash='dash'),
            annotation_text=f'±{umbral} MW' if y_ref > 0 else '',
            annotation_position='top left',
            annotation_font=dict(color='rgba(255,255,0,0.80)', size=9),
            row=2, col=1,
        )

    # ── ejes ──────────────────────────────────────────────────────────────────
    fig.update_yaxes(
        title_text='Despacho / Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=1, col=1,
    )
    fig.update_yaxes(
        title_text='Rampa (MW)',
        showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        zeroline=True, zerolinecolor='rgba(255,255,255,0.20)',
        row=2, col=1,
    )
    fig.update_xaxes(
        title_text='Día', tickmode='linear', tick0=1, dtick=1,
        showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        row=2, col=1,
    )

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{planta}</b>  '
                f'<sup><span style="color:{color}">▌</span> {tec}  |  {zona}  |  Cap: {cap_txt}</sup>  '
                f'<sup>· {anio}  |  Día: <b>{dia}</b>  |  Umbral: ±{umbral} MW</sup>'
            ),
            font=dict(size=15),
        ),
        hovermode='x unified',
        barmode='overlay',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.03,
            xanchor='right', x=1, bgcolor='rgba(0,0,0,0)',
            font=dict(size=11),
        ),
        height=750, margin=dict(t=90, l=75, r=40, b=60),
    )
    fig.show()

    # ── tabla texto ───────────────────────────────────────────────────────────
    print(f"\n── Rampas sobre umbral ±{umbral} MW · {planta} · {anio} ──")

    ramp_col = _ramp63[planta]

    for label, signo in [('🟢 Ramp-UP', 'up'), ('🔴 Ramp-DOWN', 'down')]:
        rf = (ramp_col[ramp_col >=  umbral] if signo == 'up'
              else ramp_col[ramp_col <= -umbral])
        if rf.dropna().empty:
            print(f"\n  {label}: ninguna rampa supera el umbral de {umbral} MW"); continue
        df_m = pd.DataFrame({
            'RAMP':   rf.abs().values,
            'CUARTO': rf.index.get_level_values('CUARTO').tolist(),
            'DIA':    pd.to_datetime(rf.index.get_level_values('FECHA')).day.tolist(),
        }).dropna()
        daily_max   = (df_m.loc[df_m.groupby('DIA')['RAMP'].idxmax()]
                       .set_index('DIA').sort_index())
        dia_critico = daily_max['RAMP'].idxmax()
        n_eventos   = len(df_m)
        print(f"\n  {label}  ({n_eventos} eventos sobre {umbral} MW)")
        print(f"  {'─'*45}\n  {'Día':>5}  {'Máx (MW)':>10}  {'Cuarto':>8}")
        print(f"  {'─'*45}")
        for d, row in daily_max.iterrows():
            pct_str = f'  ({row["RAMP"]/cap*100:.1f}% cap)' if cap else ''
            crit    = '  ◄ CRÍTICO' if d == dia_critico else ''
            print(f"  {d:>5}  {row['RAMP']:>10.2f} MW  @ {int(row['CUARTO']):>6} HHMM{pct_str}{crit}")
        print(f"  {'─'*45}")
        print(f"  Promedio máx diario: {daily_max['RAMP'].mean():.2f} MW")

interact(
    plot_rampa_planta,
    planta = Dropdown(options=_plantas, description='Planta:', value=_plantas[0]),
    anio   = Dropdown(options=_anios,   description='Año:',   value=_anios[0]),
    mes    = Dropdown(options=MESES_DISP, description='Mes:', value=MESES_DISP[0]),
    dia    = Dropdown(options=_dias,    description='Día:',   value=_dias[0]),
    umbral = BoundedIntText(min=1, max=10000, step=1, value=10, description='Umbral MW:'),
)

interactive(children=(Dropdown(description='Planta:', options=('ACP', 'ALGARROBOS', 'ANDREAS POWER', 'ANTON SO…

<function __main__.plot_rampa_planta(planta, anio, mes, dia, umbral)>

In [116]:

# ── DATOS BASE ────────────────────────────────────────────────────────────────
meta = (
    df[['NOMBRE_PLANTA', 'TECNOLOGIA', 'ZONA']]
    .drop_duplicates('NOMBRE_PLANTA')
    .set_index('NOMBRE_PLANTA')
)

plantas = sorted(df_pivot_plantas.columns.tolist())

# ── CACHE CURVA ───────────────────────────────────────────────────────────────
def calcular_stats_planta(planta, anio='Todos', mes=None):
    temp = (
        df_pivot_plantas[[planta]]
        .reset_index()
        .rename(columns={planta: 'DESPACHO'})
    )
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])
    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]
    if temp.empty:
        return None
    return (
        temp.groupby('CUARTO')['DESPACHO']
        .agg(
            Minima    = 'min',
            Moda_baja = moda_baja_fn,
            Mediana   = lambda x: x.quantile(0.50),
            Moda_alta = moda_alta_fn,
            Maxima    = 'max',
        )
        .round(4)
    )

stats_cache_plt = {}
for planta in plantas:
    for anio in ANIOS_DISP:
        try:
            stats_cache_plt[(planta, anio, None)] = calcular_stats_planta(planta, anio)
        except Exception:
            stats_cache_plt[(planta, anio, None)] = None
# ── FUNCIÓN CURVA ─────────────────────────────────────────────────────────────
def plot_curva_planta(planta, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    _ck = (planta, anio, mes)
    if _ck not in stats_cache_plt:
        try:
            stats_cache_plt[_ck] = calcular_stats_planta(planta, anio, mes)
        except Exception:
            stats_cache_plt[_ck] = None

    stats = stats_cache_plt[_ck]
    if stats is None:
        print(f"Sin datos para {planta} — {anio}")
        return

    tec        = meta.loc[planta, 'TECNOLOGIA'] if planta in meta.index else 'Otra'
    zona       = meta.loc[planta, 'ZONA']       if planta in meta.index else ''
    color      = tec_color.get(tec, '#888780')
    r, g, b    = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    cap        = inst_mw_final.get(planta, None)
    cap_txt    = f'{cap:.1f} MW' if cap else 'N/D'
    anio_label = str(anio) if anio != 'Todos' else 'Histórico'

    max_mw       = stats['Maxima'].max()
    moda_alta_mw = stats['Moda_alta'].max()
    med_mw       = stats['Mediana'].max()
    moda_baja_mw = stats['Moda_baja'].max()
    min_mw       = stats['Minima'][stats['Minima'] > 0].min() if (stats['Minima'] > 0).any() else 0
    activos      = (stats['Mediana'] > 0).sum()

    x     = stats.index.tolist()
    x_rev = x[::-1]
    fig   = go.Figure()

    fig.add_trace(go.Scatter(
        x=x + x_rev, y=stats['Maxima'].tolist() + stats['Minima'].tolist()[::-1],
        fill='toself', hoverinfo='skip', showlegend=False,
        fillcolor=f'rgba({r},{g},{b},0.05)', line=dict(color='rgba(0,0,0,0)'),
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Minima'], line=dict(color='rgba(0,0,0,0)'),
        hoverinfo='skip', showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_baja'], line=dict(color='rgba(0,0,0,0)'),
        fill='tonexty', fillcolor=f'rgba({r},{g},{b},0.20)',
        hoverinfo='skip', name='Rango Mín–Moda baja',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_alta'], line=dict(color='rgba(0,0,0,0)'),
        hoverinfo='skip', showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Maxima'], line=dict(color='rgba(0,0,0,0)'),
        fill='tonexty', fillcolor=f'rgba({r},{g},{b},0.20)',
        hoverinfo='skip', name='Rango Moda alta–Máx',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Minima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Mínima  {min_mw:.1f} MW',
        hovertemplate='<b>Cuarto %{x}</b><br>Mínima: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_baja'],
        line=dict(color=f'rgba({r},{g},{b},0.60)', width=1.5, dash='dashdot'),
        name=f'Moda baja  {moda_baja_mw:.1f} MW',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda baja: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Mediana'],
        line=dict(color=color, width=2.5),
        name=f'Mediana  {med_mw:.1f} MW',
        hovertemplate='<b>Cuarto %{x}</b><br>Mediana: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Moda_alta'],
        line=dict(color=f'rgba({r},{g},{b},0.60)', width=1.5, dash='dashdot'),
        name=f'Moda alta  {moda_alta_mw:.1f} MW',
        hovertemplate='<b>Cuarto %{x}</b><br>Moda alta: %{y:.2f} MW<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=x, y=stats['Maxima'],
        line=dict(color=f'rgba({r},{g},{b},0.35)', width=1, dash='dot'),
        name=f'Máxima  {max_mw:.1f} MW',
        hovertemplate='<b>Cuarto %{x}</b><br>Máxima: %{y:.2f} MW<extra></extra>',
    ))
    cap_y = [cap] * len(x) if cap else [None] * len(x)
    fig.add_trace(go.Scatter(
        x=x, y=cap_y,
        line=dict(color='rgba(255,255,255,0.35)', width=1, dash='dot'),
        name=f'Cap. inst.  {cap_txt}',
        hovertemplate='<b>Cuarto %{x}</b><br>Cap. inst.: %{y:.2f} MW<extra></extra>',
    ))

    fig.update_layout(
        title=dict(
            text=(
                f'<b>{planta}</b>  '
                f'<sup><span style="color:{color}">▌</span> {tec}</sup>  '
                f'<sup style="color:gray">{zona}</sup>  '
                f'<sup>· {anio_label}</sup><br>'
                f'<sup>'
                f'Cap: <b>{cap_txt}</b>  |  '
                f'Máx: <b>{max_mw:.2f} MW</b>  |  '
                f'Moda alta: <b>{moda_alta_mw:.2f} MW</b>  |  '
                f'Mediana: <b>{med_mw:.2f} MW</b>  |  '
                f'Moda baja: <b>{moda_baja_mw:.2f} MW</b>  |  '
                f'Mín: <b>{min_mw:.2f} MW</b>  |  '
                f'Cuartos activos: <b>{activos}/96</b>'
                f'</sup>'
            ),
            font=dict(size=14),
        ),
        xaxis=dict(
            title='Cuarto del día', tickmode='array',
            tickvals=x[::8], ticktext=[str(v) for v in x[::8]],
            tickangle=0, showgrid=True, gridcolor='rgba(255,255,255,0.06)', zeroline=False,
        ),
        yaxis=dict(title='Despacho (MW)', showgrid=True,
                   gridcolor='rgba(255,255,255,0.06)', zeroline=False),
        hovermode='x unified',
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)', font=dict(size=12)),
        height=580, margin=dict(t=130, l=60, r=40, b=60),
    )
    fig.show()

# ── FUNCIÓN HISTOGRAMA ────────────────────────────────────────────────────────
def plot_histograma_planta(planta, cuarto, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    temp = (
        df_pivot_plantas[[planta]]
        .reset_index()
        .rename(columns={planta: 'DESPACHO'})
    )
    temp['FECHA'] = pd.to_datetime(temp['FECHA'])

    if anio != 'Todos':
        temp = temp[temp['FECHA'].dt.year == anio]
    if mes is not None:
        temp = temp[temp['FECHA'].dt.month == mes]

    if cuarto == 'Día entero':
        subset     = temp
        cuarto_txt = 'Día entero'
    elif cuarto == 'Solo día':
        subset     = temp[(temp['CUARTO'] >= CUARTO_AMANECER) & (temp['CUARTO'] <= CUARTO_ATARDECER)]
        cuarto_txt = f'Solo día ({CUARTO_AMANECER}–{CUARTO_ATARDECER})'
    elif cuarto == 'Solo noche':
        subset     = temp[(temp['CUARTO'] < CUARTO_AMANECER) | (temp['CUARTO'] > CUARTO_ATARDECER)]
        cuarto_txt = 'Solo noche'
    else:
        subset     = temp[temp['CUARTO'] == cuarto]
        cuarto_txt = f'Cuarto {cuarto}'

    subset_clean     = subset.dropna(subset=['DESPACHO'])
    daily_max        = subset_clean.groupby('FECHA')['DESPACHO'].max().rename('MAX_MW')
    idx_max          = subset_clean.groupby('FECHA')['DESPACHO'].idxmax().dropna().astype(int)
    daily_cuarto_max = (
        subset_clean.loc[idx_max.values]
        .set_index('FECHA')['CUARTO'].rename('CUARTO_MAX')
    )
    daily   = pd.concat([daily_max, daily_cuarto_max], axis=1).dropna()
    valores = daily['MAX_MW']

    if valores.empty or valores.max() == 0:
        print(f"Sin generación para {planta} · {cuarto_txt}")
        return

    tec        = meta.loc[planta, 'TECNOLOGIA'] if planta in meta.index else 'N/D'
    zona       = meta.loc[planta, 'ZONA']       if planta in meta.index else 'N/D'
    cap        = inst_mw_final.get(planta, None)
    cap_txt    = f'{cap:.1f} MW' if cap else 'N/D'
    color      = tec_color.get(tec, '#888780')
    anio_label = str(anio) if anio != 'Todos' else 'Histórico'
    n_dias     = len(valores)
    mediana    = valores.median()

    def pct_cap(v): return f'{v/cap*100:.1f}%' if cap else 'N/D'

    # ── BINS FIJOS de 5 MW ────────────────────────────────────────────────────
    BIN_WIDTH = 5
    bin_start = np.floor(valores.min() / BIN_WIDTH) * BIN_WIDTH
    bin_end   = np.ceil(valores.max()  / BIN_WIDTH) * BIN_WIDTH
    bin_edges = np.arange(bin_start, bin_end + BIN_WIDTH, BIN_WIDTH)

    counts, _   = np.histogram(valores, bins=bin_edges)
    bin_idx_max = counts.argmax()
    rango_lo    = bin_edges[bin_idx_max]
    rango_hi    = bin_edges[bin_idx_max + 1]
    dias_rango  = counts[bin_idx_max]

    # ── Top franjas de 2h dentro del rango más frecuente ─────────────────────
    def cuarto_a_franja_2h(c):
        c = int(c)
        hh      = c // 100
        hh2_ini = (hh // 2) * 2
        hh2_fin = hh2_ini + 2
        return f'{hh2_ini:02d}:00–{hh2_fin:02d}:00'

    en_rango = daily[(daily['MAX_MW'] >= rango_lo) & (daily['MAX_MW'] < rango_hi)].copy()

    if not en_rango.empty:
        en_rango['FRANJA_2H'] = en_rango['CUARTO_MAX'].apply(cuarto_a_franja_2h)
        top_franjas = (
            en_rango['FRANJA_2H']
            .value_counts()
            .head(5)
            .reset_index()
        )
        top_franjas.columns = ['FRANJA', 'DIAS']
        top_txt_grafico = '  |  '.join(
            [f'{r.FRANJA} ({r.DIAS}d)' for _, r in top_franjas.iterrows()]
        )
        top_txt_print = '\n'.join(
            [f'  {r.FRANJA}  →  {r.DIAS:>3} días  ({r.DIAS / dias_rango * 100:.1f}% del rango frecuente)'
             for _, r in top_franjas.iterrows()]
        )
    else:
        top_txt_grafico = 'N/D'
        top_txt_print   = '  N/D'

    # ── Histograma ────────────────────────────────────────────────────────────
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=valores,
        xbins=dict(start=bin_start, end=bin_end, size=BIN_WIDTH),
        marker=dict(color='rgba(120,140,255,0.75)', line=dict(color='#1a1a1a', width=1)),
        hovertemplate='Despacho: %{x} MW<br>Días: %{y}<extra></extra>',
        name='Días',
    ))

    fig.update_layout(
        title=(
            f'<b>{planta}</b>  '
            f'<sup><span style="color:{color}">▌</span> {tec}</sup>  '
            f'<sup style="color:gray">{zona}</sup> · '
            f'{cuarto_txt} · {anio_label}  ({n_dias} días · máximo diario)'
        ),
    )

    fig.add_vrect(
        x0=rango_lo, x1=rango_hi,
        fillcolor=color, opacity=0.25, line_width=2, line_color=color,
        annotation_text=(
            f'<b>Rango más frecuente</b><br>'
            f'{rango_lo:.0f}–{rango_hi:.0f} MW · {dias_rango} días ({dias_rango/n_dias*100:.1f}%)<br>'
            f'{top_txt_grafico}'
        ),
        annotation_position='top',
        annotation_font=dict(size=11, color=color),
    )
    fig.add_vline(
        x=valores.min(), line_color='#378ADD', line_width=1.5, line_dash='dot',
        annotation_text=f'Mín: {valores.min():.1f} MW',
        annotation_position='bottom left', annotation_font=dict(size=10, color='#7ab8f5'),
    )
    fig.add_vline(
        x=mediana, line_color='orange', line_width=2,
        annotation_text=f'Mediana: {mediana:.1f} MW',
        annotation_position='top right',
        annotation_font=dict(size=10, color='orange'),
    )
    fig.add_vline(
        x=valores.max(), line_color='#e74c3c', line_width=1.5, line_dash='dot',
        annotation_text=f'Máx: {valores.max():.1f} MW',
        annotation_position='bottom right', annotation_font=dict(size=10, color='#e74c3c'),
    )
    if cap and cap <= valores.max():
        fig.add_vline(
            x=cap, line_color='#cccccc', line_width=1, line_dash='dot',
            annotation_text=f'Cap. inst.: {cap:.1f} MW',
            annotation_position='top left', annotation_yshift=-80,
            annotation_font=dict(size=10, color='#cccccc'),
        )

    fig.update_layout(
        xaxis=dict(
            title='Máximo diario de despacho (MW)',
            tickmode='array', tickvals=bin_edges,
            showgrid=True, gridcolor='rgba(255,255,255,0.06)',
        ),
        yaxis=dict(title='Días', showgrid=True, gridcolor='rgba(255,255,255,0.06)'),
        bargap=0.05,
        plot_bgcolor='#141414', paper_bgcolor='#1a1a1a',
        font=dict(color='#cccccc', family='Inter, sans-serif'),
        margin=dict(t=100, l=60, r=40, b=60), height=440,
        showlegend=False,
    )
    fig.show()

    print(f"{'─'*56}")
    print(f"Planta: {planta}  |  {tec}  |  {zona}  |  Cap. inst.: {cap_txt}")
    print(f"Período: {cuarto_txt}  |  Año: {anio_label}  |  N días: {n_dias}")
    print(f"{'─'*56}")
    print(f"Mínimo histórico:    {valores.min():.4f} MW  ({pct_cap(valores.min())})")
    print(f"Mediana:             {mediana:.4f} MW  ({pct_cap(mediana)})")
    print(f"Máximo histórico:    {valores.max():.4f} MW  ({pct_cap(valores.max())})")
    print(f"Desv. estándar:      {valores.std():.4f} MW")
    print(f"{'─'*56}")
    print(f"Rango más frecuente: {rango_lo:.0f} – {rango_hi:.0f} MW  →  {dias_rango} días ({dias_rango/n_dias*100:.1f}%)")
    print(f"{'─'*56}")
    print(f"Top franjas horarias en el rango más frecuente:")
    print(top_txt_print)

# ── WRAPPER ───────────────────────────────────────────────────────────────────
def plot_planta_completo(planta, cuarto, anio, mes):
    plot_curva_planta(planta, anio, mes)
    plot_histograma_planta(planta, cuarto, anio, mes)

# ── WIDGETS ───────────────────────────────────────────────────────────────────
cuartos_p         = sorted(df_pivot_plantas.index.get_level_values('CUARTO').unique())
opciones_cuarto_p = ['Día entero', 'Solo día', 'Solo noche'] + cuartos_p

interact(
    plot_planta_completo,
    planta = Dropdown(options=plantas,            description='Planta:', value=plantas[0]),
    cuarto = Dropdown(options=opciones_cuarto_p,  description='Cuarto:', value='Solo día'),
    anio   = Dropdown(options=ANIOS_DISP,         description='Año:',   value='Todos'),
    mes    = Dropdown(options=MESES_DISP,         description='Mes:',   value=MESES_DISP[0]),
)

interactive(children=(Dropdown(description='Planta:', options=('ACP', 'ALGARROBOS', 'ANDREAS POWER', 'ANTON SO…

<function __main__.plot_planta_completo(planta, cuarto, anio, mes)>

In [117]:
# ── Buscar el pico de 15.57 MW en CEDRO SOLAR ─────────────────────────────────
PLANTA_BUSCAR = 'CEDRO SOLAR'

_piv = df_pivot_plantas[PLANTA_BUSCAR].dropna()

# Momento exacto del máximo
idx_max = _piv.idxmax()
print(f"Fecha   : {idx_max[0]}")
print(f"Cuarto  : {idx_max[1]}")
print(f"MW      : {_piv.max():.2f}")
print()

# Todos los momentos que superaron la capacidad instalada
_cap = 10.0
_anomalias = _piv[_piv > _cap].reset_index()
_anomalias.columns = ['FECHA', 'CUARTO', 'MW']
_anomalias['EXCESO_%'] = ((_anomalias['MW'] / _cap) - 1) * 100
_anomalias = _anomalias.sort_values('MW', ascending=False).reset_index(drop=True)

print(f"Cuartos que superaron {_cap} MW: {len(_anomalias)}")
display(_anomalias)

Fecha   : 2023-09-07 00:00:00
Cuarto  : 1315
MW      : 13.60

Cuartos que superaron 10.0 MW: 65


,FECHA,CUARTO,MW,EXCESO_%
0,2023-09-07,1315,13.6000,36.000
1,2023-09-07,1300,13.3600,33.600
2,2023-09-07,1230,12.4400,24.400
3,2023-09-07,1330,11.9100,19.100
4,2023-09-07,1200,11.7000,17.000
...,...,...,...,...
60,2023-07-15,1215,10.0400,0.400
61,2023-06-14,930,10.0400,0.400
62,2023-01-18,1230,10.0300,0.300
63,2023-01-09,1245,10.0244,0.244


In [118]:
# ── Diagnóstico: por qué PLANTAS_MAXIMA muestra ~10 MW si el crudo tiene 15.57 ──
PLANTA_BUSCAR = 'CEDRO SOLAR'
MES_BUSCAR    = 1
CUARTO_BUSCAR = 1530

# 1. Valor real en el crudo
_raw = df_pivot_plantas.loc[
    df_pivot_plantas.index.get_level_values('CUARTO') == CUARTO_BUSCAR,
    PLANTA_BUSCAR
]
_raw_mes = _raw[pd.to_datetime(_raw.index.get_level_values('FECHA')).month == MES_BUSCAR]

print(f"CRUDO — cuarto {CUARTO_BUSCAR}, mes {MES_BUSCAR}:")
print(f"  max()          : {_raw_mes.max():.4f} MW  ← esto debería estar en MAXIMA")
print(f"  quantile(0.95) : {_raw_mes.quantile(0.95):.4f} MW")
print(f"  quantile(0.90) : {_raw_mes.quantile(0.90):.4f} MW")
print(f"  n observaciones: {len(_raw_mes)}")
print()

# 2. Cap instalada
cap = inst_mw_final.get(PLANTA_BUSCAR, None)
print(f"CAPACIDAD INSTALADA: {cap} MW")
print()

# 3. ¿Cuarto 1530 está en _cuartos_wide?
try:
    print(f"_cuartos_wide tiene {len(_cuartos_wide)} cuartos")
    print(f"1530 en _cuartos_wide: {1530 in _cuartos_wide}")
    cercanos = sorted([c for c in _cuartos_wide if abs(c - 1530) <= 30])
    print(f"Cuartos cercanos a 1530: {cercanos}")
except NameError:
    print("_cuartos_wide no está definida en esta sesión (celda de setup no se ejecutó)")

# 4. ¿_stat_fns define MAXIMA como max() o algo distinto?
try:
    print(f"\n_stat_fns: {_stat_fns}")
except NameError:
    print("_stat_fns no está definida en esta sesión")

CRUDO — cuarto 1530, mes 1:
  max()          : 9.5000 MW  ← esto debería estar en MAXIMA
  quantile(0.95) : 9.0200 MW
  quantile(0.90) : 8.6687 MW
  n observaciones: 31

CAPACIDAD INSTALADA: 9.96 MW

_cuartos_wide tiene 96 cuartos
1530 en _cuartos_wide: True
Cuartos cercanos a 1530: [1500, 1515, 1530, 1545]

_stat_fns: {'MINIMA': <function <lambda> at 0x000001F2A62F7600>, 'MODA_BAJA': <function moda_baja_fn at 0x000001F2EE43C220>, 'MEDIANA': <function <lambda> at 0x000001F2A62F77E0>, 'MODA_ALTA': <function moda_alta_fn at 0x000001F2EE43C2C0>, 'MAXIMA': <function <lambda> at 0x000001F2B0941300>}


In [119]:
meta_plt = planta_meta  # alias — planta_meta contiene TECNOLOGIA, ZONA y NODO

# ── índices ───────────────────────────────────────────────────────────────────
fecha_idx_p  = pd.to_datetime(df_ramp_plantas.index.get_level_values('FECHA'))
cuarto_idx_p = df_ramp_plantas.index.get_level_values('CUARTO')
anios_iter   = sorted(fecha_idx_p.year.unique().tolist())
plantas_iter = sorted(df_ramp_plantas.columns.tolist())

# ── Construcción del DataFrame ────────────────────────────────────────────────
registros = []

for planta in plantas_iter:
    cap  = inst_mw_final.get(planta, None)
    tec  = meta_plt.loc[planta, 'TECNOLOGIA'] if planta in meta_plt.index else 'N/D'
    zona = meta_plt.loc[planta, 'ZONA']       if planta in meta_plt.index else 'N/D'

    for anio in ['Todos'] + anios_iter:
      for mes_int in MESES_DISP:
        mask_yr  = np.ones(len(fecha_idx_p), dtype=bool) if anio == 'Todos' \
                   else (fecha_idx_p.year == anio)
        mask     = mask_yr & (fecha_idx_p.month == mes_int)
        anio_lbl = 'Todos' if anio == 'Todos' else str(anio)

        base = pd.DataFrame({
            'FECHA':  fecha_idx_p[mask].values,
            'CUARTO': cuarto_idx_p[mask],
            'RAMP':   df_ramp_plantas.loc[mask, planta].values,
        }).dropna(subset=['RAMP'])

        if base.empty:
            continue

        fila = {
            'Planta': planta, 'Tecnologia': tec,
            'Zona': zona, 'Año': anio_lbl, 'Mes': mes_int,
            'Cap_MW': round(cap, 1) if cap else None,
        }

        for direccion, signo in [('UP', 1), ('DOWN', -1)]:
            full_s = base[base['RAMP'] * signo > 0].copy()
            full_s['RAMP'] = full_s['RAMP'].abs()

            if full_s.empty:
                fila.update({
                    f'Prom_Max_{direccion}_MW'        : None,
                    f'RampMax_{direccion}_MW'         : None,
                    f'RampMax_{direccion}_PctCap'     : None,
                    f'Rango_Frec_{direccion}'         : None,
                    f'Rango_Frec_{direccion}_dias'    : None,
                    f'Rango_Frec_{direccion}_PctDias' : None,
                    f'Ventana_Temporal_{direccion}'   : None,
                })
                continue

            daily_max    = full_s.groupby('FECHA')['RAMP'].max().dropna()
            prom         = daily_max.mean()
            ramp_max     = daily_max.max()
            pct_cap      = (ramp_max / cap * 100) if cap else None
            lo, hi, cnt  = bin_mas_frecuente(daily_max)
            n_dias_total = len(daily_max)
            pct_dias     = (cnt / n_dias_total * 100) if n_dias_total > 0 else None
            rango_str    = f'{lo:.0f}–{hi:.0f} MW' if lo is not None else 'N/D'
            dias_en_rango = (
                daily_max[(daily_max >= lo) & (daily_max < hi)].index
                if lo is not None else pd.Index([])
            )
            fila.update({
                f'Prom_Max_{direccion}_MW'        : round(prom, 2),
                f'RampMax_{direccion}_MW'         : round(ramp_max, 2),
                f'RampMax_{direccion}_PctCap'     : round(pct_cap, 1) if pct_cap is not None else None,
                f'Rango_Frec_{direccion}'         : rango_str,
                f'Rango_Frec_{direccion}_dias'    : cnt,
                f'Rango_Frec_{direccion}_PctDias' : round(pct_dias, 1) if pct_dias is not None else None,
                f'Ventana_Temporal_{direccion}'   : rango_temporal_str(full_s, dias_en_rango),
            })

        registros.append(fila)

df_ramp_resumen_planta = pd.DataFrame(registros)[[
    'Planta', 'Tecnologia', 'Zona', 'Año', 'Mes', 'Cap_MW',
    'Prom_Max_UP_MW',   'RampMax_UP_MW',   'RampMax_UP_PctCap',
    'Rango_Frec_UP',    'Rango_Frec_UP_dias',    'Rango_Frec_UP_PctDias',
    'Ventana_Temporal_UP',
    'Prom_Max_DOWN_MW', 'RampMax_DOWN_MW', 'RampMax_DOWN_PctCap',
    'Rango_Frec_DOWN',  'Rango_Frec_DOWN_dias',  'Rango_Frec_DOWN_PctDias',
    'Ventana_Temporal_DOWN',
]]
print(f"df_ramp_resumen_planta: {df_ramp_resumen_planta.shape[0]} filas × {df_ramp_resumen_planta.shape[1]} columnas")

# ── Widget ────────────────────────────────────────────────────────────────────
_col_nombres = {
    'Cap_MW'                   : 'Cap. Instalada (MW)',
    'Prom_Max_UP_MW'           : 'Prom Rampa Máx UP (MW)',
    'RampMax_UP_MW'            : 'Rampa UP Máx (MW)',
    'RampMax_UP_PctCap'        : 'UP Máx / Cap (%)',
    'Rango_Frec_UP'            : 'Rango UP frecuente',
    'Rango_Frec_UP_dias'       : 'Días rango UP',
    'Rango_Frec_UP_PctDias'    : '% días rango UP',
    'Ventana_Temporal_UP'      : 'Ventana UP (P25–P75)',
    'Prom_Max_DOWN_MW'         : 'Prom Rampa Máx DOWN (MW)',
    'RampMax_DOWN_MW'          : 'Rampa DOWN Máx (MW)',
    'RampMax_DOWN_PctCap'      : 'DOWN Máx / Cap (%)',
    'Rango_Frec_DOWN'          : 'Rango DOWN frecuente',
    'Rango_Frec_DOWN_dias'     : 'Días rango DOWN',
    'Rango_Frec_DOWN_PctDias'  : '% días rango DOWN',
    'Ventana_Temporal_DOWN'    : 'Ventana DOWN (P25–P75)',
}
_fmt = {
    'Cap. Instalada (MW)'      : '{:.0f}',
    'Prom Rampa Máx UP (MW)'   : '{:.2f}', 'Rampa UP Máx (MW)'   : '{:.2f}',
    'UP Máx / Cap (%)'         : '{:.1f}%','Días rango UP'        : '{:.0f}',
    '% días rango UP'          : '{:.1f}%',
    'Prom Rampa Máx DOWN (MW)' : '{:.2f}', 'Rampa DOWN Máx (MW)' : '{:.2f}',
    'DOWN Máx / Cap (%)'       : '{:.1f}%','Días rango DOWN'      : '{:.0f}',
    '% días rango DOWN'        : '{:.1f}%',
}

def mostrar_rampas_planta(planta, anio, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    mask_anio = df_ramp_resumen_planta['Año'] == str(anio)
    mask_mes  = df_ramp_resumen_planta['Mes'] == mes
    mask_plt  = (True if planta == 'Todas'
                 else df_ramp_resumen_planta['Planta'] == planta)
    subset = (
        df_ramp_resumen_planta[mask_anio & mask_mes & mask_plt]
        .set_index(['Planta', 'Tecnologia', 'Zona'])
        .drop(columns=['Año'])
        .rename(columns=_col_nombres)
    )
    display(
        subset.style
        .format(_fmt, na_rep='N/D')
        .set_caption(f'Rampas por Planta — {planta} — {anio}')
        .background_gradient(subset=['UP Máx / Cap (%)', 'DOWN Máx / Cap (%)'], cmap='RdYlGn_r')
    )

_anios_w   = ['Todos'] + sorted([a for a in df_ramp_resumen_planta['Año'].unique() if a != 'Todos'])
_plantas_w = ['Todas'] + plantas_iter

interact(
    mostrar_rampas_planta,
    planta = Dropdown(options=_plantas_w, description='Planta:', value='Todas'),
    anio   = Dropdown(options=_anios_w,   description='Año:',   value='Todos'),
    mes    = Dropdown(options=MESES_DISP, description='Mes:',   value=MESES_DISP[0]),
)

df_ramp_resumen_planta: 3044 filas × 20 columnas


interactive(children=(Dropdown(description='Planta:', options=('Todas', 'ACP', 'ALGARROBOS', 'ANDREAS POWER', …

<function __main__.mostrar_rampas_planta(planta, anio, mes)>

## 8e — Correlaciones y Lags


In [120]:
# ── Helpers de dependencia ────────────────────────────────────────────────────
def _clean_pair(a, b):
    a = pd.Series(a).reset_index(drop=True)
    b = pd.Series(b).reset_index(drop=True)
    m = a.notna() & b.notna()
    return a[m].values, b[m].values

def dep_pearson(a, b):
    a, b = _clean_pair(a, b)
    if len(a) < 30 or np.std(a) == 0 or np.std(b) == 0: return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def dep_spearman(a, b):
    a, b = _clean_pair(a, b)
    if len(a) < 30: return np.nan
    ra, rb = pd.Series(a).rank().values, pd.Series(b).rank().values
    if np.std(ra) == 0 or np.std(rb) == 0: return np.nan
    return float(np.corrcoef(ra, rb)[0, 1])

def dep_mi(a, b, n_neighbors=3, max_n=120_000):
    a, b = _clean_pair(a, b)
    if len(a) < 50: return np.nan
    if len(a) > max_n:
        idx = np.random.default_rng(0).choice(len(a), size=max_n, replace=False)
        a, b = a[idx], b[idx]
    return float(mutual_info_regression(
        a.reshape(-1, 1), b, n_neighbors=n_neighbors, random_state=0)[0])

def matriz_dep(df_in, metrica):
    fn  = {'pearson': dep_pearson, 'spearman': dep_spearman, 'mi': dep_mi}[metrica]
    cols = df_in.columns.tolist()
    M = pd.DataFrame(np.nan, index=cols, columns=cols, dtype=float)
    for i, a in enumerate(cols):
        for j, b in enumerate(cols):
            if j < i:   M.loc[a, b] = M.loc[b, a]
            elif i == j: M.loc[a, b] = 1.0 if metrica != 'mi' else np.nan
            else:        M.loc[a, b] = fn(df_in[a], df_in[b])
    return M

print("Helpers listos: dep_pearson, dep_spearman, dep_mi, matriz_dep")

Helpers listos: dep_pearson, dep_spearman, dep_mi, matriz_dep


In [121]:
# ── Aplanar MultiIndex y excluir columnas vacías ─────────────────────────────
_cols_zt = [
    col for col in df_ramp_zona.columns
    if col[0] not in ('', None) and col[1] not in ('', None)
]
df_ramp_zt       = df_ramp_zona[_cols_zt].copy()
df_ramp_zt.columns = [f"{z} — {t}" for z, t in _cols_zt]

df_ramp_zt_up  = df_ramp_zt.where(df_ramp_zt > 0)
df_ramp_zt_dn  = df_ramp_zt.where(df_ramp_zt < 0)
df_ramp_zt_abs = df_ramp_zt.abs()

print(f"Columnas zona-tec activas: {len(df_ramp_zt.columns)}")
print(sorted(df_ramp_zt.columns))

Columnas zona-tec activas: 11
['Zona Central — Eolica', 'Zona Central — Hidro', 'Zona Central — Solar', 'Zona Central — Termica', 'Zona Occidental FOR — Hidro', 'Zona Occidental — Hidro', 'Zona Occidental — Solar', 'Zona Occidental — Termica', 'Zona Oriental BAY — Hidro', 'Zona Oriental — Solar', 'Zona Oriental — Termica']


In [122]:
# ── 3 vistas × 3 métricas = 9 matrices ───────────────────────────────────────
_vistas_zt = {
    'Ramp-Up':   df_ramp_zt_up,
    'Ramp-Down': df_ramp_zt_dn,
    '|Ramp|':    df_ramp_zt_abs,
}

_matrices_zt = {}
total = len(_vistas_zt) * 3
i = 0
for vista, df_in in _vistas_zt.items():
    for metrica in ('pearson', 'spearman', 'mi'):
        i += 1
        print(f"  [{i}/{total}] {vista} — {metrica} ...", end=' ', flush=True)
        _matrices_zt[(vista, metrica)] = matriz_dep(df_in, metrica)
        print("listo")

print("\nTodas las matrices calculadas.")

  [1/9] Ramp-Up — pearson ... listo
listo9] Ramp-Up — spearman ... 
listo9] Ramp-Up — mi ... 
listo9] Ramp-Down — pearson ... 
listo9] Ramp-Down — spearman ... 
listo9] Ramp-Down — mi ... 
listo9] |Ramp| — pearson ... 
listo9] |Ramp| — spearman ... 
listo9] |Ramp| — mi ... 

Todas las matrices calculadas.


In [123]:
def _label_color(label):
    for t, c in tec_color.items():
        if t in label: return c
    return '#aaaaaa'

_zonas_opts = ['Todas'] + sorted(set(c.split(' — ')[0] for c in df_ramp_zt.columns))
_tecs_opts  = ['Todas'] + sorted(set(c.split(' — ')[1] for c in df_ramp_zt.columns))

w_vista   = Dropdown(options=['Ramp-Up','Ramp-Down','|Ramp|'], value='|Ramp|', description='Vista:')
w_met     = Dropdown(options=['pearson','spearman','mi'],       value='spearman', description='Métrica:')
w_zona    = Dropdown(options=_zonas_opts, value='Todas',        description='Zona:')
w_tec     = Dropdown(options=_tecs_opts,  value='Todas',        description='Tecnología:')
w_mes_69  = Dropdown(options=MESES_DISP, value=MESES_DISP[0], description='Mes:')
w_umbral  = FloatSlider(value=0.25, min=0.0, max=0.99, step=0.01,
                        description='Umbral |r|:', readout_format='.2f',
                        style={'description_width':'80px'},
                        layout=widgets.Layout(width='400px'))
out_corr = Output()

def _plot_corr_zt(vista, metrica, zona, tec, umbral, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    _fidx_69 = pd.to_datetime(df_ramp_zt.index.get_level_values('FECHA'))
    _mask_mes_69 = _fidx_69.month == mes
    _src_zt = {'Ramp-Up': df_ramp_zt_up, 'Ramp-Down': df_ramp_zt_dn, '|Ramp|': df_ramp_zt_abs}
    M = matriz_dep(_src_zt[vista].loc[_mask_mes_69], metrica).copy()
    if zona != 'Todas': M = M.loc[[zona in c for c in M.columns], [zona in c for c in M.columns]]
    if tec  != 'Todas': M = M.loc[[tec  in c for c in M.columns], [tec  in c for c in M.columns]]
    if M.empty: print("Sin datos."); return

    mu = M.abs() < umbral
    np.fill_diagonal(mu.values, False)
    M[mu] = np.nan
    activas = M.notna().any(axis=1)
    M = M.loc[activas, activas]
    if M.shape[0] < 2: print(f"Ningún par supera umbral {umbral:.2f}."); return

    n  = len(M)
    sz = max(480, n * 58)
    cs = 'RdBu_r' if metrica != 'mi' else 'Viridis'
    text = np.where(np.isnan(M.values), '', np.round(M.values, 3).astype(str))
    ty = [f'<span style="color:{_label_color(c)}">{c}</span>' for c in M.index]
    tx = [f'<span style="color:{_label_color(c)}">{c}</span>' for c in M.columns]

    fig = go.Figure(go.Heatmap(
        z=M.values, x=M.columns, y=M.index,
        zmid=0 if metrica != 'mi' else None,
        colorscale=cs, text=text, texttemplate='%{text}',
        textfont=dict(size=9),
        hovertemplate='%{y} ↔ %{x}<br>%{z:.4f}<extra></extra>'
    ))
    fig.update_layout(
        title=f'',
        template='plotly_white', width=sz, height=sz,
        xaxis=dict(side='top', tickangle=45, tickmode='array',
                   tickvals=list(range(n)), ticktext=tx, tickfont=dict(size=9)),
        yaxis=dict(autorange='reversed', tickmode='array',
                   tickvals=list(range(n)), ticktext=ty, tickfont=dict(size=9)),
    )
    fig.show()

def _oc(change):
    out_corr.clear_output(wait=True)
    with out_corr: _plot_corr_zt(w_vista.value, w_met.value, w_zona.value, w_tec.value, w_umbral.value, w_mes_69.value)

for w in [w_vista, w_met, w_zona, w_tec, w_umbral, w_mes_69]: w.observe(_oc, names='value')
display(VBox([HBox([w_vista, w_met, w_mes_69]), HBox([w_zona, w_tec]), w_umbral, out_corr]))
with out_corr: _plot_corr_zt(w_vista.value, w_met.value, w_zona.value, w_tec.value, w_umbral.value, w_mes_69.value)

In [124]:
# ── Señal causa: suma de todas las plantas solares ────────────────────────────
_solar_plantas = [
    p for p in df_ramp_plantas.columns
    if p in planta_meta.index and planta_meta.loc[p, 'TECNOLOGIA'] == 'Solar'
]
_solar_causa = df_ramp_plantas[_solar_plantas].sum(axis=1)

# ── Plantas de respuesta: no-Solar, con ZONA y TECNOLOGIA definidas ───────────
_plantas_resp = [
    p for p in df_ramp_plantas.columns
    if p in planta_meta.index
    and planta_meta.loc[p, 'TECNOLOGIA'] not in ('Solar', '', None)
    and pd.notna(planta_meta.loc[p, 'TECNOLOGIA'])
    and planta_meta.loc[p, 'ZONA'] not in ('', None)
    and pd.notna(planta_meta.loc[p, 'ZONA'])
]

_lags_resp = {0:'0 min', 1:'+15 min', 2:'+30 min', 4:'+60 min'}

rows_resp = []
for p in _plantas_resp:
    meta_p = planta_meta.loc[p]
    cap    = inst_mw_final.get(p, np.nan)
    for lag, lbl in _lags_resp.items():
        x, y = _clean_pair(_solar_causa.values, df_ramp_plantas[p].shift(-lag).values)
        if len(x) < 50: continue
        r_p  = dep_pearson(x, y)
        X_   = np.column_stack([x, np.ones(len(x))])
        beta = np.linalg.lstsq(X_, y, rcond=None)[0][0]
        r2   = np.corrcoef(x, y)[0, 1] ** 2
        rows_resp.append({
            'Planta':  p,
            'Zona':    meta_p['ZONA'],
            'Tec':     meta_p['TECNOLOGIA'],
            'Cap_MW':  cap,
            'Lag':     lbl,
            'Pearson': round(r_p,   4),
            'Beta':    round(beta,  4),
            'R2':      round(r2,    4),
        })

df_resp_plantas = pd.DataFrame(rows_resp)
print(f"Tabla lista: {df_resp_plantas['Planta'].nunique()} plantas × {len(_lags_resp)} lags")

D:\Anaconda\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning:

invalid value encountered in divide

D:\Anaconda\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning:

invalid value encountered in divide



Tabla lista: 75 plantas × 4 lags


In [125]:
_lag_q_map_b3 = {'0 min': 0, '+15 min': 1, '+30 min': 2, '+60 min': 4}

w_lag_b3  = Dropdown(options=['0 min','+15 min','+30 min','+60 min'], value='+15 min', description='Lag:')
w_anio_b3 = Dropdown(options=ANIOS_DISP[1:], value=ANIOS_DISP[-1], description='Año:')
w_mes_b3  = Dropdown(options=MESES_DISP, value=MESES_DISP[0], description='Mes:')
w_dia_b3  = Dropdown(options=DIAS_DISP, value=DIAS_DISP[0], description='Día:')
w_zona_b3 = Dropdown(options=['Todas']+sorted(planta_meta['ZONA'].dropna().unique()),
                     value='Todas', description='Zona:')
w_tec_b3  = Dropdown(options=['Todas','Hidro','Termica','Eolica'],
                     value='Hidro', description='Tecnología:')
out_b3 = Output()

def _plot_peor_evento(lag_lbl, anio, mes, dia, zona, tec):
    nombre_mes_local = MESES_NOMBRES[mes]
    lag_q = _lag_q_map_b3[lag_lbl]

    # ── Señal solar nacional ──────────────────────────────────────────────────
    _solar_plt_b3 = [
        p for p in df_ramp_plantas.columns
        if p in planta_meta.index and planta_meta.loc[p, 'TECNOLOGIA'] == 'Solar'
    ]
    solar_causa_b3 = df_ramp_plantas[_solar_plt_b3].sum(axis=1)

    # ── Cuarto de peor caída solar en ese día ─────────────────────────────────
    mask_dia  = (fecha_idx_plt.year == anio) & (fecha_idx_plt.month == mes) & (fecha_idx_plt.day == dia)
    solar_dia = solar_causa_b3[mask_dia]
    if solar_dia.empty:
        print(f"Sin datos para {dia}/{mes:02d}/{anio}."); return

    idx_peor   = solar_dia.idxmin()
    caida_mw   = float(solar_dia[idx_peor])
    cuarto_str = str(cuarto_idx_plt[df_ramp_plantas.index.get_loc(idx_peor)]).zfill(4)
    cuarto_fmt = f"{cuarto_str[:2]}:{cuarto_str[2:]}"

    if caida_mw >= 0:
        print(f"No hubo caída solar el {dia}/{mes:02d}/{anio}."); return

    # ── Plantas a analizar ────────────────────────────────────────────────────
    plantas_sel = [
        p for p in df_ramp_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'TECNOLOGIA'] not in ('Solar', '', None)
        and pd.notna(planta_meta.loc[p, 'TECNOLOGIA'])
        and planta_meta.loc[p, 'ZONA'] not in ('', None)
        and pd.notna(planta_meta.loc[p, 'ZONA'])
        and (zona == 'Todas' or planta_meta.loc[p, 'ZONA']       == zona)
        and (tec  == 'Todas' or planta_meta.loc[p, 'TECNOLOGIA'] == tec)
    ]
    if not plantas_sel:
        print("Sin plantas para esta combinación."); return

    # ── Respuesta de cada planta en ese cuarto exacto (con lag) ──────────────
    rows = []
    for p in plantas_sel:
        resp_val = df_ramp_plantas[p].shift(-lag_q).loc[idx_peor]
        if pd.isna(resp_val): continue
        rows.append({
            'Planta':         p,
            'Tec':            planta_meta.loc[p, 'TECNOLOGIA'],
            'Zona':           planta_meta.loc[p, 'ZONA'],
            'Resp_MW':        round(float(resp_val), 2),
            'Es_Intercambio': False,
        })

    # ── Intercambio ───────────────────────────────────────────────────────────
    if 'INTERCAMBIO' in df_ramp_plantas.columns:
        resp_int = df_ramp_plantas['INTERCAMBIO'].shift(-lag_q).loc[idx_peor]
        if pd.notna(resp_int) and abs(float(resp_int)) >= 0.1:
            rows.append({
                'Planta':         'INTERCAMBIO',
                'Tec':            'Intercambio',
                'Zona':           'N/A',
                'Resp_MW':        round(float(resp_int), 2),
                'Es_Intercambio': True,
            })

    # ── Recuperación solar en t+lag ───────────────────────────────────────────
    idx_loc_peor = df_ramp_plantas.index.get_loc(idx_peor)
    idx_lag      = min(idx_loc_peor + lag_q, len(solar_causa_b3) - 1)
    rec_solar_mw = max(0.0, round(float(solar_causa_b3.iloc[idx_lag]), 2))

    if rec_solar_mw >= 0.1:
        rows.append({
            'Planta':         'Recuperación Solar',
            'Tec':            'Solar',
            'Zona':           'N/A',
            'Resp_MW':        rec_solar_mw,
            'Es_Intercambio': False,
        })

    if not rows:
        print("Sin datos suficientes."); return

    df_b3 = pd.DataFrame(rows).sort_values('Resp_MW', ascending=False)
    df_b3 = df_b3[df_b3['Resp_MW'].abs() >= 0.1]
    if df_b3.empty:
        print("Todas las plantas tuvieron respuesta ≈ 0 en ese cuarto."); return

    comp_total_mw = df_b3[df_b3['Resp_MW'] > 0]['Resp_MW'].sum()
    gap_mw        = abs(caida_mw) - comp_total_mw

    _tec_color_ext = {**tec_color, 'Intercambio': '#555555', 'Solar': '#E8A838'}
    colores = [_tec_color_ext.get(t, '#aaaaaa') for t in df_b3['Tec']]
    bordes  = ['#FF6600' if es else 'white' for es in df_b3['Es_Intercambio']]
    hover   = [
        f"<b>{r.Planta}</b><br>Zona: {r.Zona}<br>"
        f"Tec: {r.Tec}<br>Respuesta: {r.Resp_MW:.2f} MW"
        for r in df_b3.itertuples()
    ]

    # ── Gráfico 1: bar chart de respuesta por planta ──────────────────────────
    fig = go.Figure(go.Bar(
        x=df_b3['Resp_MW'], y=df_b3['Planta'],
        orientation='h',
        marker=dict(color=colores, line=dict(color=bordes, width=1.5)),
        text=[f'{v:.1f} MW' for v in df_b3['Resp_MW']],
        textposition='outside', textfont=dict(size=9),
        hovertext=hover, hoverinfo='text'
    ))

    fig.add_vline(x=0, line_dash='dot', line_color='gray', line_width=1)
    fig.add_vline(x=abs(caida_mw), line_dash='dash', line_color='red', line_width=1,
                  annotation_text='Magnitud caída solar',
                  annotation_font_size=9, annotation_position='top left')

    for tec_l, color_l in _tec_color_ext.items():
        if tec_l in df_b3['Tec'].values:
            fig.add_trace(go.Bar(x=[None], y=[None], name=tec_l,
                                 marker_color=color_l, showlegend=True))

    fig.update_layout(
        title=(
            f'Respuesta en peor cuarto solar — {dia}/{mes:02d}/{anio} '
            f'a las {cuarto_fmt} | lag {lag_lbl}<br>'
            f'<sup>'
            f'Caída solar: <b>{caida_mw:.1f} MW</b>  |  '
            f'Cobertura total: <b>+{comp_total_mw:.1f} MW</b>  |  '
            f'{"Déficit" if gap_mw > 0 else "Sobre-comp."}: '
            f'<b>{abs(gap_mw):.1f} MW</b>  |  '
            f'Zona: {zona} | Tec: {tec}'
            f'</sup>'
        ),
        template='plotly_white', width=900,
        height=max(420, len(df_b3)*30),
        xaxis_title='Respuesta real en ese cuarto (MW/15min) — positivo = compensa',
        yaxis=dict(tickfont=dict(size=10)),
        bargap=0.3, margin=dict(l=190, t=110),
        legend=dict(x=1.01, y=1, xanchor='left', font=dict(size=10)),
        barmode='overlay'
    )
    fig.show()

    # ── Gráfico 2: autocompensación solar post-evento ─────────────────────────
    N_CUARTOS  = 8
    idx_loc    = df_ramp_plantas.index.get_loc(idx_peor)
    idx_inicio = max(0, idx_loc - 2)
    idx_fin    = min(len(solar_causa_b3) - 1, idx_loc + N_CUARTOS)

    _solar_nivel  = df_pivot_plantas[_solar_plt_b3].sum(axis=1)
    solar_ventana = _solar_nivel.iloc[idx_inicio:idx_fin + 1].values
    ramp_ventana  = solar_causa_b3.iloc[idx_inicio:idx_fin + 1].values
    cuartos_v     = cuarto_idx_plt[idx_inicio:idx_fin + 1]
    idx_rel       = idx_loc - idx_inicio
    x_nums        = list(range(len(cuartos_v)))
    labels        = [f"{str(c).zfill(4)[:2]}:{str(c).zfill(4)[2:]}" for c in cuartos_v]

    fig2 = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.5, 0.5],
        vertical_spacing=0.08,
        subplot_titles=[
            'Nivel de despacho solar (MW) ',
            'Rampa solar (MW/15min) '
        ]
    )

    fig2.add_trace(go.Scatter(
        x=x_nums, y=solar_ventana,
        mode='lines+markers', name='Despacho solar',
        line=dict(color='#E8A838', width=2),
        marker=dict(size=6),
        text=labels,
        hovertemplate='%{text}  Solar: %{y:.1f} MW<extra></extra>'
    ), row=1, col=1)

    colores_ramp = ['#e74c3c' if v < 0 else '#2ecc71' for v in ramp_ventana]
    fig2.add_trace(go.Bar(
        x=x_nums, y=ramp_ventana,
        name='Rampa solar',
        marker_color=colores_ramp,
        text=labels,
        hovertemplate='%{text}  Rampa: %{y:.1f} MW<extra></extra>'
    ), row=2, col=1)

    fig2.add_shape(
        type='line',
        x0=idx_rel, x1=idx_rel,
        y0=0, y1=1, yref='paper',
        xref='x',
        line=dict(color='red', width=1.5, dash='dash'),
    )
    fig2.add_annotation(
        x=idx_rel, y=1.05, yref='paper', xref='x',
        text='Peor cuarto', showarrow=False,
        font=dict(size=9, color='red'), xanchor='center'
    )
    fig2.add_shape(
        type='line',
        x0=x_nums[0], x1=x_nums[-1],
        y0=0, y1=0,
        xref='x2', yref='y2',
        line=dict(color='gray', width=1, dash='dot'),
    )

    ramp_post   = ramp_ventana[idx_rel + 1:]
    cuartos_rec = np.where(ramp_post >= 0)[0]
    if len(cuartos_rec) > 0:
        n_rec     = cuartos_rec[0] + 1
        rec_txt   = (f'Solar recuperó en {n_rec} cuarto(s) '
                     f'(+{n_rec*15} min) — autocompensación probable')
        rec_color = '#27ae60'
    else:
        rec_txt   = 'Solar no recuperó en ventana de 2h — caída sostenida'
        rec_color = '#e74c3c'

    fig2.update_layout(
        title=(
            f'Evolución solar post-evento — {dia}/{mes:02d}/{anio} desde {cuarto_fmt}<br>'
            f'<sup style="color:{rec_color}"><b>{rec_txt}</b></sup>'
        ),
        template='plotly_white', width=900, height=480,
        yaxis_title='MW', yaxis2_title='MW/15min',
        showlegend=False, hovermode='x unified'
    )
    fig2.update_xaxes(
        tickmode='array',
        tickvals=x_nums,
        ticktext=labels,
        tickangle=45,
        tickfont=dict(size=9)
    )
    fig2.show()

def _ob3(change):
    out_b3.clear_output(wait=True)
    with out_b3:
        _plot_peor_evento(w_lag_b3.value, w_anio_b3.value, w_mes_b3.value,
                          w_dia_b3.value, w_zona_b3.value, w_tec_b3.value)

for w in [w_lag_b3, w_anio_b3, w_mes_b3, w_dia_b3, w_zona_b3, w_tec_b3]:
    w.observe(_ob3, names='value')

display(VBox([HBox([w_anio_b3, w_mes_b3, w_dia_b3, w_lag_b3]),
              HBox([w_zona_b3, w_tec_b3]),
              out_b3]))
with out_b3:
    _plot_peor_evento(w_lag_b3.value, w_anio_b3.value, w_mes_b3.value,
                      w_dia_b3.value, w_zona_b3.value, w_tec_b3.value)

In [126]:
# ── Análisis de cobertura solar con filtro de franja horaria ──────────────────
# Requiere: df_ramp_plantas, planta_meta, fecha_idx_plt, ANIOS_DISP,
#           tec_color, go, VBox, HBox, Output, FloatSlider, Dropdown
#           widgets, pd, np

_lag_q_map_cob = {'0 min': 0, '+15 min': 1, '+30 min': 2, '+60 min': 4}

w_mes_cob    = Dropdown(options=MESES_DISP, value=MESES_DISP[0], description='Mes:')
w_lag_cob    = Dropdown(options=['0 min','+15 min','+30 min','+60 min'],
                        value='+15 min', description='Lag:')
w_zona_cob   = Dropdown(options=['Todas']+sorted(planta_meta['ZONA'].dropna().unique()),
                        value='Todas', description='Zona:')
w_tec_cob    = Dropdown(options=['Todas','Hidro','Termica','Eolica'],
                        value='Hidro', description='Tecnología:')
w_franja_cob = Dropdown(options=['Todo el día', '10:00–14:00', '14:00–17:00'],
                        value='Todo el día', description='Franja:')
w_umb_cob    = FloatSlider(value=30.0, min=5.0, max=300.0, step=5.0,
                           description='Umbral MW:', readout_format='.0f',
                           style={'description_width': '90px'},
                           layout=widgets.Layout(width='380px'))
out_cob = Output()

# ── Mapeo franja → cuartos ────────────────────────────────────────────────────
_franja_cuartos = {
    'Todo el día':  (700,    1700),
    '10:00–14:00': (1000, 1400),
    '14:00–17:00': (1400, 1700),
}

def _plot_cobertura(lag_lbl, zona, tec, franja, umbral_mw, mes):
    nombre_mes_local = MESES_NOMBRES[mes]
    lag_q         = _lag_q_map_cob[lag_lbl]
    cuarto_min, cuarto_max = _franja_cuartos[franja]

    # ── Señal solar nacional ──────────────────────────────────────────────────
    _solar_plt_cob  = [
        p for p in df_ramp_plantas.columns
        if p in planta_meta.index and planta_meta.loc[p, 'TECNOLOGIA'] == 'Solar'
    ]
    solar_causa_cob = df_ramp_plantas[_solar_plt_cob].sum(axis=1)

    # ── Plantas de respuesta ──────────────────────────────────────────────────
    plantas_sel = [
        p for p in df_ramp_plantas.columns
        if p in planta_meta.index
        and planta_meta.loc[p, 'TECNOLOGIA'] not in ('Solar', '', None)
        and pd.notna(planta_meta.loc[p, 'TECNOLOGIA'])
        and planta_meta.loc[p, 'ZONA'] not in ('', None)
        and pd.notna(planta_meta.loc[p, 'ZONA'])
        and (zona == 'Todas' or planta_meta.loc[p, 'ZONA']       == zona)
        and (tec  == 'Todas' or planta_meta.loc[p, 'TECNOLOGIA'] == tec)
    ]
    plantas_sel_comp = plantas_sel + (['INTERCAMBIO']
                                      if 'INTERCAMBIO' in df_ramp_plantas.columns else [])

    # ── Índice de cuartos para filtrar franja ─────────────────────────────────
    cuarto_idx_cob = df_ramp_plantas.index.get_level_values('CUARTO')

    # ── Construir tabla de eventos ────────────────────────────────────────────
    rows_eventos = []
    for anio in ANIOS_DISP[1:]:
        mask_anio     = (fecha_idx_plt.year == anio) & (fecha_idx_plt.month == mes)
        fechas_unicas = np.unique(fecha_idx_plt[mask_anio].date)

        for fecha in fechas_unicas:
            mask_dia = (fecha_idx_plt.date == fecha) & mask_anio

            # filtrar por franja horaria
            if franja != 'Todo el día':
                mask_cuarto = (cuarto_idx_cob >= cuarto_min) & (cuarto_idx_cob < cuarto_max)
                mask_dia    = mask_dia & mask_cuarto

            solar_dia = solar_causa_cob[mask_dia]
            if solar_dia.empty: continue
            idx_peor = solar_dia.idxmin()
            caida    = float(solar_dia[idx_peor])
            if caida >= -umbral_mw: continue

            caida_abs  = abs(caida)
            comp_total = 0.0
            contrib    = {}

            for p in plantas_sel_comp:
                try:
                    resp = float(df_ramp_plantas[p].shift(-lag_q).loc[idx_peor])
                except Exception:
                    continue
                if pd.isna(resp) or resp <= 0: continue
                comp_total += resp
                contrib[p]  = resp

            pct_cob = comp_total / caida_abs if caida_abs > 0 else 0.0

            rows_eventos.append({
                'Año':      str(anio),
                'caida_mw': caida_abs,
                'comp_mw':  comp_total,
                'pct_cob':  pct_cob,
                'contrib':  contrib,
            })

    if not rows_eventos:
        print(f"Sin eventos con caída ≥ {umbral_mw:.0f} MW para esta selección.")
        return

    df_ev = pd.DataFrame(rows_eventos)
    print(f"Eventos encontrados: {len(df_ev)} en años {sorted(df_ev['Año'].unique())}")

    # ── Panel 1: caída vs compensación + % cobertura ──────────────────────────
    resumen_anio = (df_ev.groupby('Año')
                    .agg(caida_media=('caida_mw', 'mean'),
                         comp_media =('comp_mw',  'mean'),
                         pct_media  =('pct_cob',  'mean'))
                    .reset_index())

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        x=resumen_anio['Año'], y=resumen_anio['caida_media'],
        name='Caída solar media (MW)', marker_color='#E8A838',
        hovertemplate='Caída media: %{y:.1f} MW<extra></extra>'
    ))
    fig1.add_trace(go.Bar(
        x=resumen_anio['Año'], y=resumen_anio['comp_media'],
        name='Compensación media (MW)', marker_color='#378ADD',
        hovertemplate='Compensación media: %{y:.1f} MW<extra></extra>'
    ))
    fig1.add_trace(go.Scatter(
        x=resumen_anio['Año'],
        y=resumen_anio['pct_media'] * 100,
        mode='lines+markers+text',
        name='% cobertura media',
        yaxis='y2',
        line=dict(color='#2ecc71', width=2),
        marker=dict(size=8),
        text=[f'{v*100:.0f}%' for v in resumen_anio['pct_media']],
        textposition='top center', textfont=dict(size=10),
        hovertemplate='Cobertura media: %{y:.1f}%<extra></extra>'
    ))
    fig1.update_layout(
        barmode='group',
        title=(f'Caída solar media vs Compensación media por año | '
               f'lag {lag_lbl} | Zona: {zona} | Tec: {tec} | Franja: {franja}<br>'
               f'<sup>Umbral caída: ≥ {umbral_mw:.0f} MW | '
               f'Total eventos: {len(df_ev)}</sup>'),
        template='plotly_white', width=750, height=420,
        xaxis=dict(title='Año', type='category'),
        yaxis=dict(title='MW'),
        yaxis2=dict(title='% cobertura', overlaying='y', side='right',
                    range=[0, 150], showgrid=False),
        legend=dict(orientation='h', y=-0.22),
    )
    fig1.show()

    # ── Panel 2: contribución media por tecnología ────────────────────────────
    rows_contrib = []
    for _, ev in df_ev.iterrows():
        for p, mw in ev['contrib'].items():
            tec_p = ('Intercambio' if p == 'INTERCAMBIO'
                     else planta_meta.loc[p, 'TECNOLOGIA'] if p in planta_meta.index
                     else 'N/D')
            rows_contrib.append({'Año': ev['Año'], 'Planta': p,
                                  'Tec': tec_p, 'MW': mw})

    if not rows_contrib:
        return

    df_contrib  = pd.DataFrame(rows_contrib)
    n_ev_anio   = df_ev.groupby('Año').size().to_dict()
    df_tec_anio = df_contrib.groupby(['Año', 'Tec'])['MW'].sum().reset_index()
    df_tec_anio['MW_norm'] = df_tec_anio.apply(
        lambda r: r['MW'] / n_ev_anio.get(r['Año'], 1), axis=1)

    _tec_color_ext = {**tec_color, 'Intercambio': '#555555'}

    fig2 = go.Figure()
    for tec_v in df_tec_anio['Tec'].unique():
        sub = df_tec_anio[df_tec_anio['Tec'] == tec_v]
        fig2.add_trace(go.Bar(
            x=sub['Año'], y=sub['MW_norm'],
            name=tec_v,
            marker_color=_tec_color_ext.get(tec_v, '#aaaaaa'),
            text=[f'{v:.1f} MW' if v > 0.5 else '' for v in sub['MW_norm']],
            textposition='inside', textfont=dict(size=10, color='white'),
            hovertemplate=f'{tec_v}<br>%{{y:.2f}} MW/evento<extra></extra>'
        ))
    fig2.update_layout(
        barmode='stack',
        title=f'MW compensado por evento — desglose por tecnología | Franja: {franja}',
        template='plotly_white', width=750, height=380,
        xaxis=dict(title='Año', type='category'),
        yaxis_title='MW promedio por evento',
        legend=dict(orientation='h', y=-0.22),
    )
    fig2.show()

    # ── Panel 3: top 15 plantas ───────────────────────────────────────────────
    df_top = (df_contrib.groupby(['Planta', 'Tec'])['MW']
              .sum().reset_index()
              .sort_values('MW', ascending=False).head(15))
    df_top['MW_norm'] = df_top['MW'] / len(df_ev)

    colores_top = [_tec_color_ext.get(t, '#aaaaaa') for t in df_top['Tec']]
    fig3 = go.Figure(go.Bar(
        x=df_top['MW_norm'], y=df_top['Planta'],
        orientation='h', marker_color=colores_top,
        text=[f'{v:.1f} MW' for v in df_top['MW_norm']],
        textposition='outside', textfont=dict(size=9),
        hovertemplate='%{y}<br>%{x:.2f} MW/evento<extra></extra>'
    ))
    fig3.update_layout(
        title=f'Top 15 plantas — MW promedio compensado por evento | Franja: {franja}',
        template='plotly_white', width=750,
        height=max(350, len(df_top) * 30),
        xaxis_title='MW promedio por evento',
        yaxis=dict(tickfont=dict(size=10)),
        bargap=0.3, margin=dict(l=180)
    )
    fig3.show()

def _ocob(change):
    out_cob.clear_output(wait=True)
    with out_cob:
        _plot_cobertura(w_lag_cob.value, w_zona_cob.value,
                        w_tec_cob.value, w_franja_cob.value, w_umb_cob.value,
                        w_mes_cob.value)

for w in [w_lag_cob, w_zona_cob, w_tec_cob, w_franja_cob, w_umb_cob, w_mes_cob]:
    w.observe(_ocob, names='value')

display(VBox([HBox([w_lag_cob,    w_zona_cob]),
              HBox([w_tec_cob,    w_franja_cob]),
              HBox([w_umb_cob,    w_mes_cob]),
              out_cob]))
with out_cob:
    _plot_cobertura(w_lag_cob.value, w_zona_cob.value,
                    w_tec_cob.value, w_franja_cob.value, w_umb_cob.value,
                    w_mes_cob.value)

# Sección 9 — Exportación STATS_WIDE

Calcula estadísticas pre-agregadas por CUARTO (`MINIMA`, `MODA_BAJA`, `MEDIANA`, `MODA_ALTA`, `MAXIMA`, `N_DIAS`, `PCT_ACTIVO`) y exporta a `STATS_WIDE_<año>_<mes>.xlsx` para consumir en `ETESA_STATS_VIEWER.ipynb`.


In [127]:
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill

# ── Variables de configuración ────────────────────────────────────────────────
_tecs_sistema   = sorted(df_pivot_tec.columns.tolist())
_plantas_wide   = sorted(df_pivot_plantas.columns.tolist())
_meses_wide     = sorted(pd.to_datetime(df_pivot_tec.index.get_level_values('FECHA')).month.unique().tolist())
_cuartos_wide   = sorted(df_pivot_tec.index.get_level_values('CUARTO').unique().tolist())
_cuartos_sin_solar = [c for c in _cuartos_wide if c < CUARTO_AMANECER or c > CUARTO_ATARDECER]

# ── Capacidad por tecnología ──────────────────────────────────────────────────
_cap_por_tec = pd.Series({
    tec: sum(inst_mw_final.get(p, 0)
             for p in _plantas_wide
             if p in planta_meta.index
             and planta_meta.loc[p, 'TECNOLOGIA'] == tec)
    for tec in _tecs_sistema
})

# ── Ruta de salida ────────────────────────────────────────────────────────────
_meses_str  = '_'.join(f'{m:02d}' for m in _meses_wide)
_ruta_wide  = os.path.join(CARPETA_DATOS, f'STATS_WIDE_{AÑO_ANALISIS}_{_meses_str}.xlsx')

# ── Estadísticos — MAXIMA usa max() real ─────────────────────────────────────
_stat_fns = {
    'MINIMA':    lambda x: x.min(),
    'MODA_BAJA': moda_baja_fn,
    'MEDIANA':   lambda x: x.quantile(0.50),
    'MODA_ALTA': moda_alta_fn,
    'MAXIMA':    lambda x: x.max(),        # ← máximo real
}

# ── Estilo Excel ──────────────────────────────────────────────────────────────
_font_bold    = Font(bold=True)
_color_header = 'D9D9D9'
_COLORES_MES  = ['BDD7EE','FCE4D6','E2EFDA','FFF2CC','DDEBF7','FCE4D6',
                  'E2EFDA','FFF2CC','BDD7EE','FCE4D6','E2EFDA','FFF2CC']

# ── Columnas meta ─────────────────────────────────────────────────────────────
_META_COLS_TEC = ['TECNOLOGIA', 'CAPACIDAD_INSTALADA_MW']

# ── Función para escribir hojas de estadísticos ───────────────────────────────
_fila_mes_wide    = [m for m in _meses_wide for _ in _cuartos_wide]
_fila_cuartos_wide = _cuartos_wide * len(_meses_wide)

def _escribir_hoja(ws, meta_cols, filas_data):
    n_meta = len(meta_cols)
    ws.append(['']*n_meta + _fila_mes_wide)
    for col_idx, mes_val in enumerate(_fila_mes_wide, start=n_meta+1):
        cell           = ws.cell(row=1, column=col_idx)
        cell.font      = _font_bold
        cell.alignment = Alignment(horizontal='center')
        cell.fill      = PatternFill(fill_type='solid',
                         fgColor=_COLORES_MES[_meses_wide.index(mes_val) % len(_COLORES_MES)])
    ws.append(['']*n_meta + _fila_cuartos_wide)
    for col_idx in range(n_meta+1, n_meta+len(_fila_cuartos_wide)+1):
        cell           = ws.cell(row=2, column=col_idx)
        cell.alignment = Alignment(horizontal='center')
        cell.fill      = PatternFill(fill_type='solid', fgColor=_color_header)
    ws.append(meta_cols + ['']*len(_fila_mes_wide))
    for col_idx, _ in enumerate(meta_cols, start=1):
        cell           = ws.cell(row=3, column=col_idx)
        cell.font      = _font_bold
        cell.alignment = Alignment(horizontal='center')
        cell.fill      = PatternFill(fill_type='solid', fgColor=_color_header)
    for fila in filas_data:
        ws.append(fila)
    ws.freeze_panes    = f'{openpyxl.utils.get_column_letter(n_meta+1)}4'
    ws.auto_filter.ref = f'A3:{openpyxl.utils.get_column_letter(n_meta)}3'
    for i in range(1, n_meta+1):
        ws.column_dimensions[openpyxl.utils.get_column_letter(i)].width = 22

# ── Workbook nuevo ────────────────────────────────────────────────────────────
wb = openpyxl.Workbook()
wb.remove(wb.active)

# ── Hojas TECNOLOGIA stats ────────────────────────────────────────────────────
print("Agregando estadísticos TECNOLOGIA...")
fecha_idx_t = pd.to_datetime(df_pivot_tec.index.get_level_values('FECHA'))
mes_idx_t   = fecha_idx_t.month
cto_idx_t   = df_pivot_tec.index.get_level_values('CUARTO')

for _stat_name, _fn in _stat_fns.items():
    _rows = []
    for _tec in _tecs_sistema:
        for _m in _meses_wide:
            for _c in _cuartos_wide:
                mask = (mes_idx_t == _m) & (cto_idx_t == _c)
                sub  = df_pivot_tec.loc[mask, _tec].dropna()
                if sub.empty or sub.max() == 0 or (_tec == 'Solar' and _c in _cuartos_sin_solar):
                    val = 0.0
                else:
                    val = round(float(_fn(sub)), 4)
                _rows.append({'TECNOLOGIA': _tec, 'MES': _m, 'CUARTO': _c, _stat_name: val})
    _df_stat = pd.DataFrame(_rows)
    _filas_tec = []
    for _tec in _tecs_sistema:
        _cap = round(float(_cap_por_tec[_tec] if _tec in _cap_por_tec.index else 0), 2)
        fila = [_tec, _cap]
        for _m in _meses_wide:
            for _c in _cuartos_wide:
                sub2 = _df_stat[(_df_stat['TECNOLOGIA']==_tec)&(_df_stat['MES']==_m)&(_df_stat['CUARTO']==_c)]
                fila.append(float(sub2[_stat_name].iloc[0]) if not sub2.empty else 0.0)
        _filas_tec.append(fila)
    ws = wb.create_sheet(title=f'TECNOLOGIA_{_stat_name}')
    _escribir_hoja(ws, _META_COLS_TEC, _filas_tec)
    print(f"   ✓ TECNOLOGIA_{_stat_name}")

# ── Hojas PLANTAS stats ───────────────────────────────────────────────────────
print("\nAgregando estadísticos PLANTAS...")
_META_COLS_PLA = ['PLANTA', 'TECNOLOGIA', 'NODO', 'ZONA', 'CAPACIDAD_INSTALADA_MW']
fecha_idx_p = pd.to_datetime(df_pivot_plantas.index.get_level_values('FECHA'))
mes_idx_p   = fecha_idx_p.month
cto_idx_p   = df_pivot_plantas.index.get_level_values('CUARTO')

for _stat_name, _fn in _stat_fns.items():
    _filas_pla = []
    for _p in _plantas_wide:
        _tec_p  = planta_meta.loc[_p, 'TECNOLOGIA'] if _p in planta_meta.index else ''
        _nodo_p = planta_meta.loc[_p, 'NODO']       if _p in planta_meta.index else ''
        _zona_p = planta_meta.loc[_p, 'ZONA']        if _p in planta_meta.index else ''
        _cap_p  = round(float(inst_mw_final.get(_p, 0)), 2)
        fila = [_p, _tec_p, _nodo_p, _zona_p, _cap_p]
        for _m in _meses_wide:
            for _c in _cuartos_wide:
                mask = (mes_idx_p == _m) & (cto_idx_p == _c)
                sub  = df_pivot_plantas.loc[mask, _p].dropna()
                if sub.empty or sub.max() == 0 or (_tec_p == 'Solar' and _c in _cuartos_sin_solar):
                    val = 0.0
                else:
                    val = round(float(_fn(sub)), 4)
                fila.append(val)
        _filas_pla.append(fila)
    ws = wb.create_sheet(title=f'PLANTAS_{_stat_name}')
    _escribir_hoja(ws, _META_COLS_PLA, _filas_pla)
    print(f"   ✓ PLANTAS_{_stat_name}")

print(f"\nSetup completo. Workbook tiene {len(wb.worksheets)} hojas.")
print(f"Ruta de salida: {_ruta_wide}")

Agregando estadísticos TECNOLOGIA...
   ✓ TECNOLOGIA_MINIMA
   ✓ TECNOLOGIA_MODA_BAJA
   ✓ TECNOLOGIA_MEDIANA
   ✓ TECNOLOGIA_MODA_ALTA
   ✓ TECNOLOGIA_MAXIMA

Agregando estadísticos PLANTAS...
   ✓ PLANTAS_MINIMA
   ✓ PLANTAS_MODA_BAJA
   ✓ PLANTAS_MEDIANA
   ✓ PLANTAS_MODA_ALTA
   ✓ PLANTAS_MAXIMA

Setup completo. Workbook tiene 10 hojas.
Ruta de salida: .\STATS_WIDE_2023_01_02_03_04_05_06_07_08_09_10_11_12.xlsx


In [128]:
# ══════════════════════════════════════════════════════════════════════════════
# Sección 9 — Exportación STATS_WIDE · Rampas (Pasos 17–27 combinados)
# ══════════════════════════════════════════════════════════════════════════════

# ── Setup necesario ───────────────────────────────────────────────────────────
_stat_fns = {
    'MINIMA':    lambda x: x.min(),
    'MODA_BAJA': moda_baja_fn,
    'MEDIANA':   lambda x: x.quantile(0.50),
    'MODA_ALTA': moda_alta_fn,
    'MAXIMA':    lambda x: x.max(),        # ← máximo real
}

# ── 1. Calcular rampas TECNOLOGIA + GEN_SOLAR ────────────────────────────────
_df_ramp_src = df_pivot_tec.reset_index().copy()
# ... resto del código igual
# ── 1. Calcular rampas TECNOLOGIA + GEN_SOLAR (Paso 17) ──────────────────────

_df_ramp_src = df_pivot_tec.reset_index().copy()
_df_ramp_src['MES'] = pd.to_datetime(_df_ramp_src['FECHA']).dt.month
_df_ramp_src['DIA'] = pd.to_datetime(_df_ramp_src['FECHA']).dt.day
_df_ramp_src = _df_ramp_src.sort_values(['FECHA', 'CUARTO']).reset_index(drop=True)

for _tec in _tecs_sistema:
    if _tec not in _df_ramp_src.columns:
        continue
    _df_ramp_src[f'RAMP_{_tec}'] = (
        _df_ramp_src
        .groupby('FECHA')[_tec]
        .transform(lambda s: s.shift(-1) - s)
    )
    if _tec == 'Solar':
        _df_ramp_src.loc[
            _df_ramp_src['CUARTO'].isin(_cuartos_sin_solar), f'RAMP_{_tec}'
        ] = 0.0

print("   ✓ Forward diff calculado")

_dias_wide = list(range(1, 32))
_rows_up, _rows_down         = [], []
_rows_gen_up, _rows_gen_down = [], []

for _tec in _tecs_sistema:
    _col = f'RAMP_{_tec}'
    if _col not in _df_ramp_src.columns:
        continue
    for _m in _meses_wide:
        for _d in _dias_wide:
            _sub_full = _df_ramp_src[
                (_df_ramp_src['MES'] == _m) &
                (_df_ramp_src['DIA'] == _d)
            ]
            sub = _sub_full[_col].dropna()

            if sub.empty:
                up_val = down_val = gen_sol_up = gen_sol_down = 0.0
            else:
                if (sub > 0).any():
                    _idx_up    = sub[sub > 0].idxmax()
                    up_val     = round(float(sub[_idx_up]), 4)
                    gen_sol_up = round(
                        float(_sub_full.loc[_idx_up, 'Solar'])
                        if 'Solar' in _sub_full.columns else 0.0, 4)
                else:
                    up_val = gen_sol_up = 0.0

                if (sub < 0).any():
                    _idx_down    = sub[sub < 0].idxmin()
                    down_val     = round(float(sub[_idx_down]), 4)
                    gen_sol_down = round(
                        float(_sub_full.loc[_idx_down, 'Solar'])
                        if 'Solar' in _sub_full.columns else 0.0, 4)
                else:
                    down_val = gen_sol_down = 0.0

            _rows_up.append(     {'TECNOLOGIA': _tec, 'MES': _m, 'DIA': _d, 'RAMP_UP_MAX':   up_val})
            _rows_down.append(   {'TECNOLOGIA': _tec, 'MES': _m, 'DIA': _d, 'RAMP_DOWN_MAX': down_val})
            _rows_gen_up.append( {'TECNOLOGIA': _tec, 'MES': _m, 'DIA': _d, 'GEN_SOLAR':     gen_sol_up})
            _rows_gen_down.append({'TECNOLOGIA': _tec, 'MES': _m, 'DIA': _d, 'GEN_SOLAR':     gen_sol_down})

_df_up       = pd.DataFrame(_rows_up).set_index(    ['TECNOLOGIA', 'MES', 'DIA'])
_df_down     = pd.DataFrame(_rows_down).set_index(  ['TECNOLOGIA', 'MES', 'DIA'])
_df_gen_up   = pd.DataFrame(_rows_gen_up).set_index( ['TECNOLOGIA', 'MES', 'DIA'])
_df_gen_down = pd.DataFrame(_rows_gen_down).set_index(['TECNOLOGIA', 'MES', 'DIA'])

print(f"   ✓ RAMP_UP_MAX    shape={_df_up.shape}")
print(f"   ✓ RAMP_DOWN_MAX  shape={_df_down.shape}")
print(f"   ✓ GEN_SOLAR_UP   shape={_df_gen_up.shape}")
print(f"   ✓ GEN_SOLAR_DOWN shape={_df_gen_down.shape}")

_fila_mes_ramp  = [m for m in _meses_wide for _ in _dias_wide]
_fila_dias_ramp = _dias_wide * len(_meses_wide)


# ── 2. Función única para escribir hojas de rampas ───────────────────────────

def _escribir_hoja_ramp_nivel(ws, meta_cols, filas_data):
    n_meta = len(meta_cols)
    ws.append(['']*n_meta + _fila_mes_ramp)
    for col_idx, mes_val in enumerate(_fila_mes_ramp, start=n_meta+1):
        cell           = ws.cell(row=1, column=col_idx)
        cell.font      = _font_bold
        cell.alignment = Alignment(horizontal='center')
        cell.fill      = PatternFill(fill_type='solid',
                         fgColor=_COLORES_MES[_meses_wide.index(mes_val) % len(_COLORES_MES)])
    ws.append(['']*n_meta + _fila_dias_ramp)
    for col_idx in range(n_meta+1, n_meta+len(_fila_dias_ramp)+1):
        cell           = ws.cell(row=2, column=col_idx)
        cell.alignment = Alignment(horizontal='center')
        cell.fill      = PatternFill(fill_type='solid', fgColor=_color_header)
    ws.append(meta_cols + ['']*len(_fila_mes_ramp))
    for col_idx, _ in enumerate(meta_cols, start=1):
        cell           = ws.cell(row=3, column=col_idx)
        cell.font      = _font_bold
        cell.alignment = Alignment(horizontal='center')
        cell.fill      = PatternFill(fill_type='solid', fgColor=_color_header)
    for fila in filas_data:
        ws.append(fila)
    ws.freeze_panes    = f'{openpyxl.utils.get_column_letter(n_meta+1)}4'
    ws.auto_filter.ref = f'A3:{openpyxl.utils.get_column_letter(n_meta)}3'
    for i in range(1, n_meta+1):
        ws.column_dimensions[openpyxl.utils.get_column_letter(i)].width = 22


# ── 3. Construir filas TECNOLOGIA ─────────────────────────────────────────────

def _build_filas_ramp(df_ramp, col_name):
    filas = []
    for _tec in _tecs_sistema:
        _cap = round(float(_cap_por_tec[_tec] if _tec in _cap_por_tec.index else 0), 2)
        fila = [_tec, _cap]
        for _m in _meses_wide:
            for _d in _dias_wide:
                try:
                    val = df_ramp.loc[(_tec, _m, _d), col_name]
                    fila.append(round(float(val), 4) if pd.notna(val) else 0.0)
                except KeyError:
                    fila.append(0.0)
        filas.append(fila)
    return filas


# ── 4. Construir filas ZONA/NODO ──────────────────────────────────────────────

def _build_filas_ramp_nivel(df_ramp, col_name, claves, cap_serie, dim1_nombre):
    filas = []
    for _clave in claves:
        _dim1, _tec = _clave
        _cap = round(float(cap_serie.get((_dim1, _tec), 0)), 2)
        fila = [_dim1, _tec, _cap]
        for _m in _meses_wide:
            for _d in _dias_wide:
                try:
                    val = df_ramp.loc[(_dim1, _tec, _m, _d), col_name]
                    fila.append(round(float(val), 4) if pd.notna(val) else 0.0)
                except KeyError:
                    fila.append(0.0)
        filas.append(fila)
    return filas


# ── 5. Escribir hojas TECNOLOGIA ──────────────────────────────────────────────

for _titulo, _df_r, _col in [
    ('TECNOLOGIA_RAMP_UP_MAX',        _df_up,       'RAMP_UP_MAX'),
    ('TECNOLOGIA_RAMP_DOWN_MAX',      _df_down,     'RAMP_DOWN_MAX'),
    ('TECNOLOGIA_RAMP_UP_GEN_SOLAR',  _df_gen_up,   'GEN_SOLAR'),
    ('TECNOLOGIA_RAMP_DOWN_GEN_SOLAR',_df_gen_down, 'GEN_SOLAR'),
]:
    ws = wb.create_sheet(title=_titulo)
    _escribir_hoja_ramp_nivel(ws, _META_COLS_TEC, _build_filas_ramp(_df_r, _col))
    print(f"   ✓ {_titulo}")


# ── 6. Calcular estadísticos y rampas ZONA y NODO ────────────────────────────

def _cap_por_nivel(dim_col):
    return (
        pd.DataFrame([
            {dim_col:      planta_meta.loc[p, dim_col],
             'TECNOLOGIA': planta_meta.loc[p, 'TECNOLOGIA'],
             'CAP_MW':     inst_mw_final.get(p, 0)}
            for p in _plantas_wide
            if p in planta_meta.index and pd.notna(planta_meta.loc[p, dim_col])
        ])
        .groupby([dim_col, 'TECNOLOGIA'])['CAP_MW'].sum()
    )

_cap_zona_tec = _cap_por_nivel('ZONA')
_cap_nodo_tec = _cap_por_nivel('NODO')


def _agregar_nivel(pivot_df, claves):
    fecha_idx = pd.to_datetime(pivot_df.index.get_level_values('FECHA'))
    mes_idx   = fecha_idx.month
    cto_idx   = pivot_df.index.get_level_values('CUARTO')
    stat_dfs  = {}
    for _stat_name, _fn in _stat_fns.items():
        _rows = []
        for _clave in claves:
            _dim1, _tec = _clave
            for _m in _meses_wide:
                for _c in _cuartos_wide:
                    mask = (mes_idx == _m) & (cto_idx == _c)
                    sub  = pivot_df.loc[mask, _clave].dropna()
                    if sub.empty or sub.max() == 0 or (_tec == 'Solar' and _c in _cuartos_sin_solar):
                        val = 0.0
                    else:
                        val = round(float(_fn(sub)), 4)
                    _rows.append({'DIM1': _dim1, 'TECNOLOGIA': _tec,
                                  'MES': _m, 'CUARTO': _c, _stat_name: val})
        stat_dfs[_stat_name] = (pd.DataFrame(_rows)
                                  .set_index(['DIM1', 'TECNOLOGIA', 'MES', 'CUARTO']))
        print(f"   ✓ {_stat_name:<12s} shape={stat_dfs[_stat_name].shape}")
    return stat_dfs


def _build_filas_nivel(stat_dfs, claves, cap_serie, stat_name):
    filas = []
    for _clave in claves:
        _dim1, _tec = _clave
        _cap = round(float(cap_serie.get((_dim1, _tec), 0)), 2)
        fila = [_dim1, _tec, _cap]
        for _m in _meses_wide:
            for _c in _cuartos_wide:
                try:
                    val = stat_dfs[stat_name].loc[(_dim1, _tec, _m, _c), stat_name]
                    fila.append(round(float(val), 4) if pd.notna(val) else 0.0)
                except KeyError:
                    fila.append(0.0)
        filas.append(fila)
    return filas


def _calcular_ramps_nivel(pivot_df, claves, dim1_nombre):
    fecha_arr = pd.to_datetime(pivot_df.index.get_level_values('FECHA'))
    mes_arr   = fecha_arr.month
    dia_arr   = fecha_arr.day
    cto_arr   = pivot_df.index.get_level_values('CUARTO')
    _rows_up, _rows_down = [], []
    for _clave in claves:
        _dim1, _tec = _clave
        ramp = (pivot_df[_clave]
                .groupby(level='FECHA')
                .transform(lambda s: s.shift(-1) - s))
        if _tec == 'Solar':
            ramp[cto_arr.isin(_cuartos_sin_solar)] = 0.0
        for _m in _meses_wide:
            for _d in _dias_wide:
                mask = (mes_arr == _m) & (dia_arr == _d)
                sub  = ramp[mask].dropna()
                if sub.empty:
                    up_val, down_val = 0.0, 0.0
                else:
                    up_val   = round(float(sub[sub > 0].max()) if (sub > 0).any() else 0.0, 4)
                    down_val = round(float(sub[sub < 0].min()) if (sub < 0).any() else 0.0, 4)
                base = {dim1_nombre: _dim1, 'TECNOLOGIA': _tec, 'MES': _m, 'DIA': _d}
                _rows_up.append(  {**base, 'RAMP_UP_MAX':   up_val})
                _rows_down.append({**base, 'RAMP_DOWN_MAX': down_val})
    idx = [dim1_nombre, 'TECNOLOGIA', 'MES', 'DIA']
    return (pd.DataFrame(_rows_up).set_index(idx),
            pd.DataFrame(_rows_down).set_index(idx))


# ── 7. Escribir hojas ZONA y NODO ─────────────────────────────────────────────

for _dim, _pivot, _claves_fn, _meta_cols, _cap_serie in [
    ('ZONA', df_pivot_zona_tec,  lambda: sorted(df_pivot_zona_tec.columns.tolist()),
     ['ZONA', 'TECNOLOGIA', 'CAPACIDAD_INSTALADA_MW'],  _cap_zona_tec),
    ('NODO', df_pivot_nodo_tec2, lambda: sorted(df_pivot_nodo_tec2.columns.tolist()),
     ['NODO', 'TECNOLOGIA', 'CAPACIDAD_INSTALADA_MW'],  _cap_nodo_tec),
]:
    _claves = _claves_fn()
    print(f"\nAgregando estadísticos {_dim}...")
    _stat_dfs_nivel = _agregar_nivel(_pivot, _claves)
    for _stat_name in _stat_fns:
        ws = wb.create_sheet(title=f'{_dim}_{_stat_name}')
        _escribir_hoja(ws, _meta_cols,
                       _build_filas_nivel(_stat_dfs_nivel, _claves, _cap_serie, _stat_name))
        print(f"   ✓ {_dim}_{_stat_name}")

    print(f"\nCalculando rampas {_dim}...")
    _df_up_n, _df_down_n = _calcular_ramps_nivel(_pivot, _claves, _dim)
    for _titulo, _df_r, _col in [
        (f'{_dim}_RAMP_UP_MAX',   _df_up_n,   'RAMP_UP_MAX'),
        (f'{_dim}_RAMP_DOWN_MAX', _df_down_n, 'RAMP_DOWN_MAX'),
    ]:
        ws = wb.create_sheet(title=_titulo)
        _escribir_hoja_ramp_nivel(ws, _meta_cols,
            _build_filas_ramp_nivel(_df_r, _col, _claves, _cap_serie, _dim))
        print(f"   ✓ {_titulo}")


# ── 8. Guardar ────────────────────────────────────────────────────────────────

wb.save(_ruta_wide)
print(f"\n✅  {os.path.basename(_ruta_wide)}")
print(f"    Sheets ({len(wb.worksheets)}): {[ws.title for ws in wb.worksheets]}")
print(f"    Meses  : {_meses_wide}")
print(f"    Ruta   : {os.path.abspath(_ruta_wide)}")

   ✓ Forward diff calculado
   ✓ RAMP_UP_MAX    shape=(1488, 1)
   ✓ RAMP_DOWN_MAX  shape=(1488, 1)
   ✓ GEN_SOLAR_UP   shape=(1488, 1)
   ✓ GEN_SOLAR_DOWN shape=(1488, 1)
   ✓ TECNOLOGIA_RAMP_UP_MAX
   ✓ TECNOLOGIA_RAMP_DOWN_MAX
   ✓ TECNOLOGIA_RAMP_UP_GEN_SOLAR
   ✓ TECNOLOGIA_RAMP_DOWN_GEN_SOLAR

Agregando estadísticos ZONA...
   ✓ MINIMA       shape=(12672, 1)
   ✓ MODA_BAJA    shape=(12672, 1)
   ✓ MEDIANA      shape=(12672, 1)
   ✓ MODA_ALTA    shape=(12672, 1)
   ✓ MAXIMA       shape=(12672, 1)
   ✓ ZONA_MINIMA
   ✓ ZONA_MODA_BAJA
   ✓ ZONA_MEDIANA
   ✓ ZONA_MODA_ALTA
   ✓ ZONA_MAXIMA

Calculando rampas ZONA...
   ✓ ZONA_RAMP_UP_MAX
   ✓ ZONA_RAMP_DOWN_MAX

Agregando estadísticos NODO...
   ✓ MINIMA       shape=(40320, 1)
   ✓ MODA_BAJA    shape=(40320, 1)
   ✓ MEDIANA      shape=(40320, 1)
   ✓ MODA_ALTA    shape=(40320, 1)
   ✓ MAXIMA       shape=(40320, 1)
   ✓ NODO_MINIMA
   ✓ NODO_MODA_BAJA
   ✓ NODO_MEDIANA
   ✓ NODO_MODA_ALTA
   ✓ NODO_MAXIMA

Calculando rampas NODO...
  

In [129]:
# ── Exportar gen_horario Solar + Eólica + Hidro ───────────────────────────────
# Requiere: df_pivot_tec ya construido (Sección 3)

gen = df_pivot_tec[['Solar', 'Eolica', 'Hidro']].reset_index()

# Solo las horas en punto (CUARTO = 0, 100, 200, ... 2300)
gen = gen[gen['CUARTO'] % 100 == 0].copy()

gen['datetime'] = (
    pd.to_datetime(gen['FECHA'].astype(str))
    + pd.to_timedelta(gen['CUARTO'] // 100, unit='h')
)

gen_horario = (
    gen.set_index('datetime')[['Solar', 'Eolica', 'Hidro']]
    .rename(columns={
        'Solar':  'solar_mw_real',
        'Eolica': 'eolica_mw_real',
        'Hidro':  'hidro_mw_real',
    })
    .astype('float32')
)

nombre_archivo = f'solar_eolica_hidro_horario_{AÑO_ANALISIS}.parquet'
gen_horario.to_parquet(os.path.join(CARPETA_DATOS, nombre_archivo))

print(f'✅  {len(gen_horario)} horas exportadas → {nombre_archivo}')
print(f'   Rango : {gen_horario.index.min()} → {gen_horario.index.max()}')
print(f'   Shape : {gen_horario.shape}')
display(gen_horario.head(5))
display(gen_horario.describe().round(2))

✅  8760 horas exportadas → solar_eolica_hidro_horario_2023.parquet
   Rango : 2023-01-01 01:00:00 → 2024-01-01 00:00:00
   Shape : (8760, 3)


TECNOLOGIA,solar_mw_real,eolica_mw_real,hidro_mw_real
datetime,,,
2023-01-01 01:00:00,0.0,32.662998,470.101807
2023-01-01 02:00:00,0.0,24.846001,481.831604
2023-01-01 03:00:00,0.0,10.436300,445.753204
2023-01-01 04:00:00,0.0,11.479800,424.532104
2023-01-01 05:00:00,0.0,9.416400,420.381195


TECNOLOGIA,solar_mw_real,eolica_mw_real,hidro_mw_real
count,8760.00,8760.00,8760.00
mean,93.68,101.83,688.79
std,124.73,96.66,223.56
min,0.00,0.03,5.88
25%,0.00,12.10,530.09
50%,0.06,69.94,711.09
75%,202.44,183.49,856.40
max,467.63,315.08,1375.91
